In [7]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations
from scipy.optimize import approx_fprime

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp, Statevector
# Using the local V2 Estimator simulator instead of IBM Runtime
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 9
k = 3

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        paulis[c[0]], paulis[c[1]], paulis[c[2]] = pauli, pauli, pauli
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=2)
#estimator = Estimator()

ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])
estimator = EstimatorV2()

# Highly recommended: Set the number of shots. 
# Finite shots introduce noise, so you need a high number for stable gradients.
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. THE NON-LINEAR LOSS FUNCTION
# ==========================================
alpha_loss = 3
beta_loss = 0.5
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def loss_func_estimator(x, silent=False):
    job = estimator.run([
        (ansatz, observables_x, x),
        (ansatz, observables_y, x),
        (ansatz, observables_z, x)
    ])
    result = job.result()
    
    node_exp_map = {}
    idx = 0
    for r in result:
        for ev in r.data.evs:
            node_exp_map[idx] = ev
            idx += 1
            
    loss = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss += w * np.tanh(alpha_loss * node_exp_map[edge0]) * np.tanh(alpha_loss * node_exp_map[edge1])
        
    regulation_term = 0
    for i in range(num_nodes):
        regulation_term += np.tanh(alpha_loss * node_exp_map[i])**2
        
    regulation_term = regulation_term / num_nodes
    regulation_term = beta_loss * v * (regulation_term**2)
    loss = loss + regulation_term
    
    if not silent:
        experiment_result.append({"loss": loss, "exp_map": node_exp_map})
        print(f"Iter {len(experiment_result)}: {loss:.6f}")
        
    return loss

def silent_loss_func(x):
    """Wrapper to prevent terminal spam during gradient calculation"""
    return loss_func_estimator(x, silent=True)

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam.npy"

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    initial_params = np.load(warm_start_file)
    if len(initial_params) != ansatz.num_parameters:
        print("Warning: Loaded parameters do not match ansatz size! Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(ansatz.num_parameters)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(ansatz.num_parameters)

# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize(loss_fn, theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    """Custom Adam optimizer loop using Euclidean gradients."""
    print(f"Starting optimization with Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    # Initialize momentum (m) and velocity (v) vectors
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    # Do an initial non-silent run to register the first point
    _ = loss_func_estimator(theta_i, silent=False)
    
    for i in range(max_iter):
        print(f"\n--- Adam Step {i+1}/{max_iter} ---")
        t += 1
        
        # 1. Calculate the Euclidean gradient (quietly)
        grad = approx_fprime(theta_i, silent_loss_func, epsilon=1e-5)
        
        # 2. Update biased first moment estimate (momentum)
        m_t = beta1 * m_t + (1 - beta1) * grad
        
        # 3. Update biased second raw moment estimate (velocity)
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        # 4. Compute bias-corrected estimates
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        # 5. Update parameters
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        # Check for convergence (if step size is incredibly small)
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            loss_func_estimator(theta_i, silent=False)
            break
            
        theta_i = theta_next
        
        # Run non-silently to log the official step taken
        loss_func_estimator(theta_i, silent=False)
        
    return theta_i

# Execute the Adam Optimizer
# Note: Adam learning rates (lr) usually range between 0.001 and 0.1 depending on the landscape
best_params = adam_optimize(
    loss_fn=silent_loss_func, 
    theta0=initial_params,
    lr=0.05,        # Adjust this! Higher = faster but might overshoot
    max_iter=50
)

print("\nAdam Optimization Complete!")

# Optional: Save the new Adam parameters
np.save("pce_optimal_theta_adam.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

best_cut = 0
best_bits = []

for edge0, edge1 in graph.edges():
    swapped_bits = cur_bits.copy()
    swapped_bits[edge0], swapped_bits[edge1] = swapped_bits[edge1], swapped_bits[edge0]
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if best_cut < cut_size:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam.npy...
Starting optimization with Adam (lr=0.05)...
Iter 1: -935.590166

--- Adam Step 1/50 ---
Iter 2: -927.725081

--- Adam Step 2/50 ---
Iter 3: -942.526889

--- Adam Step 3/50 ---
Iter 4: -944.035572

--- Adam Step 4/50 ---
Iter 5: -946.511555

--- Adam Step 5/50 ---
Iter 6: -950.930698

--- Adam Step 6/50 ---
Iter 7: -954.292525

--- Adam Step 7/50 ---
Iter 8: -955.719754

--- Adam Step 8/50 ---
Iter 9: -956.217190

--- Adam Step 9/50 ---
Iter 10: -957.240624

--- Adam Step 10/50 ---
Iter 11: -959.355922

--- Adam Step 11/50 ---
Iter 12: -961.221370

--- Adam Step 12/50 ---
Iter 13: -961.869513

--- Adam Step 13/50 ---
Iter 14: -962.293465

--- Adam Step 14/50 ---
Iter 15: -963.469761

--- Adam Step 15/50 ---
Iter 16: -964.863251

--- Adam Step 16/50 ---
Iter 17: -965.946675

--- Adam Step 17/50 ---
Iter 18: -966.825061

--- Adam Step 18/50 ---
Iter 19: -967.624243

--- Adam Step 19/50 ---
Iter 20: -968.3

In [11]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations
from scipy.optimize import approx_fprime

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp, Statevector
# Using the local V2 Estimator simulator instead of IBM Runtime
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 9
k = 3

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        paulis[c[0]], paulis[c[1]], paulis[c[2]] = pauli, pauli, pauli
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
#estimator = Estimator()

ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])
estimator = EstimatorV2()

# Highly recommended: Set the number of shots. 
# Finite shots introduce noise, so you need a high number for stable gradients.
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. THE NON-LINEAR LOSS FUNCTION
# ==========================================
alpha_loss = 3
beta_loss = 0.5
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def loss_func_estimator(x, silent=False):
    job = estimator.run([
        (ansatz, observables_x, x),
        (ansatz, observables_y, x),
        (ansatz, observables_z, x)
    ])
    result = job.result()
    
    node_exp_map = {}
    idx = 0
    for r in result:
        for ev in r.data.evs:
            node_exp_map[idx] = ev
            idx += 1
            
    loss = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss += w * np.tanh(alpha_loss * node_exp_map[edge0]) * np.tanh(alpha_loss * node_exp_map[edge1])
        
    regulation_term = 0
    for i in range(num_nodes):
        regulation_term += np.tanh(alpha_loss * node_exp_map[i])**2
        
    regulation_term = regulation_term / num_nodes
    regulation_term = beta_loss * v * (regulation_term**2)
    loss = loss + regulation_term
    
    if not silent:
        experiment_result.append({"loss": loss, "exp_map": node_exp_map})
        print(f"Iter {len(experiment_result)}: {loss:.6f}")
        
    return loss

def silent_loss_func(x):
    """Wrapper to prevent terminal spam during gradient calculation"""
    return loss_func_estimator(x, silent=True)

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam.npy"

#warm_start_file = "pce_optimal_theta.npy" # or whatever you named your Adam output
num_new_params = ansatz.num_parameters # For reps=4, this will be 90

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        # Perfect match (e.g., if you run reps=4 again later)
        initial_params = old_params
        
    elif num_old_params < num_new_params:
        # PADDING LOGIC: Transferring from a shallower circuit to a deeper one
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        
        # Initialize the new array with tiny random noise (e.g., between -0.01 and 0.01)
        # We use tiny noise instead of perfect 0.0 to prevent barren plateaus in the new layers
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        
        # Inject the highly optimized parameters into the beginning of the circuit
        initial_params[:num_old_params] = old_params
        
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)
# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize(loss_fn, theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    """Custom Adam optimizer loop using Euclidean gradients."""
    print(f"Starting optimization with Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    # Initialize momentum (m) and velocity (v) vectors
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    # Do an initial non-silent run to register the first point
    _ = loss_func_estimator(theta_i, silent=False)
    
    for i in range(max_iter):
        print(f"\n--- Adam Step {i+1}/{max_iter} ---")
        t += 1
        
        # 1. Calculate the Euclidean gradient (quietly)
        grad = approx_fprime(theta_i, silent_loss_func, epsilon=1e-5)
        
        # 2. Update biased first moment estimate (momentum)
        m_t = beta1 * m_t + (1 - beta1) * grad
        
        # 3. Update biased second raw moment estimate (velocity)
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        # 4. Compute bias-corrected estimates
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        # 5. Update parameters
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        # Check for convergence (if step size is incredibly small)
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            loss_func_estimator(theta_i, silent=False)
            break
            
        theta_i = theta_next
        
        # Run non-silently to log the official step taken
        loss_func_estimator(theta_i, silent=False)
        
    return theta_i

# Execute the Adam Optimizer
# Note: Adam learning rates (lr) usually range between 0.001 and 0.1 depending on the landscape
best_params = adam_optimize(
    loss_fn=silent_loss_func, 
    theta0=initial_params,
    lr=0.05,        # Adjust this! Higher = faster but might overshoot
    max_iter=50
)

print("\nAdam Optimization Complete!")

# Optional: Save the new Adam parameters
np.save("pce_optimal_theta_adam_reps4.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

best_cut = 0
best_bits = []

for edge0, edge1 in graph.edges():
    swapped_bits = cur_bits.copy()
    swapped_bits[edge0], swapped_bits[edge1] = swapped_bits[edge1], swapped_bits[edge0]
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if best_cut < cut_size:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam.npy...
Padding parameters! Transferring 54 angles into a 90 angle circuit.
Starting optimization with Adam (lr=0.05)...
Iter 1: 85.438264

--- Adam Step 1/50 ---
Iter 2: -18.178518

--- Adam Step 2/50 ---
Iter 3: -127.458567

--- Adam Step 3/50 ---
Iter 4: -243.117238

--- Adam Step 4/50 ---
Iter 5: -346.133323

--- Adam Step 5/50 ---
Iter 6: -418.864444

--- Adam Step 6/50 ---
Iter 7: -464.813448

--- Adam Step 7/50 ---
Iter 8: -500.062777

--- Adam Step 8/50 ---
Iter 9: -538.097284

--- Adam Step 9/50 ---
Iter 10: -580.306128

--- Adam Step 10/50 ---
Iter 11: -622.743534

--- Adam Step 11/50 ---
Iter 12: -662.295777

--- Adam Step 12/50 ---
Iter 13: -699.272488

--- Adam Step 13/50 ---
Iter 14: -735.146358

--- Adam Step 14/50 ---
Iter 15: -770.433331

--- Adam Step 15/50 ---
Iter 16: -804.364924

--- Adam Step 16/50 ---
Iter 17: -835.181563

--- Adam Step 17/50 ---
Iter 18: -860.786189

--- Adam Step 18/50 

In [16]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations
from scipy.optimize import approx_fprime

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp, Statevector
# Using the local V2 Estimator simulator instead of IBM Runtime
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 9
k = 3

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        paulis[c[0]], paulis[c[1]], paulis[c[2]] = pauli, pauli, pauli
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
#estimator = Estimator()

ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])
estimator = EstimatorV2()

# Highly recommended: Set the number of shots. 
# Finite shots introduce noise, so you need a high number for stable gradients.
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. THE NON-LINEAR LOSS FUNCTION
# ==========================================
alpha_loss = 3
beta_loss = 0.5
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def loss_func_estimator(x, silent=False):
    job = estimator.run([
        (ansatz, observables_x, x),
        (ansatz, observables_y, x),
        (ansatz, observables_z, x)
    ])
    result = job.result()
    
    node_exp_map = {}
    idx = 0
    for r in result:
        for ev in r.data.evs:
            node_exp_map[idx] = ev
            idx += 1
            
    loss = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss += w * np.tanh(alpha_loss * node_exp_map[edge0]) * np.tanh(alpha_loss * node_exp_map[edge1])
        
    regulation_term = 0
    for i in range(num_nodes):
        regulation_term += np.tanh(alpha_loss * node_exp_map[i])**2
        
    regulation_term = regulation_term / num_nodes
    regulation_term = beta_loss * v * (regulation_term**2)
    loss = loss + regulation_term
    
    if not silent:
        experiment_result.append({"loss": loss, "exp_map": node_exp_map})
        print(f"Iter {len(experiment_result)}: {loss:.6f}")
        
    return loss

def silent_loss_func(x):
    """Wrapper to prevent terminal spam during gradient calculation"""
    return loss_func_estimator(x, silent=True)

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4.npy"

#warm_start_file = "pce_optimal_theta.npy" # or whatever you named your Adam output
num_new_params = ansatz.num_parameters # For reps=4, this will be 90

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        # Perfect match (e.g., if you run reps=4 again later)
        initial_params = old_params
        
    elif num_old_params < num_new_params:
        # PADDING LOGIC: Transferring from a shallower circuit to a deeper one
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        
        # Initialize the new array with tiny random noise (e.g., between -0.01 and 0.01)
        # We use tiny noise instead of perfect 0.0 to prevent barren plateaus in the new layers
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        
        # Inject the highly optimized parameters into the beginning of the circuit
        initial_params[:num_old_params] = old_params
        
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)
# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize(loss_fn, theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    """Custom Adam optimizer loop using Euclidean gradients."""
    print(f"Starting optimization with Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    # Initialize momentum (m) and velocity (v) vectors
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    # Do an initial non-silent run to register the first point
    _ = loss_func_estimator(theta_i, silent=False)
    
    for i in range(max_iter):
        print(f"\n--- Adam Step {i+1}/{max_iter} ---")
        t += 1
        
        # 1. Calculate the Euclidean gradient (quietly)
        grad = approx_fprime(theta_i, silent_loss_func, epsilon=1e-5)
        
        # 2. Update biased first moment estimate (momentum)
        m_t = beta1 * m_t + (1 - beta1) * grad
        
        # 3. Update biased second raw moment estimate (velocity)
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        # 4. Compute bias-corrected estimates
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        # 5. Update parameters
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        # Check for convergence (if step size is incredibly small)
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            loss_func_estimator(theta_i, silent=False)
            break
            
        theta_i = theta_next
        
        # Run non-silently to log the official step taken
        loss_func_estimator(theta_i, silent=False)
        
    return theta_i

# Execute the Adam Optimizer
# Note: Adam learning rates (lr) usually range between 0.001 and 0.1 depending on the landscape
best_params = adam_optimize(
    loss_fn=silent_loss_func, 
    theta0=initial_params,
    lr=0.05,        # Adjust this! Higher = faster but might overshoot
    max_iter=20
)

print("\nAdam Optimization Complete!")

# Optional: Save the new Adam parameters
np.save("pce_optimal_theta_adam_reps4qbt12.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

best_cut = 0
best_bits = []

for edge0, edge1 in graph.edges():
    swapped_bits = cur_bits.copy()
    swapped_bits[edge0], swapped_bits[edge1] = swapped_bits[edge1], swapped_bits[edge0]
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if best_cut < cut_size:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4.npy...
Starting optimization with Adam (lr=0.05)...
Iter 1: -1138.403755

--- Adam Step 1/20 ---
Iter 2: -1100.713720

--- Adam Step 2/20 ---
Iter 3: -1133.612872

--- Adam Step 3/20 ---
Iter 4: -1127.610777

--- Adam Step 4/20 ---
Iter 5: -1128.092691

--- Adam Step 5/20 ---
Iter 6: -1138.370062

--- Adam Step 6/20 ---
Iter 7: -1143.524313

--- Adam Step 7/20 ---
Iter 8: -1143.258991

--- Adam Step 8/20 ---
Iter 9: -1143.378293

--- Adam Step 9/20 ---
Iter 10: -1146.034107

--- Adam Step 10/20 ---
Iter 11: -1149.230796

--- Adam Step 11/20 ---
Iter 12: -1151.402752

--- Adam Step 12/20 ---
Iter 13: -1153.126784

--- Adam Step 13/20 ---
Iter 14: -1155.261009

--- Adam Step 14/20 ---
Iter 15: -1157.482549

--- Adam Step 15/20 ---
Iter 16: -1159.194726

--- Adam Step 16/20 ---
Iter 17: -1160.566842

--- Adam Step 17/20 ---
Iter 18: -1162.239106

--- Adam Step 18/20 ---
Iter 19: -1164.486749

--- Adam Step 

In [18]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations
from scipy.optimize import approx_fprime

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp, Statevector
# Using the local V2 Estimator simulator instead of IBM Runtime
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 9
k = 3

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        paulis[c[0]], paulis[c[1]], paulis[c[2]] = pauli, pauli, pauli
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
#estimator = Estimator()

ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])
estimator = EstimatorV2()

# Highly recommended: Set the number of shots. 
# Finite shots introduce noise, so you need a high number for stable gradients.
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. THE NON-LINEAR LOSS FUNCTION
# ==========================================
alpha_loss = 3
beta_loss = 0.5
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def loss_func_estimator(x, silent=False):
    job = estimator.run([
        (ansatz, observables_x, x),
        (ansatz, observables_y, x),
        (ansatz, observables_z, x)
    ])
    result = job.result()
    
    node_exp_map = {}
    idx = 0
    for r in result:
        for ev in r.data.evs:
            node_exp_map[idx] = ev
            idx += 1
            
    loss = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss += w * np.tanh(alpha_loss * node_exp_map[edge0]) * np.tanh(alpha_loss * node_exp_map[edge1])
        
    regulation_term = 0
    for i in range(num_nodes):
        regulation_term += np.tanh(alpha_loss * node_exp_map[i])**2
        
    regulation_term = regulation_term / num_nodes
    regulation_term = beta_loss * v * (regulation_term**2)
    loss = loss + regulation_term
    
    if not silent:
        experiment_result.append({"loss": loss, "exp_map": node_exp_map})
        print(f"Iter {len(experiment_result)}: {loss:.6f}")
        
    return loss

def silent_loss_func(x):
    """Wrapper to prevent terminal spam during gradient calculation"""
    return loss_func_estimator(x, silent=True)

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue.npy"

#warm_start_file = "pce_optimal_theta.npy" # or whatever you named your Adam output
num_new_params = ansatz.num_parameters # For reps=4, this will be 90

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        # Perfect match (e.g., if you run reps=4 again later)
        initial_params = old_params
        
    elif num_old_params < num_new_params:
        # PADDING LOGIC: Transferring from a shallower circuit to a deeper one
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        
        # Initialize the new array with tiny random noise (e.g., between -0.01 and 0.01)
        # We use tiny noise instead of perfect 0.0 to prevent barren plateaus in the new layers
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        
        # Inject the highly optimized parameters into the beginning of the circuit
        initial_params[:num_old_params] = old_params
        
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)
# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize(loss_fn, theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    """Custom Adam optimizer loop using Euclidean gradients."""
    print(f"Starting optimization with Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    # Initialize momentum (m) and velocity (v) vectors
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    # Do an initial non-silent run to register the first point
    _ = loss_func_estimator(theta_i, silent=False)
    
    for i in range(max_iter):
        print(f"\n--- Adam Step {i+1}/{max_iter} ---")
        t += 1
        
        # 1. Calculate the Euclidean gradient (quietly)
        grad = approx_fprime(theta_i, silent_loss_func, epsilon=1e-5)
        
        # 2. Update biased first moment estimate (momentum)
        m_t = beta1 * m_t + (1 - beta1) * grad
        
        # 3. Update biased second raw moment estimate (velocity)
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        # 4. Compute bias-corrected estimates
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        # 5. Update parameters
        #theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        
        # 5. Update parameters with Learning Rate Decay
        # The learning rate will smoothly shrink by 2% every iteration
        current_lr = lr * (0.98 ** t)
        theta_next = theta_i - current_lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        # Check for convergence (if step size is incredibly small)
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            loss_func_estimator(theta_i, silent=False)
            break
            
        theta_i = theta_next
        
        # Run non-silently to log the official step taken
        loss_func_estimator(theta_i, silent=False)
        
    return theta_i

# Execute the Adam Optimizer
# Note: Adam learning rates (lr) usually range between 0.001 and 0.1 depending on the landscape
best_params = adam_optimize(
    loss_fn=silent_loss_func, 
    theta0=initial_params,
    lr=0.05,        # Adjust this! Higher = faster but might overshoot
    max_iter=20
)

print("\nAdam Optimization Complete!")

# Optional: Save the new Adam parameters
np.save("pce_optimal_theta_adam_reps4qbt12Continue.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

best_cut = 0
best_bits = []

for edge0, edge1 in graph.edges():
    swapped_bits = cur_bits.copy()
    swapped_bits[edge0], swapped_bits[edge1] = swapped_bits[edge1], swapped_bits[edge0]
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if best_cut < cut_size:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue.npy...
Starting optimization with Adam (lr=0.05)...
Iter 1: -1190.950442

--- Adam Step 1/20 ---
Iter 2: -1136.546986

--- Adam Step 2/20 ---
Iter 3: -1183.481152

--- Adam Step 3/20 ---
Iter 4: -1181.379706

--- Adam Step 4/20 ---
Iter 5: -1171.709436

--- Adam Step 5/20 ---
Iter 6: -1176.282787

--- Adam Step 6/20 ---
Iter 7: -1187.169403

--- Adam Step 7/20 ---
Iter 8: -1193.367392

--- Adam Step 8/20 ---
Iter 9: -1192.794082

--- Adam Step 9/20 ---
Iter 10: -1190.699077

--- Adam Step 10/20 ---
Iter 11: -1191.980132

--- Adam Step 11/20 ---
Iter 12: -1196.227364

--- Adam Step 12/20 ---
Iter 13: -1200.339631

--- Adam Step 13/20 ---
Iter 14: -1202.202818

--- Adam Step 14/20 ---
Iter 15: -1202.144683

--- Adam Step 15/20 ---
Iter 16: -1202.221735

--- Adam Step 16/20 ---
Iter 17: -1204.031561

--- Adam Step 17/20 ---
Iter 18: -1207.159260

--- Adam Step 18/20 ---
Iter 19: -1210.084467

-

In [20]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations
from scipy.optimize import approx_fprime

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp, Statevector
# Using the local V2 Estimator simulator instead of IBM Runtime
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 9
k = 3

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        paulis[c[0]], paulis[c[1]], paulis[c[2]] = pauli, pauli, pauli
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
#estimator = Estimator()

ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])
estimator = EstimatorV2()

# Highly recommended: Set the number of shots. 
# Finite shots introduce noise, so you need a high number for stable gradients.
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. THE NON-LINEAR LOSS FUNCTION
# ==========================================
alpha_loss = 3
beta_loss = 0.5
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def loss_func_estimator(x, silent=False):
    job = estimator.run([
        (ansatz, observables_x, x),
        (ansatz, observables_y, x),
        (ansatz, observables_z, x)
    ])
    result = job.result()
    
    node_exp_map = {}
    idx = 0
    for r in result:
        for ev in r.data.evs:
            node_exp_map[idx] = ev
            idx += 1
            
    loss = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss += w * np.tanh(alpha_loss * node_exp_map[edge0]) * np.tanh(alpha_loss * node_exp_map[edge1])
        
    regulation_term = 0
    for i in range(num_nodes):
        regulation_term += np.tanh(alpha_loss * node_exp_map[i])**2
        
    regulation_term = regulation_term / num_nodes
    regulation_term = beta_loss * v * (regulation_term**2)
    loss = loss + regulation_term
    
    if not silent:
        experiment_result.append({"loss": loss, "exp_map": node_exp_map})
        print(f"Iter {len(experiment_result)}: {loss:.6f}")
        
    return loss

def silent_loss_func(x):
    """Wrapper to prevent terminal spam during gradient calculation"""
    return loss_func_estimator(x, silent=True)

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue.npy"

#warm_start_file = "pce_optimal_theta.npy" # or whatever you named your Adam output
num_new_params = ansatz.num_parameters # For reps=4, this will be 90

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        # Perfect match (e.g., if you run reps=4 again later)
        initial_params = old_params
        
    elif num_old_params < num_new_params:
        # PADDING LOGIC: Transferring from a shallower circuit to a deeper one
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        
        # Initialize the new array with tiny random noise (e.g., between -0.01 and 0.01)
        # We use tiny noise instead of perfect 0.0 to prevent barren plateaus in the new layers
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        
        # Inject the highly optimized parameters into the beginning of the circuit
        initial_params[:num_old_params] = old_params
        
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)
# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize(loss_fn, theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    """Custom Adam optimizer loop using Euclidean gradients."""
    print(f"Starting optimization with Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    # Initialize momentum (m) and velocity (v) vectors
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    # Do an initial non-silent run to register the first point
    _ = loss_func_estimator(theta_i, silent=False)
    
    for i in range(max_iter):
        print(f"\n--- Adam Step {i+1}/{max_iter} ---")
        t += 1
        
        # 1. Calculate the Euclidean gradient (quietly)
        grad = approx_fprime(theta_i, silent_loss_func, epsilon=1e-5)
        
        # 2. Update biased first moment estimate (momentum)
        m_t = beta1 * m_t + (1 - beta1) * grad
        
        # 3. Update biased second raw moment estimate (velocity)
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        # 4. Compute bias-corrected estimates
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        # 5. Update parameters
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        
        # 5. Update parameters with Learning Rate Decay
        # The learning rate will smoothly shrink by 2% every iteration
        #current_lr = lr * (0.98 ** t)
        #theta_next = theta_i - current_lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        # Check for convergence (if step size is incredibly small)
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            loss_func_estimator(theta_i, silent=False)
            break
            
        theta_i = theta_next
        
        # Run non-silently to log the official step taken
        loss_func_estimator(theta_i, silent=False)
        
    return theta_i

# Execute the Adam Optimizer
# Note: Adam learning rates (lr) usually range between 0.001 and 0.1 depending on the landscape
best_params = adam_optimize(
    loss_fn=silent_loss_func, 
    theta0=initial_params,
    lr=0.04,        # Adjust this! Higher = faster but might overshoot
    max_iter=100
)

print("\nAdam Optimization Complete!")

# Optional: Save the new Adam parameters
np.save("pce_optimal_theta_adam_reps4qbt12Continue.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

best_cut = 0
best_bits = []

for edge0, edge1 in graph.edges():
    swapped_bits = cur_bits.copy()
    swapped_bits[edge0], swapped_bits[edge1] = swapped_bits[edge1], swapped_bits[edge0]
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if best_cut < cut_size:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue.npy...
Starting optimization with Adam (lr=0.05)...
Iter 1: -1248.360280

--- Adam Step 1/100 ---
Iter 2: -1203.633612

--- Adam Step 2/100 ---
Iter 3: -1238.954974

--- Adam Step 3/100 ---
Iter 4: -1243.302577

--- Adam Step 4/100 ---
Iter 5: -1234.499066

--- Adam Step 5/100 ---
Iter 6: -1236.505250

--- Adam Step 6/100 ---
Iter 7: -1244.931600

--- Adam Step 7/100 ---
Iter 8: -1250.750703

--- Adam Step 8/100 ---
Iter 9: -1252.687875

--- Adam Step 9/100 ---
Iter 10: -1252.460151

--- Adam Step 10/100 ---
Iter 11: -1251.613602

--- Adam Step 11/100 ---
Iter 12: -1252.424318

--- Adam Step 12/100 ---
Iter 13: -1255.208951

--- Adam Step 13/100 ---
Iter 14: -1257.938718

--- Adam Step 14/100 ---
Iter 15: -1259.431565

--- Adam Step 15/100 ---
Iter 16: -1260.005508

--- Adam Step 16/100 ---
Iter 17: -1260.312211

--- Adam Step 17/100 ---
Iter 18: -1260.870457

--- Adam Step 18/100 ---
Iter 1

In [24]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations
from scipy.optimize import approx_fprime

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp, Statevector
# Using the local V2 Estimator simulator instead of IBM Runtime
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 9
k = 3

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        paulis[c[0]], paulis[c[1]], paulis[c[2]] = pauli, pauli, pauli
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
#estimator = Estimator()

ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])
estimator = EstimatorV2()

# Highly recommended: Set the number of shots. 
# Finite shots introduce noise, so you need a high number for stable gradients.
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. THE NON-LINEAR LOSS FUNCTION
# ==========================================
alpha_loss = 3
beta_loss = 0.5
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def loss_func_estimator(x, silent=False):
    job = estimator.run([
        (ansatz, observables_x, x),
        (ansatz, observables_y, x),
        (ansatz, observables_z, x)
    ])
    result = job.result()
    
    node_exp_map = {}
    idx = 0
    for r in result:
        for ev in r.data.evs:
            node_exp_map[idx] = ev
            idx += 1
            
    loss = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss += w * np.tanh(alpha_loss * node_exp_map[edge0]) * np.tanh(alpha_loss * node_exp_map[edge1])
        
    regulation_term = 0
    for i in range(num_nodes):
        regulation_term += np.tanh(alpha_loss * node_exp_map[i])**2
        
    regulation_term = regulation_term / num_nodes
    regulation_term = beta_loss * v * (regulation_term**2)
    loss = loss + regulation_term
    
    if not silent:
        experiment_result.append({"loss": loss, "exp_map": node_exp_map})
        print(f"Iter {len(experiment_result)}: {loss:.6f}")
        
    return loss

def silent_loss_func(x):
    """Wrapper to prevent terminal spam during gradient calculation"""
    return loss_func_estimator(x, silent=True)

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue.npy"

#warm_start_file = "pce_optimal_theta.npy" # or whatever you named your Adam output
num_new_params = ansatz.num_parameters # For reps=4, this will be 90

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        # Perfect match (e.g., if you run reps=4 again later)
        initial_params = old_params
        
    elif num_old_params < num_new_params:
        # PADDING LOGIC: Transferring from a shallower circuit to a deeper one
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        
        # Initialize the new array with tiny random noise (e.g., between -0.01 and 0.01)
        # We use tiny noise instead of perfect 0.0 to prevent barren plateaus in the new layers
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        
        # Inject the highly optimized parameters into the beginning of the circuit
        initial_params[:num_old_params] = old_params
        
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)
# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize(loss_fn, theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    """Custom Adam optimizer loop using Euclidean gradients."""
    print(f"Starting optimization with Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    # Initialize momentum (m) and velocity (v) vectors
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    # Do an initial non-silent run to register the first point
    _ = loss_func_estimator(theta_i, silent=False)
    
    for i in range(max_iter):
        print(f"\n--- Adam Step {i+1}/{max_iter} ---")
        t += 1
        
        # 1. Calculate the Euclidean gradient (quietly)
        grad = approx_fprime(theta_i, silent_loss_func, epsilon=1e-5)
        
        # 2. Update biased first moment estimate (momentum)
        m_t = beta1 * m_t + (1 - beta1) * grad
        
        # 3. Update biased second raw moment estimate (velocity)
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        # 4. Compute bias-corrected estimates
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        # 5. Update parameters
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        
        # 5. Update parameters with Learning Rate Decay
        # The learning rate will smoothly shrink by 2% every iteration
        #current_lr = lr * (0.98 ** t)
        #theta_next = theta_i - current_lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        # Check for convergence (if step size is incredibly small)
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            loss_func_estimator(theta_i, silent=False)
            break
            
        theta_i = theta_next
        
        # Run non-silently to log the official step taken
        loss_func_estimator(theta_i, silent=False)
        
    return theta_i

# Execute the Adam Optimizer
# Note: Adam learning rates (lr) usually range between 0.001 and 0.1 depending on the landscape
best_params = adam_optimize(
    loss_fn=silent_loss_func, 
    theta0=initial_params,
    lr=0.2,        # Adjust this! Higher = faster but might overshoot
    max_iter=100
)

print("\nAdam Optimization Complete!")

# Optional: Save the new Adam parameters
np.save("pce_optimal_theta_adam_reps4qbt12Continue.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

best_cut = 0
best_bits = []

for edge0, edge1 in graph.edges():
    swapped_bits = cur_bits.copy()
    swapped_bits[edge0], swapped_bits[edge1] = swapped_bits[edge1], swapped_bits[edge0]
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if best_cut < cut_size:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue.npy...
Starting optimization with Adam (lr=0.2)...
Iter 1: -1297.175829

--- Adam Step 1/100 ---
Iter 2: -1010.991178

--- Adam Step 2/100 ---
Iter 3: -874.482577

--- Adam Step 3/100 ---
Iter 4: -964.216689

--- Adam Step 4/100 ---
Iter 5: -1002.635276

--- Adam Step 5/100 ---
Iter 6: -1083.174189

--- Adam Step 6/100 ---
Iter 7: -1125.753299

--- Adam Step 7/100 ---
Iter 8: -1133.842486

--- Adam Step 8/100 ---
Iter 9: -1155.885760

--- Adam Step 9/100 ---
Iter 10: -1156.037202

--- Adam Step 10/100 ---
Iter 11: -1165.507343

--- Adam Step 11/100 ---
Iter 12: -1205.091083

--- Adam Step 12/100 ---
Iter 13: -1215.150894

--- Adam Step 13/100 ---
Iter 14: -1209.115542

--- Adam Step 14/100 ---
Iter 15: -1229.496608

--- Adam Step 15/100 ---
Iter 16: -1245.145540

--- Adam Step 16/100 ---
Iter 17: -1240.466202

--- Adam Step 17/100 ---
Iter 18: -1250.830937

--- Adam Step 18/100 ---
Iter 19: 

In [25]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations
from scipy.optimize import approx_fprime

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp, Statevector
# Using the local V2 Estimator simulator instead of IBM Runtime
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 9
k = 3

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        paulis[c[0]], paulis[c[1]], paulis[c[2]] = pauli, pauli, pauli
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
#estimator = Estimator()

ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])
estimator = EstimatorV2()

# Highly recommended: Set the number of shots. 
# Finite shots introduce noise, so you need a high number for stable gradients.
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. THE NON-LINEAR LOSS FUNCTION
# ==========================================
alpha_loss = 3
beta_loss = 0.5
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def loss_func_estimator(x, silent=False):
    job = estimator.run([
        (ansatz, observables_x, x),
        (ansatz, observables_y, x),
        (ansatz, observables_z, x)
    ])
    result = job.result()
    
    node_exp_map = {}
    idx = 0
    for r in result:
        for ev in r.data.evs:
            node_exp_map[idx] = ev
            idx += 1
            
    loss = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss += w * np.tanh(alpha_loss * node_exp_map[edge0]) * np.tanh(alpha_loss * node_exp_map[edge1])
        
    regulation_term = 0
    for i in range(num_nodes):
        regulation_term += np.tanh(alpha_loss * node_exp_map[i])**2
        
    regulation_term = regulation_term / num_nodes
    regulation_term = beta_loss * v * (regulation_term**2)
    loss = loss + regulation_term
    
    if not silent:
        experiment_result.append({"loss": loss, "exp_map": node_exp_map})
        print(f"Iter {len(experiment_result)}: {loss:.6f}")
        
    return loss

def silent_loss_func(x):
    """Wrapper to prevent terminal spam during gradient calculation"""
    return loss_func_estimator(x, silent=True)

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue.npy"

#warm_start_file = "pce_optimal_theta.npy" # or whatever you named your Adam output
num_new_params = ansatz.num_parameters # For reps=4, this will be 90

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        # Perfect match (e.g., if you run reps=4 again later)
        initial_params = old_params
        
    elif num_old_params < num_new_params:
        # PADDING LOGIC: Transferring from a shallower circuit to a deeper one
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        
        # Initialize the new array with tiny random noise (e.g., between -0.01 and 0.01)
        # We use tiny noise instead of perfect 0.0 to prevent barren plateaus in the new layers
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        
        # Inject the highly optimized parameters into the beginning of the circuit
        initial_params[:num_old_params] = old_params
        
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)
# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize(loss_fn, theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    """Custom Adam optimizer loop using Euclidean gradients."""
    print(f"Starting optimization with Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    # Initialize momentum (m) and velocity (v) vectors
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    # Do an initial non-silent run to register the first point
    _ = loss_func_estimator(theta_i, silent=False)
    
    for i in range(max_iter):
        print(f"\n--- Adam Step {i+1}/{max_iter} ---")
        t += 1
        
        # 1. Calculate the Euclidean gradient (quietly)
        grad = approx_fprime(theta_i, silent_loss_func, epsilon=1e-5)
        
        # 2. Update biased first moment estimate (momentum)
        m_t = beta1 * m_t + (1 - beta1) * grad
        
        # 3. Update biased second raw moment estimate (velocity)
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        # 4. Compute bias-corrected estimates
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        # 5. Update parameters
        #theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        # 4. Compute bias-corrected estimates
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        adam_direction = m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        # 5. The Armijo Line Search for maximum safe step size
        step_size = 1.0 # Start with a massive step
        alpha_armijo = 0.01
        
        for k in range(6): # Max 6 halving attempts
            theta_next = theta_i - step_size * adam_direction
            f_next = silent_loss_func(theta_next)
            
            # If the loss drops sufficiently, keep this massive step!
            if _ - f_next >= alpha_armijo * step_size * np.linalg.norm(adam_direction)**2:
                break
            
            step_size /= 2.0 # Otherwise, cut step in half and try again
        
        # 5. Update parameters with Learning Rate Decay
        # The learning rate will smoothly shrink by 2% every iteration
        #current_lr = lr * (0.98 ** t)
        #theta_next = theta_i - current_lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        # Check for convergence (if step size is incredibly small)
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            loss_func_estimator(theta_i, silent=False)
            break
            
        theta_i = theta_next
        
        # Run non-silently to log the official step taken
        loss_func_estimator(theta_i, silent=False)
        
    return theta_i

# Execute the Adam Optimizer
# Note: Adam learning rates (lr) usually range between 0.001 and 0.1 depending on the landscape
best_params = adam_optimize(
    loss_fn=silent_loss_func, 
    theta0=initial_params,
    lr=0.2,        # Adjust this! Higher = faster but might overshoot
    max_iter=100
)

print("\nAdam Optimization Complete!")

# Optional: Save the new Adam parameters
np.save("pce_optimal_theta_adam_reps4qbt12Continue.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

best_cut = 0
best_bits = []

for edge0, edge1 in graph.edges():
    swapped_bits = cur_bits.copy()
    swapped_bits[edge0], swapped_bits[edge1] = swapped_bits[edge1], swapped_bits[edge0]
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if best_cut < cut_size:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue.npy...
Starting optimization with Adam (lr=0.2)...
Iter 1: -1334.147006

--- Adam Step 1/100 ---
Iter 2: -1310.320953

--- Adam Step 2/100 ---
Iter 3: -1324.056708

--- Adam Step 3/100 ---
Iter 4: -1325.957243

--- Adam Step 4/100 ---
Iter 5: -1323.691146

--- Adam Step 5/100 ---
Iter 6: -1324.541788

--- Adam Step 6/100 ---
Iter 7: -1329.390822

--- Adam Step 7/100 ---
Iter 8: -1331.608383

--- Adam Step 8/100 ---
Iter 9: -1330.653016

--- Adam Step 9/100 ---
Iter 10: -1330.593882

--- Adam Step 10/100 ---
Iter 11: -1331.921767

--- Adam Step 11/100 ---
Iter 12: -1332.608530

--- Adam Step 12/100 ---
Iter 13: -1332.882809

--- Adam Step 13/100 ---
Iter 14: -1333.758517

--- Adam Step 14/100 ---
Iter 15: -1334.532556

--- Adam Step 15/100 ---
Iter 16: -1334.511253

--- Adam Step 16/100 ---
Iter 17: -1334.492816

--- Adam Step 17/100 ---
Iter 18: -1335.087549

--- Adam Step 18/100 ---
Iter 19

In [26]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations
from scipy.optimize import approx_fprime

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())
num_qubits = 9
k = 3

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        paulis[c[0]], paulis[c[1]], paulis[c[2]] = pauli, pauli, pauli
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. THE NON-LINEAR LOSS FUNCTION
# ==========================================
alpha_loss = 3
beta_loss = 0.5
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def loss_func_estimator(x, silent=False):
    job = estimator.run([
        (ansatz, observables_x, x),
        (ansatz, observables_y, x),
        (ansatz, observables_z, x)
    ])
    result = job.result()
    
    node_exp_map = {}
    idx = 0
    for r in result:
        for ev in r.data.evs:
            node_exp_map[idx] = ev
            idx += 1
            
    loss = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss += w * np.tanh(alpha_loss * node_exp_map[edge0]) * np.tanh(alpha_loss * node_exp_map[edge1])
        
    regulation_term = 0
    for i in range(num_nodes):
        regulation_term += np.tanh(alpha_loss * node_exp_map[i])**2
        
    regulation_term = regulation_term / num_nodes
    regulation_term = beta_loss * v * (regulation_term**2)
    loss = loss + regulation_term
    
    if not silent:
        experiment_result.append({"loss": loss, "exp_map": node_exp_map})
        print(f"Iter {len(experiment_result)}: {loss:.6f}")
        
    return loss

def silent_loss_func(x):
    return loss_func_estimator(x, silent=True)

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue.npy"
num_new_params = ansatz.num_parameters 

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

# ==========================================
# 6. CUSTOM ADAM OPTIMIZER (WITH TRACKING & CHECKPOINTS)
# ==========================================
def adam_optimize(loss_fn, theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=500):
    print(f"Starting optimization with Adam + Armijo (lr={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    # Track the absolute best parameters
    best_overall_loss = float('inf')
    best_overall_theta = np.copy(theta_i)
    
    # Register the first point properly
    current_loss = loss_func_estimator(theta_i, silent=False)
    
    if current_loss < best_overall_loss:
        best_overall_loss = current_loss
        best_overall_theta = np.copy(theta_i)
    
    for i in range(1, max_iter + 1):
        print(f"\n--- Adam Step {i}/{max_iter} ---")
        t += 1
        
        grad = approx_fprime(theta_i, silent_loss_func, epsilon=1e-5)
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        adam_direction = m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        # Armijo Line Search
        step_size = 1.0 
        alpha_armijo = 0.01
        
        for k in range(6): 
            theta_next = theta_i - step_size * adam_direction
            f_next = silent_loss_func(theta_next)
            
            # Use current_loss to evaluate drop correctly
            if current_loss - f_next >= alpha_armijo * step_size * np.linalg.norm(adam_direction)**2:
                break
            
            step_size /= 2.0 
        
        # Convergence check
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            current_loss = loss_func_estimator(theta_i, silent=False)
            if current_loss < best_overall_loss:
                best_overall_loss = current_loss
                best_overall_theta = np.copy(theta_i)
            break
            
        theta_i = theta_next
        current_loss = f_next # Update loss memory for the next loop
        
        # Run non-silently to log the official step taken
        loss_func_estimator(theta_i, silent=False)
        
        # Track the best solution found so far
        if current_loss < best_overall_loss:
            best_overall_loss = current_loss
            best_overall_theta = np.copy(theta_i)
            
        # CHECKPOINTING: Save every 100 iterations
        if i % 100 == 0:
            chkpt_filename = f"pce_optimal_theta_adam_chkpt_{i}.npy"
            np.save(chkpt_filename, best_overall_theta)
            print(f">>> CHECKPOINT SAVED: {chkpt_filename} | Best Loss: {best_overall_loss:.6f} <<<")
            
    print(f"\nAdam Optimization Complete! Absolute Best Loss Found: {best_overall_loss:.6f}")
    return best_overall_theta # Return the BEST parameters, not just the last ones!

# Execute the Adam Optimizer
best_params = adam_optimize(
    loss_fn=silent_loss_func, 
    theta0=initial_params,
    lr=0.2,        
    max_iter=700  # Running for a large number of iterations
)

# Save the absolute best final parameters
np.save("pce_optimal_theta_adam_FINAL.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
# (Crucial: The Bit-swap processing will now decode the BEST parameters found, 
# not just whatever happened to be the last step of the loop.)

# We need to run the estimator one final time with best_params so experiment_result[-1] is accurate
print("Loading absolute best parameters for final bit-swap decoding...")
_ = loss_func_estimator(best_params, silent=False)

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

best_cut = 0
best_bits = []

for edge0, edge1 in graph.edges():
    swapped_bits = cur_bits.copy()
    swapped_bits[edge0], swapped_bits[edge1] = swapped_bits[edge1], swapped_bits[edge0]
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if best_cut < cut_size:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue.npy...
Starting optimization with Adam + Armijo (lr=0.2)...
Iter 1: -1388.275076

--- Adam Step 1/700 ---
Iter 2: -1401.877302

--- Adam Step 2/700 ---
Iter 3: -1397.092691

--- Adam Step 3/700 ---
Iter 4: -1404.136036

--- Adam Step 4/700 ---
Iter 5: -1407.566506

--- Adam Step 5/700 ---
Iter 6: -1410.646940

--- Adam Step 6/700 ---
Iter 7: -1411.241088

--- Adam Step 7/700 ---
Iter 8: -1411.641112

--- Adam Step 8/700 ---
Iter 9: -1412.468041

--- Adam Step 9/700 ---
Iter 10: -1415.464766

--- Adam Step 10/700 ---
Iter 11: -1415.502797

--- Adam Step 11/700 ---
Iter 12: -1416.205371

--- Adam Step 12/700 ---
Iter 13: -1416.768510

--- Adam Step 13/700 ---
Iter 14: -1417.213253

--- Adam Step 14/700 ---
Iter 15: -1417.845490

--- Adam Step 15/700 ---
Iter 16: -1417.857021

--- Adam Step 16/700 ---
Iter 17: -1418.388287

--- Adam Step 17/700 ---
Iter 18: -1418.594477

--- Adam Step 18/700 --

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations
from scipy.optimize import approx_fprime

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())
num_qubits = 9
k = 3

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        paulis[c[0]], paulis[c[1]], paulis[c[2]] = pauli, pauli, pauli
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=6)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. THE NON-LINEAR LOSS FUNCTION
# ==========================================
alpha_loss = 3
beta_loss = 0.5
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def loss_func_estimator(x, silent=False):
    job = estimator.run([
        (ansatz, observables_x, x),
        (ansatz, observables_y, x),
        (ansatz, observables_z, x)
    ])
    result = job.result()
    
    node_exp_map = {}
    idx = 0
    for r in result:
        for ev in r.data.evs:
            node_exp_map[idx] = ev
            idx += 1
            
    loss = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss += w * np.tanh(alpha_loss * node_exp_map[edge0]) * np.tanh(alpha_loss * node_exp_map[edge1])
        
    regulation_term = 0
    for i in range(num_nodes):
        regulation_term += np.tanh(alpha_loss * node_exp_map[i])**2
        
    regulation_term = regulation_term / num_nodes
    regulation_term = beta_loss * v * (regulation_term**2)
    loss = loss + regulation_term
    
    if not silent:
        experiment_result.append({"loss": loss, "exp_map": node_exp_map})
        print(f"Iter {len(experiment_result)}: {loss:.6f}")
        
    return loss

def silent_loss_func(x):
    return loss_func_estimator(x, silent=True)

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_chkpt_200.npy"
num_new_params = ansatz.num_parameters 

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

# ==========================================
# 6. CUSTOM ADAM OPTIMIZER (WITH TRACKING & CHECKPOINTS)
# ==========================================
def adam_optimize(loss_fn, theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=500):
    print(f"Starting optimization with Adam + Armijo (lr={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    # Track the absolute best parameters
    best_overall_loss = float('inf')
    best_overall_theta = np.copy(theta_i)
    
    # Register the first point properly
    current_loss = loss_func_estimator(theta_i, silent=False)
    
    if current_loss < best_overall_loss:
        best_overall_loss = current_loss
        best_overall_theta = np.copy(theta_i)
    
    for i in range(1, max_iter + 1):
        print(f"\n--- Adam Step {i}/{max_iter} ---")
        t += 1
        
        grad = approx_fprime(theta_i, silent_loss_func, epsilon=1e-5)
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        adam_direction = m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        # Armijo Line Search
        step_size = 1.0 
        alpha_armijo = 0.01
        
        for k in range(6): 
            theta_next = theta_i - step_size * adam_direction
            f_next = silent_loss_func(theta_next)
            
            # Use current_loss to evaluate drop correctly
            if current_loss - f_next >= alpha_armijo * step_size * np.linalg.norm(adam_direction)**2:
                break
            
            step_size /= 2.0 
        
        # Convergence check
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            current_loss = loss_func_estimator(theta_i, silent=False)
            if current_loss < best_overall_loss:
                best_overall_loss = current_loss
                best_overall_theta = np.copy(theta_i)
            break
            
        theta_i = theta_next
        current_loss = f_next # Update loss memory for the next loop
        
        # Run non-silently to log the official step taken
        loss_func_estimator(theta_i, silent=False)
        
        # Track the best solution found so far
        if current_loss < best_overall_loss:
            best_overall_loss = current_loss
            best_overall_theta = np.copy(theta_i)
            
        # CHECKPOINTING: Save every 100 iterations
        if i % 100 == 0:
            chkpt_filename = f"pce_optimal_theta_adam_chkpt2_{i}.npy"
            np.save(chkpt_filename, best_overall_theta)
            print(f">>> CHECKPOINT SAVED: {chkpt_filename} | Best Loss: {best_overall_loss:.6f} <<<")
            
    print(f"\nAdam Optimization Complete! Absolute Best Loss Found: {best_overall_loss:.6f}")
    return best_overall_theta # Return the BEST parameters, not just the last ones!

# Execute the Adam Optimizer
best_params = adam_optimize(
    loss_fn=silent_loss_func, 
    theta0=initial_params,
    lr=0.2,        
    max_iter=700  # Running for a large number of iterations
)

# Save the absolute best final parameters
np.save("pce_optimal_theta_adam_FINAL.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
# (Crucial: The Bit-swap processing will now decode the BEST parameters found, 
# not just whatever happened to be the last step of the loop.)

# We need to run the estimator one final time with best_params so experiment_result[-1] is accurate
print("Loading absolute best parameters for final bit-swap decoding...")
_ = loss_func_estimator(best_params, silent=False)

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

best_cut = 0
best_bits = []

for edge0, edge1 in graph.edges():
    swapped_bits = cur_bits.copy()
    swapped_bits[edge0], swapped_bits[edge1] = swapped_bits[edge1], swapped_bits[edge0]
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if best_cut < cut_size:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_chkpt_200.npy...
Padding parameters! Transferring 90 angles into a 126 angle circuit.
Starting optimization with Adam + Armijo (lr=0.2)...
Iter 1: -224.524643

--- Adam Step 1/700 ---
Iter 2: -309.997041

--- Adam Step 2/700 ---
Iter 3: -320.252785

--- Adam Step 3/700 ---
Iter 4: -445.673035

--- Adam Step 4/700 ---
Iter 5: -579.325223

--- Adam Step 5/700 ---
Iter 6: -698.465920

--- Adam Step 6/700 ---
Iter 7: -723.660331

--- Adam Step 7/700 ---
Iter 8: -725.668103

--- Adam Step 8/700 ---
Iter 9: -774.455768

--- Adam Step 9/700 ---
Iter 10: -799.036633

--- Adam Step 10/700 ---
Iter 11: -829.888967

--- Adam Step 11/700 ---
Iter 12: -866.377989

--- Adam Step 12/700 ---
Iter 13: -884.482848

--- Adam Step 13/700 ---
Iter 14: -897.340058

--- Adam Step 14/700 ---
Iter 15: -902.808090

--- Adam Step 15/700 ---
Iter 16: -926.234218

--- Adam Step 16/700 ---
Iter 17: -970.606077

--- Adam Step 17/700 ---
Ite

In [1]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp

# Explicitly using EstimatorV2 from Qiskit Aer
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# Import SPSA
from qiskit_algorithms.optimizers import SPSA

# ==========================================
# 1. LOAD GRAPH FROM PARQUET (For dimensions)
# ==========================================
print("Loading dataset dimensions...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # 180
num_qubits = 9
k = 3

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        paulis[c[0]], paulis[c[1]], paulis[c[2]] = pauli, pauli, pauli
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATORV2 SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

# Initialize Aer EstimatorV2
estimator = EstimatorV2()

experiment_result = []

# ==========================================
# 4. LOAD CLASSICAL TARGET & DEFINE MSE LOSS
# ==========================================
target_file = "classical_target_bitstring.npy"

if not os.path.exists(target_file):
    raise FileNotFoundError(f"Could not find {target_file}. Run the classical NetworkX script first!")

classical_bitstring = np.load(target_file)
print("Successfully loaded classical target bitstring.")

# Convert 0s and 1s into quantum expectation targets of -1.0 and 1.0
target_expectations = np.array([1.0 if bit == 1 else -1.0 for bit in classical_bitstring])

def pretrain_mse_loss(x, silent=True):
    # EstimatorV2 uses Primitive Unified Blocs (PUBs)
    job = estimator.run([
        (ansatz, observables_x, x),
        (ansatz, observables_y, x),
        (ansatz, observables_z, x)
    ])
    result = job.result()
    
    idx = 0
    node_exp_map = {}
    
    # Iterate through the V2 PubResults
    for r in result:
        for ev in r.data.evs:
            node_exp_map[idx] = ev
            idx += 1
            
    # Calculate Mean Squared Error (MSE)
    mse_loss = 0
    for i in range(num_nodes):
        mse_loss += (node_exp_map[i] - target_expectations[i]) ** 2
        
    mse_loss = mse_loss / num_nodes
    
    # Track the loss history for our final evaluation check
    experiment_result.append(mse_loss)
    
    if not silent:
        print(f"Evaluation {len(experiment_result)}: MSE Loss = {mse_loss:.6f}")
        
    return mse_loss

# SPSA expects a function that takes only `x`. This wrapper keeps it silent during eval.
def silent_objective(x):
    return pretrain_mse_loss(x, silent=True)

# ==========================================
# 5. SPSA OPTIMIZER & CALLBACK
# ==========================================
print("\nStarting Supervised Pre-training with SPSA...")

# Track the best absolute loss found by SPSA
best_loss = float('inf')
best_theta = None

# SPSA Callback function to cleanly print progress per step (instead of per evaluation)
def spsa_callback(nfev, x_next, fx_next, stepsize, accepted):
    global best_loss, best_theta
    step = int(nfev / 2)
    
    # Update our absolute best tracker
    if fx_next < best_loss:
        best_loss = fx_next
        best_theta = np.copy(x_next)
        
    print(f"--- SPSA Step {step} --- | Current MSE: {fx_next:.6f} | Best MSE: {best_loss:.6f}")

# Initialize SPSA. 
# We run 300 iterations (which will be 600 circuit evaluations) to let it dig deep.
spsa = SPSA(maxiter=300, callback=spsa_callback)

# ==========================================
# 6. EXECUTE & SAVE WARM-START PARAMETERS
# ==========================================
np.random.seed(42)
initial_guess = np.random.rand(ansatz.num_parameters)

# Run the optimization
result = spsa.minimize(
    fun=silent_objective, 
    x0=initial_guess
)

print(f"\n✅ SPSA Pre-training Complete! Absolute Best MSE Found: {best_loss:.6f}")

if best_loss < 0.1:
    print("The quantum circuit has successfully memorized the classical cut!")
else:
    print("⚠️ Warning: The MSE is slightly high. The circuit may need more 'reps' to fully express this specific cut.")

save_filename = "pce_optimal_theta_CLASSICAL_WARMSTART.npy"
# Save the BEST parameters found, not necessarily the last ones
np.save(save_filename, best_theta)
print(f"\nSaved parameters to '{save_filename}'.")
print("You can now feed this file into your main VQE script!")

Loading dataset dimensions...
Successfully loaded classical target bitstring.

Starting Supervised Pre-training with SPSA...
--- SPSA Step 1 --- | Current MSE: 1.004354 | Best MSE: 1.004354
--- SPSA Step 3 --- | Current MSE: 0.992980 | Best MSE: 0.992980
--- SPSA Step 4 --- | Current MSE: 0.993146 | Best MSE: 0.992980
--- SPSA Step 6 --- | Current MSE: 0.993086 | Best MSE: 0.992980
--- SPSA Step 7 --- | Current MSE: 0.983524 | Best MSE: 0.983524
--- SPSA Step 9 --- | Current MSE: 0.981520 | Best MSE: 0.981520
--- SPSA Step 10 --- | Current MSE: 0.981502 | Best MSE: 0.981502
--- SPSA Step 12 --- | Current MSE: 0.981205 | Best MSE: 0.981205
--- SPSA Step 13 --- | Current MSE: 0.981246 | Best MSE: 0.981205
--- SPSA Step 15 --- | Current MSE: 0.979854 | Best MSE: 0.979854
--- SPSA Step 16 --- | Current MSE: 0.981273 | Best MSE: 0.979854
--- SPSA Step 18 --- | Current MSE: 0.984517 | Best MSE: 0.979854
--- SPSA Step 19 --- | Current MSE: 0.985015 | Best MSE: 0.979854
--- SPSA Step 21 --- | 

In [5]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())
num_qubits = 9
k = 3

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        paulis[c[0]], paulis[c[1]], paulis[c[2]] = pauli, pauli, pauli
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. THE NON-LINEAR LOSS FUNCTION
# ==========================================
alpha_loss = 3
beta_loss = 0.5
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def loss_func_estimator(x, silent=False, return_exp=False):
    job = estimator.run([
        (ansatz, observables_x, x),
        (ansatz, observables_y, x),
        (ansatz, observables_z, x)
    ])
    result = job.result()
    
    # Vectorized extraction of expectation values
    node_exp_map = np.concatenate([
        result[0].data.evs,
        result[1].data.evs,
        result[2].data.evs
    ])
    
    # Vectorized calculation of the Tanh values
    T = np.tanh(alpha_loss * node_exp_map)
    
    loss = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss += w * T[edge0] * T[edge1]
        
    R = np.sum(T**2) / num_nodes
    regulation_term = beta_loss * v * (R**2)
    loss += regulation_term
    
    if not silent:
        # Reconstruct the dictionary for compatibility with the final decoding step
        exp_dict = {i: node_exp_map[i] for i in range(num_nodes)}
        experiment_result.append({"loss": loss, "exp_map": exp_dict})
        print(f"Iter {len(experiment_result)}: {loss:.6f}")
        
    if return_exp:
        return loss, node_exp_map
    return loss

def silent_loss_func(x):
    return loss_func_estimator(x, silent=True)

# ==========================================
# 5. CHAIN RULE & PARAMETER SHIFT ENGINE
# ==========================================
def compute_dL_dE(E_vals):
    """Classical analytical derivative of the loss function w.r.t expectation values"""
    T = np.tanh(alpha_loss * E_vals)
    dT_dE = alpha_loss * (1.0 - T**2) # Derivative of tanh(alpha * E)

    dL_dT = np.zeros(num_nodes)
    
    # 1. Derivative of the Max-Cut edges
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        dL_dT[edge0] += w * T[edge1]
        dL_dT[edge1] += w * T[edge0]

    # 2. Derivative of the Regularization term
    R = np.sum(T**2) / num_nodes
    dL_dT += (4.0 * beta_loss * v / num_nodes) * R * T

    # Chain rule: dL/dE = dL/dT * dT/dE
    return dL_dT * dT_dE

def compute_gradient_param_shift(x, dL_dE):
    """Calculates exactly dE/dTheta using quantum shifts, and applies classical chain rule."""
    grad = np.zeros(len(x))
    shift = np.pi / 2.0
    pubs = []
    
    # Bundle all shifted parameters into one massive job for extreme efficiency
    for i in range(len(x)):
        x_plus = x.copy()
        x_plus[i] += shift
        x_minus = x.copy()
        x_minus[i] -= shift
        
        pubs.extend([
            (ansatz, observables_x, x_plus),
            (ansatz, observables_y, x_plus),
            (ansatz, observables_z, x_plus),
            (ansatz, observables_x, x_minus),
            (ansatz, observables_y, x_minus),
            (ansatz, observables_z, x_minus)
        ])
        
    job = estimator.run(pubs)
    result = job.result()
    
    for i in range(len(x)):
        base_idx = i * 6
        # Combine the 3 sets of observables for the +pi/2 shift
        E_plus = np.concatenate([
            result[base_idx].data.evs,
            result[base_idx+1].data.evs,
            result[base_idx+2].data.evs
        ])
        # Combine the 3 sets of observables for the -pi/2 shift
        E_minus = np.concatenate([
            result[base_idx+3].data.evs,
            result[base_idx+4].data.evs,
            result[base_idx+5].data.evs
        ])
        
        # Parameter shift formula: 0.5 * (E_plus - E_minus)
        dE_dtheta = 0.5 * (E_plus - E_minus)
        
        # Final Chain Rule Application: dot product of dL/dE and dE/dtheta
        grad[i] = np.dot(dL_dE, dE_dtheta)
        
    return grad

# ==========================================
# 6. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue.npy"
num_new_params = ansatz.num_parameters 

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

# ==========================================
# 7. EXACT ADAM OPTIMIZER (WITH TRACKING & CHECKPOINTS)
# ==========================================
def adam_optimize(theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=500):
    print(f"Starting exact analytical optimization with Adam + Armijo (lr={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    best_overall_loss = float('inf')
    best_overall_theta = np.copy(theta_i)
    
    current_loss = loss_func_estimator(theta_i, silent=False)
    if current_loss < best_overall_loss:
        best_overall_loss = current_loss
        best_overall_theta = np.copy(theta_i)
    
    for i in range(1, max_iter + 1):
        print(f"\n--- Adam Step {i}/{max_iter} ---")
        t += 1
        
        # --- THE EXACT GRADIENT ENGINE ---
        # 1. Measure the current expectation values
        _, current_E_vals = loss_func_estimator(theta_i, silent=True, return_exp=True)
        # 2. Classically calculate the exact derivative of the loss function
        dL_dE = compute_dL_dE(current_E_vals)
        # 3. Apply the parameter shift rule on the quantum chip
        grad = compute_gradient_param_shift(theta_i, dL_dE)
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        adam_direction = m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        # Armijo Line Search
        step_size = 1.0 
        alpha_armijo = 0.01
        
        for k in range(6): 
            theta_next = theta_i - step_size * adam_direction
            f_next = silent_loss_func(theta_next)
            
            if current_loss - f_next >= alpha_armijo * step_size * np.linalg.norm(adam_direction)**2:
                break
            
            step_size /= 2.0 
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            current_loss = loss_func_estimator(theta_i, silent=False)
            if current_loss < best_overall_loss:
                best_overall_loss = current_loss
                best_overall_theta = np.copy(theta_i)
            break
            
        theta_i = theta_next
        current_loss = f_next 
        
        # Log the official step taken
        loss_func_estimator(theta_i, silent=False)
        
        if current_loss < best_overall_loss:
            best_overall_loss = current_loss
            best_overall_theta = np.copy(theta_i)
            
        if i % 100 == 0:
            chkpt_filename = f"pce_optimal_theta_adam_chkpt_{i}.npy"
            np.save(chkpt_filename, best_overall_theta)
            print(f">>> CHECKPOINT SAVED: {chkpt_filename} | Best Loss: {best_overall_loss:.6f} <<<")
            
    print(f"\nAdam Optimization Complete! Absolute Best Loss Found: {best_overall_loss:.6f}")
    return best_overall_theta 



# Execute the exact Adam Optimizer
best_params = adam_optimize(
    theta0=initial_params,
    lr=0.2,        
    max_iter=700 
)

np.save("pce_optimal_theta_adam_FINAL.npy", best_params)

# ==========================================
# 8. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
print("Loading absolute best parameters for final bit-swap decoding...")
_ = loss_func_estimator(best_params, silent=False)

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

best_cut = 0
best_bits = []

for edge0, edge1 in graph.edges():
    swapped_bits = cur_bits.copy()
    swapped_bits[edge0], swapped_bits[edge1] = swapped_bits[edge1], swapped_bits[edge0]
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if best_cut < cut_size:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue.npy...
Starting exact analytical optimization with Adam + Armijo (lr=0.2)...
Iter 1: -1388.275076

--- Adam Step 1/700 ---
Iter 2: -1401.877302

--- Adam Step 2/700 ---
Iter 3: -1397.092397

--- Adam Step 3/700 ---
Iter 4: -1404.137819

--- Adam Step 4/700 ---
Iter 5: -1407.567055

--- Adam Step 5/700 ---
Iter 6: -1410.646535

--- Adam Step 6/700 ---
Iter 7: -1411.241826

--- Adam Step 7/700 ---


KeyboardInterrupt: 

In [2]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())
num_qubits = 9
k = 3

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        paulis[c[0]], paulis[c[1]], paulis[c[2]] = pauli, pauli, pauli
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. THE NON-LINEAR LOSS FUNCTION
# ==========================================
alpha_loss = 3
beta_loss = 0.5
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def loss_func_estimator(x, silent=False, return_exp=False):
    job = estimator.run([
        (ansatz, observables_x, x),
        (ansatz, observables_y, x),
        (ansatz, observables_z, x)
    ])
    result = job.result()
    
    # Vectorized extraction of expectation values
    node_exp_map = np.concatenate([
        result[0].data.evs,
        result[1].data.evs,
        result[2].data.evs
    ])
    
    # Vectorized calculation of the Tanh values
    T = np.tanh(alpha_loss * node_exp_map)
    
    loss = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss += w * T[edge0] * T[edge1]
        
    R = np.sum(T**2) / num_nodes
    regulation_term = beta_loss * v * (R**2)
    loss += regulation_term
    
    if not silent:
        # Reconstruct the dictionary for compatibility with the final decoding step
        exp_dict = {i: node_exp_map[i] for i in range(num_nodes)}
        experiment_result.append({"loss": loss, "exp_map": exp_dict})
        print(f"Iter {len(experiment_result)}: {loss:.6f}")
        
    if return_exp:
        return loss, node_exp_map
    return loss

def silent_loss_func(x):
    return loss_func_estimator(x, silent=True)

# ==========================================
# 5. CHAIN RULE & PARAMETER SHIFT ENGINE
# ==========================================
def compute_dL_dE(E_vals):
    """Classical analytical derivative of the loss function w.r.t expectation values"""
    T = np.tanh(alpha_loss * E_vals)
    dT_dE = alpha_loss * (1.0 - T**2) # Derivative of tanh(alpha * E)

    dL_dT = np.zeros(num_nodes)
    
    # 1. Derivative of the Max-Cut edges
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        dL_dT[edge0] += w * T[edge1]
        dL_dT[edge1] += w * T[edge0]

    # 2. Derivative of the Regularization term
    R = np.sum(T**2) / num_nodes
    dL_dT += (4.0 * beta_loss * v / num_nodes) * R * T

    # Chain rule: dL/dE = dL/dT * dT/dE
    return dL_dT * dT_dE

def compute_gradient_param_shift(x, dL_dE):
    """Calculates exactly dE/dTheta using quantum shifts, and applies classical chain rule."""
    grad = np.zeros(len(x))
    shift = np.pi / 2.0
    pubs = []
    
    # Bundle all shifted parameters into one massive job for extreme efficiency
    for i in range(len(x)):
        x_plus = x.copy()
        x_plus[i] += shift
        x_minus = x.copy()
        x_minus[i] -= shift
        
        pubs.extend([
            (ansatz, observables_x, x_plus),
            (ansatz, observables_y, x_plus),
            (ansatz, observables_z, x_plus),
            (ansatz, observables_x, x_minus),
            (ansatz, observables_y, x_minus),
            (ansatz, observables_z, x_minus)
        ])
        
    job = estimator.run(pubs)
    result = job.result()
    
    for i in range(len(x)):
        base_idx = i * 6
        # Combine the 3 sets of observables for the +pi/2 shift
        E_plus = np.concatenate([
            result[base_idx].data.evs,
            result[base_idx+1].data.evs,
            result[base_idx+2].data.evs
        ])
        # Combine the 3 sets of observables for the -pi/2 shift
        E_minus = np.concatenate([
            result[base_idx+3].data.evs,
            result[base_idx+4].data.evs,
            result[base_idx+5].data.evs
        ])
        
        # Parameter shift formula: 0.5 * (E_plus - E_minus)
        dE_dtheta = 0.5 * (E_plus - E_minus)
        
        # Final Chain Rule Application: dot product of dL/dE and dE/dtheta
        grad[i] = np.dot(dL_dE, dE_dtheta)
        
    return grad

# ==========================================
# 6. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue.npy"
num_new_params = ansatz.num_parameters 

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

# ==========================================
# 7. EXACT ADAM OPTIMIZER (WITH COSINE DECAY & CHECKPOINTS)
# ==========================================
def adam_optimize(theta0, lr=0.1, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=500):
    print(f"Starting exact analytical optimization with Adam + Cosine Decay (Max LR={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    best_overall_loss = float('inf')
    best_overall_theta = np.copy(theta_i)
    
    current_loss = loss_func_estimator(theta_i, silent=False)
    if current_loss < best_overall_loss:
        best_overall_loss = current_loss
        best_overall_theta = np.copy(theta_i)
    
    for i in range(1, max_iter + 1):
        t += 1
        
        # Calculate Cosine Annealing Learning Rate
        lr_t = 0.005 + 0.5 * (lr - 0.005) * (1 + np.cos(np.pi * t / max_iter))
        
        print(f"\n--- Adam Step {i}/{max_iter} | LR: {lr_t:.6f} ---")
        
        # --- THE EXACT GRADIENT ENGINE ---
        # 1. Measure the current expectation values
        _, current_E_vals = loss_func_estimator(theta_i, silent=True, return_exp=True)
        # 2. Classically calculate the exact derivative of the loss function
        dL_dE = compute_dL_dE(current_E_vals)
        # 3. Apply the parameter shift rule on the quantum chip
        grad = compute_gradient_param_shift(theta_i, dL_dE)
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        adam_direction = m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        # Update with Cosine Annealed Step Size
        theta_next = theta_i - lr_t * adam_direction
        f_next = silent_loss_func(theta_next)
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            current_loss = loss_func_estimator(theta_i, silent=False)
            if current_loss < best_overall_loss:
                best_overall_loss = current_loss
                best_overall_theta = np.copy(theta_i)
            break
            
        theta_i = theta_next
        current_loss = f_next 
        
        # Log the official step taken
        loss_func_estimator(theta_i, silent=False)
        
        if current_loss < best_overall_loss:
            best_overall_loss = current_loss
            best_overall_theta = np.copy(theta_i)
            
        if i % 100 == 0:
            chkpt_filename = f"pce_optimal_theta_adam_chkpt_{i}.npy"
            np.save(chkpt_filename, best_overall_theta)
            print(f">>> CHECKPOINT SAVED: {chkpt_filename} | Best Loss: {best_overall_loss:.6f} <<<")
            
    print(f"\nAdam Optimization Complete! Absolute Best Loss Found: {best_overall_loss:.6f}")
    return best_overall_theta 

# Execute the exact Adam Optimizer
# Set max_lr to 0.1 or 0.2 depending on how aggressively you want to start
best_params = adam_optimize(
    theta0=initial_params,
    lr=0.1,        
    max_iter=700 
)

np.save("pce_optimal_theta_adam_FINAL.npy", best_params)

# ==========================================
# 8. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
print("Loading absolute best parameters for final bit-swap decoding...")
_ = loss_func_estimator(best_params, silent=False)

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

best_cut = 0
best_bits = []

for edge0, edge1 in graph.edges():
    swapped_bits = cur_bits.copy()
    swapped_bits[edge0], swapped_bits[edge1] = swapped_bits[edge1], swapped_bits[edge0]
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if best_cut < cut_size:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue.npy...
Starting exact analytical optimization with Adam + Cosine Decay (Max LR=0.1)...
Iter 1: -1388.275076

--- Adam Step 1/700 | LR: 0.100000 ---
Iter 2: -989.396243

--- Adam Step 2/700 | LR: 0.099998 ---
Iter 3: -1299.575038

--- Adam Step 3/700 | LR: 0.099996 ---
Iter 4: -1313.594431

--- Adam Step 4/700 | LR: 0.099992 ---
Iter 5: -1224.913105

--- Adam Step 5/700 | LR: 0.099988 ---
Iter 6: -1293.435664

--- Adam Step 6/700 | LR: 0.099983 ---
Iter 7: -1373.217929

--- Adam Step 7/700 | LR: 0.099977 ---
Iter 8: -1358.878092

--- Adam Step 8/700 | LR: 0.099969 ---
Iter 9: -1317.904085

--- Adam Step 9/700 | LR: 0.099961 ---
Iter 10: -1329.281509

--- Adam Step 10/700 | LR: 0.099952 ---
Iter 11: -1373.064291

--- Adam Step 11/700 | LR: 0.099942 ---
Iter 12: -1391.162030

--- Adam Step 12/700 | LR: 0.099931 ---
Iter 13: -1375.933635

--- Adam Step 13/700 | LR: 0.099919 ---
Iter 14: -1366.45

KeyboardInterrupt: 

In [4]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())
num_qubits = 9
k = 3

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        paulis[c[0]], paulis[c[1]], paulis[c[2]] = pauli, pauli, pauli
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. THE NON-LINEAR LOSS FUNCTION
# ==========================================
# ==========================================
# 4. THE NON-LINEAR LOSS FUNCTION (Frustration-Aware)
# ==========================================
alpha_loss = 3
beta_frust = 1.5 # New hyperparameter to tune the frustration penalty

def loss_func_estimator(x, silent=False, return_exp=False):
    job = estimator.run([
        (ansatz, observables_x, x),
        (ansatz, observables_y, x),
        (ansatz, observables_z, x)
    ])
    result = job.result()
    
    node_exp_map = np.concatenate([
        result[0].data.evs,
        result[1].data.evs,
        result[2].data.evs
    ])
    
    T = np.tanh(alpha_loss * node_exp_map)
    
    # 1. The Base Max-Cut Loss
    loss = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        # Optional: Apply degree normalization here if desired
        loss += w * T[edge0] * T[edge1]
        
    # 2. The Anti-Frustration Penalty
    # Sum of (1 - T^2). Peaks at T=0 (frustrated), drops to 0 at T=+-1 (decided)
    frustration_penalty = beta_frust * np.sum(1.0 - T**2)
    loss += frustration_penalty
    
    if not silent:
        exp_dict = {i: node_exp_map[i] for i in range(num_nodes)}
        experiment_result.append({"loss": loss, "exp_map": exp_dict})
        
        # Calculate how many nodes are highly frustrated (between -0.2 and 0.2)
        frustrated_count = np.sum(np.abs(T) < 0.2)
        print(f"Iter {len(experiment_result)}: Loss = {loss:.6f} | Frustrated Nodes: {frustrated_count}")
        
    if return_exp:
        return loss, node_exp_map
    return loss

# ==========================================
# 5. CHAIN RULE (Updated for Frustration Penalty)
# ==========================================
def compute_dL_dE(E_vals):
    """Classical analytical derivative of the frustration-aware loss function"""
    T = np.tanh(alpha_loss * E_vals)
    dT_dE = alpha_loss * (1.0 - T**2) 

    dL_dT = np.zeros(num_nodes)
    
    # 1. Derivative of the Max-Cut edges
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        dL_dT[edge0] += w * T[edge1]
        dL_dT[edge1] += w * T[edge0]

    # 2. Derivative of the Anti-Frustration penalty
    # The derivative of (1 - T^2) w.r.t T is exactly -2T
    dL_dT += beta_frust * (-2.0 * T)

    return dL_dT * dT_dE

def compute_gradient_param_shift(x, dL_dE):
    """Calculates exactly dE/dTheta using quantum shifts, and applies classical chain rule."""
    grad = np.zeros(len(x))
    shift = np.pi / 2.0
    pubs = []
    
    # Bundle all shifted parameters into one massive job for extreme efficiency
    for i in range(len(x)):
        x_plus = x.copy()
        x_plus[i] += shift
        x_minus = x.copy()
        x_minus[i] -= shift
        
        pubs.extend([
            (ansatz, observables_x, x_plus),
            (ansatz, observables_y, x_plus),
            (ansatz, observables_z, x_plus),
            (ansatz, observables_x, x_minus),
            (ansatz, observables_y, x_minus),
            (ansatz, observables_z, x_minus)
        ])
        
    job = estimator.run(pubs)
    result = job.result()
    
    for i in range(len(x)):
        base_idx = i * 6
        # Combine the 3 sets of observables for the +pi/2 shift
        E_plus = np.concatenate([
            result[base_idx].data.evs,
            result[base_idx+1].data.evs,
            result[base_idx+2].data.evs
        ])
        # Combine the 3 sets of observables for the -pi/2 shift
        E_minus = np.concatenate([
            result[base_idx+3].data.evs,
            result[base_idx+4].data.evs,
            result[base_idx+5].data.evs
        ])
        
        # Parameter shift formula: 0.5 * (E_plus - E_minus)
        dE_dtheta = 0.5 * (E_plus - E_minus)
        
        # Final Chain Rule Application: dot product of dL/dE and dE/dtheta
        grad[i] = np.dot(dL_dE, dE_dtheta)
        
    return grad

# ==========================================
# 6. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue.npy"
num_new_params = ansatz.num_parameters 

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

# ==========================================
# 7. EXACT ADAM OPTIMIZER (WITH COSINE DECAY & CHECKPOINTS)
# ==========================================
def adam_optimize(theta0, lr=0.1, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=500):
    print(f"Starting exact analytical optimization with Adam + Cosine Decay (Max LR={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    best_overall_loss = float('inf')
    best_overall_theta = np.copy(theta_i)
    
    current_loss = loss_func_estimator(theta_i, silent=False)
    if current_loss < best_overall_loss:
        best_overall_loss = current_loss
        best_overall_theta = np.copy(theta_i)
    
    for i in range(1, max_iter + 1):
        t += 1
        
        # Calculate Cosine Annealing Learning Rate
        lr_t = 0.005 + 0.5 * (lr - 0.005) * (1 + np.cos(np.pi * t / max_iter))
        
        print(f"\n--- Adam Step {i}/{max_iter} | LR: {lr_t:.6f} ---")
        
        # --- THE EXACT GRADIENT ENGINE ---
        # 1. Measure the current expectation values
        _, current_E_vals = loss_func_estimator(theta_i, silent=True, return_exp=True)
        # 2. Classically calculate the exact derivative of the loss function
        dL_dE = compute_dL_dE(current_E_vals)
        # 3. Apply the parameter shift rule on the quantum chip
        grad = compute_gradient_param_shift(theta_i, dL_dE)
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        adam_direction = m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        # Update with Cosine Annealed Step Size
        theta_next = theta_i - lr_t * adam_direction
        f_next = silent_loss_func(theta_next)
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            current_loss = loss_func_estimator(theta_i, silent=False)
            if current_loss < best_overall_loss:
                best_overall_loss = current_loss
                best_overall_theta = np.copy(theta_i)
            break
            
        theta_i = theta_next
        current_loss = f_next 
        
        # Log the official step taken
        loss_func_estimator(theta_i, silent=False)
        
        if current_loss < best_overall_loss:
            best_overall_loss = current_loss
            best_overall_theta = np.copy(theta_i)
            
        if i % 100 == 0:
            chkpt_filename = f"pce_optimal_theta_adam_chkpt_{i}.npy"
            np.save(chkpt_filename, best_overall_theta)
            print(f">>> CHECKPOINT SAVED: {chkpt_filename} | Best Loss: {best_overall_loss:.6f} <<<")
            
    print(f"\nAdam Optimization Complete! Absolute Best Loss Found: {best_overall_loss:.6f}")
    return best_overall_theta 

# Execute the exact Adam Optimizer
# Set max_lr to 0.1 or 0.2 depending on how aggressively you want to start
best_params = adam_optimize(
    theta0=initial_params,
    lr=0.1,        
    max_iter=700 
)

np.save("pce_optimal_theta_adam_FINAL.npy", best_params)

# ==========================================
# 8. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
print("Loading absolute best parameters for final bit-swap decoding...")
_ = loss_func_estimator(best_params, silent=False)

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

best_cut = 0
best_bits = []

for edge0, edge1 in graph.edges():
    swapped_bits = cur_bits.copy()
    swapped_bits[edge0], swapped_bits[edge1] = swapped_bits[edge1], swapped_bits[edge0]
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if best_cut < cut_size:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue.npy...
Starting exact analytical optimization with Adam + Cosine Decay (Max LR=0.1)...
Iter 1: Loss = -1226.385579 | Frustrated Nodes: 12

--- Adam Step 1/700 | LR: 0.100000 ---
Iter 2: Loss = -793.501385 | Frustrated Nodes: 32

--- Adam Step 2/700 | LR: 0.099998 ---
Iter 3: Loss = -1138.374584 | Frustrated Nodes: 22

--- Adam Step 3/700 | LR: 0.099996 ---
Iter 4: Loss = -1156.407529 | Frustrated Nodes: 22

--- Adam Step 4/700 | LR: 0.099992 ---
Iter 5: Loss = -1047.795848 | Frustrated Nodes: 20

--- Adam Step 5/700 | LR: 0.099988 ---
Iter 6: Loss = -1119.995342 | Frustrated Nodes: 13

--- Adam Step 6/700 | LR: 0.099983 ---
Iter 7: Loss = -1209.622745 | Frustrated Nodes: 8

--- Adam Step 7/700 | LR: 0.099977 ---
Iter 8: Loss = -1196.648424 | Frustrated Nodes: 7

--- Adam Step 8/700 | LR: 0.099969 ---
Iter 9: Loss = -1157.218065 | Frustrated Nodes: 10

--- Adam Step 9/700 | LR: 0.099961 ---
I

KeyboardInterrupt: 

In [5]:
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()

In [6]:
# Sum all the 'weight' attributes of the edges. 
# It defaults to 1.0 if an edge accidentally doesn't have a weight.
total_edge_weight = sum(data.get('weight', 1.0) for _, _, data in graph.edges(data=True))

print(f"Total sum of all edge weights: {total_edge_weight}")

Total sum of all edge weights: 0


In [7]:
# Get the total count of edges in the graph
total_edge_count = graph.number_of_edges()

print(f"Total number of edges: {total_edge_count}")

Total number of edges: 0


In [9]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations
from scipy.optimize import approx_fprime

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp, Statevector
# Using the local V2 Estimator simulator instead of IBM Runtime
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12
k = 2

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        
        # This loop makes it work for any k (e.g., k=2, k=3, k=4)
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
#estimator = Estimator()

ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])
estimator = EstimatorV2()

# Highly recommended: Set the number of shots. 
# Finite shots introduce noise, so you need a high number for stable gradients.
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. THE NON-LINEAR LOSS FUNCTION
# ==========================================
alpha_loss = 3
beta_loss = 0.5
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def loss_func_estimator(x, silent=False):
    job = estimator.run([
        (ansatz, observables_x, x),
        (ansatz, observables_y, x),
        (ansatz, observables_z, x)
    ])
    result = job.result()
    
    node_exp_map = {}
    idx = 0
    for r in result:
        for ev in r.data.evs:
            node_exp_map[idx] = ev
            idx += 1
            
    loss = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss += w * np.tanh(alpha_loss * node_exp_map[edge0]) * np.tanh(alpha_loss * node_exp_map[edge1])
        
    regulation_term = 0
    for i in range(num_nodes):
        regulation_term += np.tanh(alpha_loss * node_exp_map[i])**2
        
    regulation_term = regulation_term / num_nodes
    regulation_term = beta_loss * v * (regulation_term**2)
    loss = loss + regulation_term
    
    if not silent:
        experiment_result.append({"loss": loss, "exp_map": node_exp_map})
        print(f"Iter {len(experiment_result)}: {loss:.6f}")
        
    return loss

def silent_loss_func(x):
    """Wrapper to prevent terminal spam during gradient calculation"""
    return loss_func_estimator(x, silent=True)

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue.npy"

#warm_start_file = "pce_optimal_theta.npy" # or whatever you named your Adam output
num_new_params = ansatz.num_parameters # For reps=4, this will be 90

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        # Perfect match (e.g., if you run reps=4 again later)
        initial_params = old_params
        
    elif num_old_params < num_new_params:
        # PADDING LOGIC: Transferring from a shallower circuit to a deeper one
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        
        # Initialize the new array with tiny random noise (e.g., between -0.01 and 0.01)
        # We use tiny noise instead of perfect 0.0 to prevent barren plateaus in the new layers
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        
        # Inject the highly optimized parameters into the beginning of the circuit
        initial_params[:num_old_params] = old_params
        
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)
# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize(loss_fn, theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    """Custom Adam optimizer loop using Euclidean gradients."""
    print(f"Starting optimization with Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    # Initialize momentum (m) and velocity (v) vectors
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    # Do an initial non-silent run to register the first point
    _ = loss_func_estimator(theta_i, silent=False)
    
    for i in range(max_iter):
        print(f"\n--- Adam Step {i+1}/{max_iter} ---")
        t += 1
        
        # 1. Calculate the Euclidean gradient (quietly)
        grad = approx_fprime(theta_i, silent_loss_func, epsilon=1e-5)
        
        # 2. Update biased first moment estimate (momentum)
        m_t = beta1 * m_t + (1 - beta1) * grad
        
        # 3. Update biased second raw moment estimate (velocity)
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        # 4. Compute bias-corrected estimates
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        # 5. Update parameters
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        
        # 5. Update parameters with Learning Rate Decay
        # The learning rate will smoothly shrink by 2% every iteration
        #current_lr = lr * (0.98 ** t)
        #theta_next = theta_i - current_lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        # Check for convergence (if step size is incredibly small)
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            loss_func_estimator(theta_i, silent=False)
            break
            
        theta_i = theta_next
        
        # Run non-silently to log the official step taken
        loss_func_estimator(theta_i, silent=False)
        
    return theta_i

# Execute the Adam Optimizer
# Note: Adam learning rates (lr) usually range between 0.001 and 0.1 depending on the landscape
best_params = adam_optimize(
    loss_fn=silent_loss_func, 
    theta0=initial_params,
    lr=0.04,        # Adjust this! Higher = faster but might overshoot
    max_iter=100
)

print("\nAdam Optimization Complete!")

# Optional: Save the new Adam parameters
np.save("pce_optimal_theta_adam_reps4qbt12Continue.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

best_cut = 0
best_bits = []

for edge0, edge1 in graph.edges():
    swapped_bits = cur_bits.copy()
    swapped_bits[edge0], swapped_bits[edge1] = swapped_bits[edge1], swapped_bits[edge0]
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if best_cut < cut_size:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue.npy...
Padding parameters! Transferring 90 angles into a 120 angle circuit.
Starting optimization with Adam (lr=0.04)...
Iter 1: -6.636011

--- Adam Step 1/100 ---
Iter 2: -39.738594

--- Adam Step 2/100 ---
Iter 3: -67.247804

--- Adam Step 3/100 ---
Iter 4: -89.845718

--- Adam Step 4/100 ---
Iter 5: -109.765435

--- Adam Step 5/100 ---
Iter 6: -128.366351

--- Adam Step 6/100 ---
Iter 7: -146.921051

--- Adam Step 7/100 ---
Iter 8: -166.724891

--- Adam Step 8/100 ---
Iter 9: -188.875023

--- Adam Step 9/100 ---
Iter 10: -214.252312

--- Adam Step 10/100 ---
Iter 11: -243.498494

--- Adam Step 11/100 ---
Iter 12: -276.808501

--- Adam Step 12/100 ---
Iter 13: -314.300126

--- Adam Step 13/100 ---
Iter 14: -355.561167

--- Adam Step 14/100 ---
Iter 15: -399.497233

--- Adam Step 15/100 ---
Iter 16: -445.933190

--- Adam Step 16/100 ---
Iter 17: -495.146117

--- Adam Step 17/100 ---
Iter 18

In [10]:
def greedy_local_search(bits, graph, max_rounds=100):
    bits = list(bits)
    best_cut = calc_cut_size(graph, 
                              {i for i,b in enumerate(bits) if b==1},
                              {i for i,b in enumerate(bits) if b==0})
    improved = True
    rounds = 0
    while improved and rounds < max_rounds:
        improved = False
        rounds += 1
        for node in range(len(bits)):
            bits[node] ^= 1  # flip node
            p0 = {i for i,b in enumerate(bits) if b==1}
            p1 = {i for i,b in enumerate(bits) if b==0}
            new_cut = calc_cut_size(graph, p0, p1)
            if new_cut > best_cut:
                best_cut = new_cut
                improved = True
            else:
                bits[node] ^= 1  # flip back
    return bits, best_cut

# Apply AFTER your current decoder
best_bits_greedy, best_cut_greedy = greedy_local_search(best_bits, graph)
print(f"After greedy local search: {best_cut_greedy}")

After greedy local search: 6687.214980535943


In [11]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations
from scipy.optimize import approx_fprime

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp, Statevector
# Using the local V2 Estimator simulator instead of IBM Runtime
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12
k = 2

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        
        # This loop makes it work for any k (e.g., k=2, k=3, k=4)
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
#estimator = Estimator()

ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])
estimator = EstimatorV2()

# Highly recommended: Set the number of shots. 
# Finite shots introduce noise, so you need a high number for stable gradients.
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. THE NON-LINEAR LOSS FUNCTION
# ==========================================
alpha_loss = 3
beta_loss = 0.5
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def loss_func_estimator(x, silent=False):
    job = estimator.run([
        (ansatz, observables_x, x),
        (ansatz, observables_y, x),
        (ansatz, observables_z, x)
    ])
    result = job.result()
    
    node_exp_map = {}
    idx = 0
    for r in result:
        for ev in r.data.evs:
            node_exp_map[idx] = ev
            idx += 1
            
    loss = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss += w * np.tanh(alpha_loss * node_exp_map[edge0]) * np.tanh(alpha_loss * node_exp_map[edge1])
        
    regulation_term = 0
    for i in range(num_nodes):
        regulation_term += np.tanh(alpha_loss * node_exp_map[i])**2
        
    regulation_term = regulation_term / num_nodes
    regulation_term = beta_loss * v * (regulation_term**2)
    loss = loss + regulation_term
    
    if not silent:
        experiment_result.append({"loss": loss, "exp_map": node_exp_map})
        print(f"Iter {len(experiment_result)}: {loss:.6f}")
        
    return loss

def silent_loss_func(x):
    """Wrapper to prevent terminal spam during gradient calculation"""
    return loss_func_estimator(x, silent=True)

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue.npy"

#warm_start_file = "pce_optimal_theta.npy" # or whatever you named your Adam output
num_new_params = ansatz.num_parameters # For reps=4, this will be 90

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        # Perfect match (e.g., if you run reps=4 again later)
        initial_params = old_params
        
    elif num_old_params < num_new_params:
        # PADDING LOGIC: Transferring from a shallower circuit to a deeper one
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        
        # Initialize the new array with tiny random noise (e.g., between -0.01 and 0.01)
        # We use tiny noise instead of perfect 0.0 to prevent barren plateaus in the new layers
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        
        # Inject the highly optimized parameters into the beginning of the circuit
        initial_params[:num_old_params] = old_params
        
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)
# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize(loss_fn, theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    """Custom Adam optimizer loop using Euclidean gradients."""
    print(f"Starting optimization with Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    # Initialize momentum (m) and velocity (v) vectors
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    # Do an initial non-silent run to register the first point
    _ = loss_func_estimator(theta_i, silent=False)
    
    for i in range(max_iter):
        print(f"\n--- Adam Step {i+1}/{max_iter} ---")
        t += 1
        
        # 1. Calculate the Euclidean gradient (quietly)
        grad = approx_fprime(theta_i, silent_loss_func, epsilon=1e-5)
        
        # 2. Update biased first moment estimate (momentum)
        m_t = beta1 * m_t + (1 - beta1) * grad
        
        # 3. Update biased second raw moment estimate (velocity)
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        # 4. Compute bias-corrected estimates
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        # 5. Update parameters
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        
        # 5. Update parameters with Learning Rate Decay
        # The learning rate will smoothly shrink by 2% every iteration
        #current_lr = lr * (0.98 ** t)
        #theta_next = theta_i - current_lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        # Check for convergence (if step size is incredibly small)
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            loss_func_estimator(theta_i, silent=False)
            break
            
        theta_i = theta_next
        
        # Run non-silently to log the official step taken
        loss_func_estimator(theta_i, silent=False)
        
    return theta_i

# Execute the Adam Optimizer
# Note: Adam learning rates (lr) usually range between 0.001 and 0.1 depending on the landscape
best_params = adam_optimize(
    loss_fn=silent_loss_func, 
    theta0=initial_params,
    lr=0.04,        # Adjust this! Higher = faster but might overshoot
    max_iter=100
)

print("\nAdam Optimization Complete!")

# Optional: Save the new Adam parameters
np.save("pce_optimal_theta_adam_reps4qbt12Continue.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

best_cut = 0
best_bits = []

for edge0, edge1 in graph.edges():
    swapped_bits = cur_bits.copy()
    swapped_bits[edge0], swapped_bits[edge1] = swapped_bits[edge1], swapped_bits[edge0]
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if best_cut < cut_size:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue.npy...
Starting optimization with Adam (lr=0.04)...
Iter 1: -1846.175756

--- Adam Step 1/100 ---
Iter 2: -1818.613917

--- Adam Step 2/100 ---
Iter 3: -1843.589651

--- Adam Step 3/100 ---
Iter 4: -1848.090002

--- Adam Step 4/100 ---
Iter 5: -1847.527473

--- Adam Step 5/100 ---
Iter 6: -1849.515554

--- Adam Step 6/100 ---
Iter 7: -1856.262292

--- Adam Step 7/100 ---
Iter 8: -1862.014876

--- Adam Step 8/100 ---
Iter 9: -1863.294050

--- Adam Step 9/100 ---
Iter 10: -1863.570732

--- Adam Step 10/100 ---
Iter 11: -1865.669958

--- Adam Step 11/100 ---
Iter 12: -1868.686273

--- Adam Step 12/100 ---
Iter 13: -1870.952268

--- Adam Step 13/100 ---
Iter 14: -1871.813762

--- Adam Step 14/100 ---
Iter 15: -1872.193155

--- Adam Step 15/100 ---
Iter 16: -1873.219654

--- Adam Step 16/100 ---
Iter 17: -1874.818321

--- Adam Step 17/100 ---
Iter 18: -1876.204033

--- Adam Step 18/100 ---
Iter 1

In [12]:
def greedy_local_search(bits, graph, max_rounds=100):
    bits = list(bits)
    best_cut = calc_cut_size(graph, 
                              {i for i,b in enumerate(bits) if b==1},
                              {i for i,b in enumerate(bits) if b==0})
    improved = True
    rounds = 0
    while improved and rounds < max_rounds:
        improved = False
        rounds += 1
        for node in range(len(bits)):
            bits[node] ^= 1  # flip node
            p0 = {i for i,b in enumerate(bits) if b==1}
            p1 = {i for i,b in enumerate(bits) if b==0}
            new_cut = calc_cut_size(graph, p0, p1)
            if new_cut > best_cut:
                best_cut = new_cut
                improved = True
            else:
                bits[node] ^= 1  # flip back
    return bits, best_cut

# Apply AFTER your current decoder
best_bits_greedy, best_cut_greedy = greedy_local_search(best_bits, graph)
print(f"After greedy local search: {best_cut_greedy}")

After greedy local search: 6660.475377292293


In [14]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations
from scipy.optimize import approx_fprime

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp, Statevector
# Using the local V2 Estimator simulator instead of IBM Runtime
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12
k = 2

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        
        # This loop makes it work for any k (e.g., k=2, k=3, k=4)
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
#estimator = Estimator()

ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])
estimator = EstimatorV2()

# Highly recommended: Set the number of shots. 
# Finite shots introduce noise, so you need a high number for stable gradients.
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. THE NON-LINEAR LOSS FUNCTION
# ==========================================
alpha_loss = 3
beta_loss = 0.5
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def loss_func_estimator(x, silent=False):
    job = estimator.run([
        (ansatz, observables_x, x),
        (ansatz, observables_y, x),
        (ansatz, observables_z, x)
    ])
    result = job.result()
    
    node_exp_map = {}
    idx = 0
    for r in result:
        for ev in r.data.evs:
            node_exp_map[idx] = ev
            idx += 1
            
    loss = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss += w * np.tanh(alpha_loss * node_exp_map[edge0]) * np.tanh(alpha_loss * node_exp_map[edge1])
        
    regulation_term = 0
    for i in range(num_nodes):
        regulation_term += np.tanh(alpha_loss * node_exp_map[i])**2
        
    regulation_term = regulation_term / num_nodes
    regulation_term = beta_loss * v * (regulation_term**2)
    loss = loss + regulation_term
    
    if not silent:
        experiment_result.append({"loss": loss, "exp_map": node_exp_map})
        print(f"Iter {len(experiment_result)}: {loss:.6f}")
        
    return loss

def silent_loss_func(x):
    """Wrapper to prevent terminal spam during gradient calculation"""
    return loss_func_estimator(x, silent=True)

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue.npy"

#warm_start_file = "pce_optimal_theta.npy" # or whatever you named your Adam output
num_new_params = ansatz.num_parameters # For reps=4, this will be 90

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        # Perfect match (e.g., if you run reps=4 again later)
        initial_params = old_params
        
    elif num_old_params < num_new_params:
        # PADDING LOGIC: Transferring from a shallower circuit to a deeper one
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        
        # Initialize the new array with tiny random noise (e.g., between -0.01 and 0.01)
        # We use tiny noise instead of perfect 0.0 to prevent barren plateaus in the new layers
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        
        # Inject the highly optimized parameters into the beginning of the circuit
        initial_params[:num_old_params] = old_params
        
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)
# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize(loss_fn, theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    """Custom Adam optimizer loop using Euclidean gradients."""
    print(f"Starting optimization with Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    # Initialize momentum (m) and velocity (v) vectors
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    # Do an initial non-silent run to register the first point
    _ = loss_func_estimator(theta_i, silent=False)
    
    for i in range(max_iter):
        print(f"\n--- Adam Step {i+1}/{max_iter} ---")
        t += 1
        
        # 1. Calculate the Euclidean gradient (quietly)
        grad = approx_fprime(theta_i, silent_loss_func, epsilon=1e-5)
        
        # 2. Update biased first moment estimate (momentum)
        m_t = beta1 * m_t + (1 - beta1) * grad
        
        # 3. Update biased second raw moment estimate (velocity)
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        # 4. Compute bias-corrected estimates
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        # 5. Update parameters
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        
        # 5. Update parameters with Learning Rate Decay
        # The learning rate will smoothly shrink by 2% every iteration
        #current_lr = lr * (0.98 ** t)
        #theta_next = theta_i - current_lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        # Check for convergence (if step size is incredibly small)
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            loss_func_estimator(theta_i, silent=False)
            break
            
        theta_i = theta_next
        
        # Run non-silently to log the official step taken
        loss_func_estimator(theta_i, silent=False)
        
    return theta_i

# Execute the Adam Optimizer
# Note: Adam learning rates (lr) usually range between 0.001 and 0.1 depending on the landscape
best_params = adam_optimize(
    loss_fn=silent_loss_func, 
    theta0=initial_params,
    lr=0.06,        # Adjust this! Higher = faster but might overshoot
    max_iter=100
)

print("\nAdam Optimization Complete!")

# Optional: Save the new Adam parameters
np.save("pce_optimal_theta_adam_reps4qbt12Continue.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

best_cut = 0
best_bits = []

for edge0, edge1 in graph.edges():
    swapped_bits = cur_bits.copy()
    swapped_bits[edge0], swapped_bits[edge1] = swapped_bits[edge1], swapped_bits[edge0]
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if best_cut < cut_size:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue.npy...
Starting optimization with Adam (lr=0.06)...
Iter 1: -1930.449022

--- Adam Step 1/100 ---
Iter 2: -1836.930326

--- Adam Step 2/100 ---
Iter 3: -1876.060530

--- Adam Step 3/100 ---
Iter 4: -1901.176321

--- Adam Step 4/100 ---
Iter 5: -1880.459020

--- Adam Step 5/100 ---
Iter 6: -1879.210658

--- Adam Step 6/100 ---
Iter 7: -1905.716621

--- Adam Step 7/100 ---
Iter 8: -1923.666765

--- Adam Step 8/100 ---
Iter 9: -1918.106300

--- Adam Step 9/100 ---
Iter 10: -1907.474731

--- Adam Step 10/100 ---
Iter 11: -1909.813700

--- Adam Step 11/100 ---
Iter 12: -1921.228591

--- Adam Step 12/100 ---
Iter 13: -1930.604612

--- Adam Step 13/100 ---
Iter 14: -1934.032965

--- Adam Step 14/100 ---
Iter 15: -1933.314291

--- Adam Step 15/100 ---
Iter 16: -1932.894718

--- Adam Step 16/100 ---
Iter 17: -1936.562592

--- Adam Step 17/100 ---
Iter 18: -1943.546242

--- Adam Step 18/100 ---
Iter 1

KeyboardInterrupt: 

In [4]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12                 # Increased to 12 to support 180 nodes at k=2
k = 2                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
# High shots reduce statistical noise for the parameter shift gradients
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. LOSS FUNCTION & CHAIN RULE GRADIENTS
# ==========================================
alpha_loss = 12  # Rescaling factor scaled to approx n^(k/2)
beta_loss = 0.5
# Edwards-Erdös bound for the regularization parameter v
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def evaluate_expectations(theta):
    """Helper to get pure expectation values for a single parameter array."""
    job = estimator.run([
        (ansatz, observables_x, theta),
        (ansatz, observables_y, theta),
        (ansatz, observables_z, theta)
    ])
    results = job.result()
    
    # Extract 1D arrays of expectation values
    E_x = results[0].data.evs
    E_y = results[1].data.evs
    E_z = results[2].data.evs
    return np.concatenate([E_x, E_y, E_z])

def calc_loss_and_chain_rule(E):
    """Analytically calculates the loss and the gradient dL/dE."""
    # Precompute common terms for speed
    tE = np.tanh(alpha_loss * E)
    sech2E = 1.0 - tE**2
    
    loss_cut = 0.0
    dL_dE = np.zeros(num_nodes)
    
    # Chain rule for the Cut Loss term
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss_cut += w * tE[edge0] * tE[edge1]
        
        # dL/dE for connected nodes
        dL_dE[edge0] += w * alpha_loss * sech2E[edge0] * tE[edge1]
        dL_dE[edge1] += w * alpha_loss * sech2E[edge1] * tE[edge0]
        
    # Chain rule for the Regularization term
    S = np.sum(tE**2) / num_nodes
    reg_term = beta_loss * v * (S**2)
    loss = loss_cut + reg_term
    
    # Add regularization gradient to dL/dE
    reg_grad_factor = (4.0 * beta_loss * v * S / num_nodes) * alpha_loss
    dL_dE += reg_grad_factor * tE * sech2E
    
    return loss, dL_dE

# ==========================================
# 4.5. PARAMETER SHIFT RULE JACOBIAN (CORRECTED)
# ==========================================
def param_shift_gradient(theta):
    """Calculates the full gradient using batched Parameter Shift + Chain Rule."""
    num_params = len(theta)
    shift = np.pi / 2.0
    
    # Step A: Build a batched array of shifted parameters
    theta_shifted = []
    for j in range(num_params):
        t_plus = theta.copy()
        t_plus[j] += shift
        t_minus = theta.copy()
        t_minus[j] -= shift
        theta_shifted.append(t_plus)
        theta_shifted.append(t_minus)
        
    theta_shifted = np.array(theta_shifted) # Shape: (2*num_params, num_params)
    
    # FIX: Add a new axis to force a Cartesian product broadcast
    # Now it evaluates all 240 parameter sets against all 60 observables
    theta_shifted_broadcast = theta_shifted[:, np.newaxis, :] 
    
    # Step B: Run batched estimator
    job = estimator.run([
        (ansatz, observables_x, theta_shifted_broadcast),
        (ansatz, observables_y, theta_shifted_broadcast),
        (ansatz, observables_z, theta_shifted_broadcast)
    ])
    results = job.result()
    
    # Safely extract and orient the results to shape (len(observables), 2*num_params)
    def extract_evs(pub_result):
        evs = np.atleast_2d(pub_result.data.evs)
        # Transpose if the shape is (240, 60) to make it (60, 240)
        if evs.shape[0] == 2 * num_params:
            evs = evs.T
        return evs

    evs_x = extract_evs(results[0])
    evs_y = extract_evs(results[1])
    evs_z = extract_evs(results[2])
    
    # Stack vertically to get shape (num_nodes, 2*num_params)
    all_evs = np.vstack([evs_x, evs_y, evs_z])
    
    # Step C: Compute Jacobian dE/dTheta
    J = np.zeros((num_nodes, num_params))
    for j in range(num_params):
        evs_plus = all_evs[:, 2*j]
        evs_minus = all_evs[:, 2*j + 1]
        J[:, j] = (evs_plus - evs_minus) / 2.0
        
    # Step D: Apply Chain Rule
    E_current = evaluate_expectations(theta)
    loss, dL_dE = calc_loss_and_chain_rule(E_current)
    grad = dL_dE @ J
    
    # Save for bit-swap decoding
    global experiment_result
    experiment_result.append({"loss": loss, "exp_map": dict(enumerate(E_current))})
    
    return loss, grad

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue.npy"
num_new_params = ansatz.num_parameters # For reps=4 and 12 qubits, this will be 120

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize_param_shift(theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    print(f"Starting optimization with Parameter Shift Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    for i in range(max_iter):
        t += 1
        
        loss, grad = param_shift_gradient(theta_i)
        print(f"--- Adam Step {i+1}/{max_iter} --- | Loss: {loss:.6f}")
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            break
            
        theta_i = theta_next
        
    return theta_i

best_params = adam_optimize_param_shift(
    theta0=initial_params,
    lr=0.06,        
    max_iter=100
)

print("\nAdam Optimization Complete!")
np.save("pce_optimal_theta_adam_reps4qbt12Continue.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"\nInitial Cut Size before Local Search: {best_cut}")

# Single-bit swap search as described in the paper
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] # Flip 1 to 0 or 0 to 1
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    # Keep the swap if it improves the cut
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue.npy...
Starting optimization with Parameter Shift Adam (lr=0.06)...
--- Adam Step 1/100 --- | Loss: -3139.916259
--- Adam Step 2/100 --- | Loss: -3280.242524
--- Adam Step 3/100 --- | Loss: -3243.119267
--- Adam Step 4/100 --- | Loss: -3439.480645
--- Adam Step 5/100 --- | Loss: -3489.305326
--- Adam Step 6/100 --- | Loss: -3517.680644
--- Adam Step 7/100 --- | Loss: -3561.452551
--- Adam Step 8/100 --- | Loss: -3608.561149
--- Adam Step 9/100 --- | Loss: -3647.794819
--- Adam Step 10/100 --- | Loss: -3671.316587
--- Adam Step 11/100 --- | Loss: -3673.507067
--- Adam Step 12/100 --- | Loss: -3678.635827
--- Adam Step 13/100 --- | Loss: -3692.957694
--- Adam Step 14/100 --- | Loss: -3710.561208
--- Adam Step 15/100 --- | Loss: -3723.571452
--- Adam Step 16/100 --- | Loss: -3738.075082
--- Adam Step 17/100 --- | Loss: -3749.977693
--- Adam Step 18/100 --- | Loss: -3754.866042
--- Adam Step 19/

In [5]:
def greedy_local_search(bits, graph, max_rounds=100):
    bits = list(bits)
    best_cut = calc_cut_size(graph, 
                              {i for i,b in enumerate(bits) if b==1},
                              {i for i,b in enumerate(bits) if b==0})
    improved = True
    rounds = 0
    while improved and rounds < max_rounds:
        improved = False
        rounds += 1
        for node in range(len(bits)):
            bits[node] ^= 1  # flip node
            p0 = {i for i,b in enumerate(bits) if b==1}
            p1 = {i for i,b in enumerate(bits) if b==0}
            new_cut = calc_cut_size(graph, p0, p1)
            if new_cut > best_cut:
                best_cut = new_cut
                improved = True
            else:
                bits[node] ^= 1  # flip back
    return bits, best_cut

# Apply AFTER your current decoder
best_bits_greedy, best_cut_greedy = greedy_local_search(best_bits, graph)
print(f"After greedy local search: {best_cut_greedy}")

After greedy local search: 6587.41669658521


In [6]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12                 # Increased to 12 to support 180 nodes at k=2
k = 2                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
# High shots reduce statistical noise for the parameter shift gradients
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. LOSS FUNCTION & CHAIN RULE GRADIENTS
# ==========================================
alpha_loss = 12  # Rescaling factor scaled to approx n^(k/2)
beta_loss = 0.5
# Edwards-Erdös bound for the regularization parameter v
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def evaluate_expectations(theta):
    """Helper to get pure expectation values for a single parameter array."""
    job = estimator.run([
        (ansatz, observables_x, theta),
        (ansatz, observables_y, theta),
        (ansatz, observables_z, theta)
    ])
    results = job.result()
    
    # Extract 1D arrays of expectation values
    E_x = results[0].data.evs
    E_y = results[1].data.evs
    E_z = results[2].data.evs
    return np.concatenate([E_x, E_y, E_z])

def calc_loss_and_chain_rule(E):
    """Analytically calculates the loss and the gradient dL/dE."""
    # Precompute common terms for speed
    tE = np.tanh(alpha_loss * E)
    sech2E = 1.0 - tE**2
    
    loss_cut = 0.0
    dL_dE = np.zeros(num_nodes)
    
    # Chain rule for the Cut Loss term
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss_cut += w * tE[edge0] * tE[edge1]
        
        # dL/dE for connected nodes
        dL_dE[edge0] += w * alpha_loss * sech2E[edge0] * tE[edge1]
        dL_dE[edge1] += w * alpha_loss * sech2E[edge1] * tE[edge0]
        
    # Chain rule for the Regularization term
    S = np.sum(tE**2) / num_nodes
    reg_term = beta_loss * v * (S**2)
    loss = loss_cut + reg_term
    
    # Add regularization gradient to dL/dE
    reg_grad_factor = (4.0 * beta_loss * v * S / num_nodes) * alpha_loss
    dL_dE += reg_grad_factor * tE * sech2E
    
    return loss, dL_dE

# ==========================================
# 4.5. PARAMETER SHIFT RULE JACOBIAN (CORRECTED)
# ==========================================
def param_shift_gradient(theta):
    """Calculates the full gradient using batched Parameter Shift + Chain Rule."""
    num_params = len(theta)
    shift = np.pi / 2.0
    
    # Step A: Build a batched array of shifted parameters
    theta_shifted = []
    for j in range(num_params):
        t_plus = theta.copy()
        t_plus[j] += shift
        t_minus = theta.copy()
        t_minus[j] -= shift
        theta_shifted.append(t_plus)
        theta_shifted.append(t_minus)
        
    theta_shifted = np.array(theta_shifted) # Shape: (2*num_params, num_params)
    
    # FIX: Add a new axis to force a Cartesian product broadcast
    # Now it evaluates all 240 parameter sets against all 60 observables
    theta_shifted_broadcast = theta_shifted[:, np.newaxis, :] 
    
    # Step B: Run batched estimator
    job = estimator.run([
        (ansatz, observables_x, theta_shifted_broadcast),
        (ansatz, observables_y, theta_shifted_broadcast),
        (ansatz, observables_z, theta_shifted_broadcast)
    ])
    results = job.result()
    
    # Safely extract and orient the results to shape (len(observables), 2*num_params)
    def extract_evs(pub_result):
        evs = np.atleast_2d(pub_result.data.evs)
        # Transpose if the shape is (240, 60) to make it (60, 240)
        if evs.shape[0] == 2 * num_params:
            evs = evs.T
        return evs

    evs_x = extract_evs(results[0])
    evs_y = extract_evs(results[1])
    evs_z = extract_evs(results[2])
    
    # Stack vertically to get shape (num_nodes, 2*num_params)
    all_evs = np.vstack([evs_x, evs_y, evs_z])
    
    # Step C: Compute Jacobian dE/dTheta
    J = np.zeros((num_nodes, num_params))
    for j in range(num_params):
        evs_plus = all_evs[:, 2*j]
        evs_minus = all_evs[:, 2*j + 1]
        J[:, j] = (evs_plus - evs_minus) / 2.0
        
    # Step D: Apply Chain Rule
    E_current = evaluate_expectations(theta)
    loss, dL_dE = calc_loss_and_chain_rule(E_current)
    grad = dL_dE @ J
    
    # Save for bit-swap decoding
    global experiment_result
    experiment_result.append({"loss": loss, "exp_map": dict(enumerate(E_current))})
    
    return loss, grad

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue.npy"
num_new_params = ansatz.num_parameters # For reps=4 and 12 qubits, this will be 120

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize_param_shift(theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    print(f"Starting optimization with Parameter Shift Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    for i in range(max_iter):
        t += 1
        
        loss, grad = param_shift_gradient(theta_i)
        print(f"--- Adam Step {i+1}/{max_iter} --- | Loss: {loss:.6f}")
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            break
            
        theta_i = theta_next
        
    return theta_i

best_params = adam_optimize_param_shift(
    theta0=initial_params,
    lr=0.06,        
    max_iter=100
)

print("\nAdam Optimization Complete!")
np.save("pce_optimal_theta_adam_reps4qbt12Continue.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"\nInitial Cut Size before Local Search: {best_cut}")

# Single-bit swap search as described in the paper
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] # Flip 1 to 0 or 0 to 1
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    # Keep the swap if it improves the cut
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue.npy...
Starting optimization with Parameter Shift Adam (lr=0.06)...
--- Adam Step 1/100 --- | Loss: -4025.394719
--- Adam Step 2/100 --- | Loss: -3957.531571
--- Adam Step 3/100 --- | Loss: -3950.472606
--- Adam Step 4/100 --- | Loss: -4006.045968
--- Adam Step 5/100 --- | Loss: -3987.437569
--- Adam Step 6/100 --- | Loss: -3999.211749
--- Adam Step 7/100 --- | Loss: -4036.168635
--- Adam Step 8/100 --- | Loss: -4057.314608
--- Adam Step 9/100 --- | Loss: -4043.613519
--- Adam Step 10/100 --- | Loss: -4053.445460
--- Adam Step 11/100 --- | Loss: -4082.141825
--- Adam Step 12/100 --- | Loss: -4081.372456
--- Adam Step 13/100 --- | Loss: -4077.422466
--- Adam Step 14/100 --- | Loss: -4081.814997
--- Adam Step 15/100 --- | Loss: -4092.874748
--- Adam Step 16/100 --- | Loss: -4106.735062
--- Adam Step 17/100 --- | Loss: -4111.219887
--- Adam Step 18/100 --- | Loss: -4110.861095
--- Adam Step 19/

In [7]:
def greedy_local_search(bits, graph, max_rounds=100):
    bits = list(bits)
    best_cut = calc_cut_size(graph, 
                              {i for i,b in enumerate(bits) if b==1},
                              {i for i,b in enumerate(bits) if b==0})
    improved = True
    rounds = 0
    while improved and rounds < max_rounds:
        improved = False
        rounds += 1
        for node in range(len(bits)):
            bits[node] ^= 1  # flip node
            p0 = {i for i,b in enumerate(bits) if b==1}
            p1 = {i for i,b in enumerate(bits) if b==0}
            new_cut = calc_cut_size(graph, p0, p1)
            if new_cut > best_cut:
                best_cut = new_cut
                improved = True
            else:
                bits[node] ^= 1  # flip back
    return bits, best_cut

# Apply AFTER your current decoder
best_bits_greedy, best_cut_greedy = greedy_local_search(best_bits, graph)
print(f"After greedy local search: {best_cut_greedy}")

After greedy local search: 6610.582369320807


In [10]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12                 # Increased to 12 to support 180 nodes at k=2
k = 2                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
# High shots reduce statistical noise for the parameter shift gradients
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. LOSS FUNCTION & CHAIN RULE GRADIENTS
# ==========================================
alpha_loss = 12  # Rescaling factor scaled to approx n^(k/2)
beta_loss = 0.5
# Edwards-Erdös bound for the regularization parameter v
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def evaluate_expectations(theta):
    """Helper to get pure expectation values for a single parameter array."""
    job = estimator.run([
        (ansatz, observables_x, theta),
        (ansatz, observables_y, theta),
        (ansatz, observables_z, theta)
    ])
    results = job.result()
    
    # Extract 1D arrays of expectation values
    E_x = results[0].data.evs
    E_y = results[1].data.evs
    E_z = results[2].data.evs
    return np.concatenate([E_x, E_y, E_z])

def calc_loss_and_chain_rule(E):
    """Analytically calculates the loss and the gradient dL/dE."""
    # Precompute common terms for speed
    tE = np.tanh(alpha_loss * E)
    sech2E = 1.0 - tE**2
    
    loss_cut = 0.0
    dL_dE = np.zeros(num_nodes)
    
    # Chain rule for the Cut Loss term
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss_cut += w * tE[edge0] * tE[edge1]
        
        # dL/dE for connected nodes
        dL_dE[edge0] += w * alpha_loss * sech2E[edge0] * tE[edge1]
        dL_dE[edge1] += w * alpha_loss * sech2E[edge1] * tE[edge0]
        
    # Chain rule for the Regularization term
    S = np.sum(tE**2) / num_nodes
    reg_term = beta_loss * v * (S**2)
    loss = loss_cut + reg_term
    
    # Add regularization gradient to dL/dE
    reg_grad_factor = (4.0 * beta_loss * v * S / num_nodes) * alpha_loss
    dL_dE += reg_grad_factor * tE * sech2E
    
    return loss, dL_dE

# ==========================================
# 4.5. PARAMETER SHIFT RULE JACOBIAN (CORRECTED)
# ==========================================
def param_shift_gradient(theta):
    """Calculates the full gradient using batched Parameter Shift + Chain Rule."""
    num_params = len(theta)
    shift = np.pi / 2.0
    
    # Step A: Build a batched array of shifted parameters
    theta_shifted = []
    for j in range(num_params):
        t_plus = theta.copy()
        t_plus[j] += shift
        t_minus = theta.copy()
        t_minus[j] -= shift
        theta_shifted.append(t_plus)
        theta_shifted.append(t_minus)
        
    theta_shifted = np.array(theta_shifted) # Shape: (2*num_params, num_params)
    
    # FIX: Add a new axis to force a Cartesian product broadcast
    # Now it evaluates all 240 parameter sets against all 60 observables
    theta_shifted_broadcast = theta_shifted[:, np.newaxis, :] 
    
    # Step B: Run batched estimator
    job = estimator.run([
        (ansatz, observables_x, theta_shifted_broadcast),
        (ansatz, observables_y, theta_shifted_broadcast),
        (ansatz, observables_z, theta_shifted_broadcast)
    ])
    results = job.result()
    
    # Safely extract and orient the results to shape (len(observables), 2*num_params)
    def extract_evs(pub_result):
        evs = np.atleast_2d(pub_result.data.evs)
        # Transpose if the shape is (240, 60) to make it (60, 240)
        if evs.shape[0] == 2 * num_params:
            evs = evs.T
        return evs

    evs_x = extract_evs(results[0])
    evs_y = extract_evs(results[1])
    evs_z = extract_evs(results[2])
    
    # Stack vertically to get shape (num_nodes, 2*num_params)
    all_evs = np.vstack([evs_x, evs_y, evs_z])
    
    # Step C: Compute Jacobian dE/dTheta
    J = np.zeros((num_nodes, num_params))
    for j in range(num_params):
        evs_plus = all_evs[:, 2*j]
        evs_minus = all_evs[:, 2*j + 1]
        J[:, j] = (evs_plus - evs_minus) / 2.0
        
    # Step D: Apply Chain Rule
    E_current = evaluate_expectations(theta)
    loss, dL_dE = calc_loss_and_chain_rule(E_current)
    grad = dL_dE @ J
    
    # Save for bit-swap decoding
    global experiment_result
    experiment_result.append({"loss": loss, "exp_map": dict(enumerate(E_current))})
    
    return loss, grad

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue.npy"
num_new_params = ansatz.num_parameters # For reps=4 and 12 qubits, this will be 120

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize_param_shift(theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    print(f"Starting optimization with Parameter Shift Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    for i in range(max_iter):
        t += 1
        
        loss, grad = param_shift_gradient(theta_i)
        print(f"--- Adam Step {i+1}/{max_iter} --- | Loss: {loss:.6f}")
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            break
            
        theta_i = theta_next
        
    return theta_i

best_params = adam_optimize_param_shift(
    theta0=initial_params,
    lr=0.06,        
    max_iter=200
)

print("\nAdam Optimization Complete!")
np.save("pce_optimal_theta_adam_reps4qbt12Continue.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"\nInitial Cut Size before Local Search: {best_cut}")

# Single-bit swap search as described in the paper
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] # Flip 1 to 0 or 0 to 1
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    # Keep the swap if it improves the cut
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue.npy...
Starting optimization with Parameter Shift Adam (lr=0.06)...
--- Adam Step 1/200 --- | Loss: -4255.643288
--- Adam Step 2/200 --- | Loss: -4091.471355
--- Adam Step 3/200 --- | Loss: -4081.734914
--- Adam Step 4/200 --- | Loss: -4089.850325
--- Adam Step 5/200 --- | Loss: -4138.885493
--- Adam Step 6/200 --- | Loss: -4175.760994
--- Adam Step 7/200 --- | Loss: -4179.681279
--- Adam Step 8/200 --- | Loss: -4178.368011
--- Adam Step 9/200 --- | Loss: -4193.512262
--- Adam Step 10/200 --- | Loss: -4208.407390
--- Adam Step 11/200 --- | Loss: -4214.083919
--- Adam Step 12/200 --- | Loss: -4211.239152
--- Adam Step 13/200 --- | Loss: -4214.015562
--- Adam Step 14/200 --- | Loss: -4227.963297
--- Adam Step 15/200 --- | Loss: -4238.158031
--- Adam Step 16/200 --- | Loss: -4239.944416
--- Adam Step 17/200 --- | Loss: -4235.383162
--- Adam Step 18/200 --- | Loss: -4232.165720
--- Adam Step 19/

In [11]:
# ==========================================
# 7. DEEP DECODING & BIT-SWAP POST-PROCESSING
# ==========================================

# 1. Extract initial bitstring from the Quantum Circuit's Expectation Values
cur_bits = np.zeros(num_nodes, dtype=int)
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits[i] = 1
    else:
        cur_bits[i] = 0

# Helper to calculate the full absolute cut size once
def calculate_full_cut(graph, bits):
    cut = 0
    for u, v, data in graph.edges(data=True):
        if bits[u] != bits[v]:
            cut += data.get('weight', 1.0)
    return cut

current_cut = calculate_full_cut(graph, cur_bits)
print(f"\nInitial Quantum Cut (Before Local Search): {current_cut}")

# 2. Iterative 1-Bit and 2-Bit Swap Search
search_iteration = 0
improvement_found = True

while improvement_found:
    improvement_found = False
    search_iteration += 1
    
    # --- PHASE 1: 1-BIT SWAP SEARCH ---
    # We sweep through all nodes. If flipping a node improves the cut, we keep it.
    for u in graph.nodes():
        delta = 0
        # Calculate exactly how the cut changes if we flip node `u`
        for v in graph.neighbors(u):
            w = graph[u][v].get('weight', 1.0)
            if cur_bits[u] == cur_bits[v]:
                delta += w  # They were in the same partition. Flipping u cuts this edge.
            else:
                delta -= w  # They were in different partitions. Flipping u un-cuts it.
                
        # If flipping strictly improves the cut, apply the flip
        if delta > 0:
            cur_bits[u] = 1 - cur_bits[u]
            current_cut += delta
            improvement_found = True

    # --- PHASE 2: 2-BIT SWAP SEARCH ---
    # If 1-bit swaps get stuck in a local minimum, we try flipping pairs of connected nodes
    if not improvement_found:
        for u, v in graph.edges():
            delta = 0
            
            # Change in cut for neighbors of u (excluding v)
            for n_u in graph.neighbors(u):
                if n_u == v: continue
                w = graph[u][n_u].get('weight', 1.0)
                if cur_bits[u] == cur_bits[n_u]: delta += w
                else: delta -= w
                    
            # Change in cut for neighbors of v (excluding u)
            for n_v in graph.neighbors(v):
                if n_v == u: continue
                w = graph[v][n_v].get('weight', 1.0)
                if cur_bits[v] == cur_bits[n_v]: delta += w
                else: delta -= w
                    
            if delta > 0:
                cur_bits[u] = 1 - cur_bits[u]
                cur_bits[v] = 1 - cur_bits[v]
                current_cut += delta
                improvement_found = True
                break # Break to restart the 1-bit sweep with the new landscape

print(f"Post-processing complete after {search_iteration} sweeps.")
print(f"Final Deep-Optimized Cut Size: {current_cut}")


Initial Quantum Cut (Before Local Search): 6218.935850141958
Post-processing complete after 11 sweeps.
Final Deep-Optimized Cut Size: 6767.718383338153


In [15]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12                 # Increased to 12 to support 180 nodes at k=2
k = 2                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
# High shots reduce statistical noise for the parameter shift gradients
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. LOSS FUNCTION & CHAIN RULE GRADIENTS
# ==========================================
alpha_loss = 16  # Rescaling factor scaled to approx n^(k/2)
beta_loss = 0.5
# Edwards-Erdös bound for the regularization parameter v
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def evaluate_expectations(theta):
    """Helper to get pure expectation values for a single parameter array."""
    job = estimator.run([
        (ansatz, observables_x, theta),
        (ansatz, observables_y, theta),
        (ansatz, observables_z, theta)
    ])
    results = job.result()
    
    # Extract 1D arrays of expectation values
    E_x = results[0].data.evs
    E_y = results[1].data.evs
    E_z = results[2].data.evs
    return np.concatenate([E_x, E_y, E_z])

def calc_loss_and_chain_rule(E):
    """Analytically calculates the loss and the gradient dL/dE."""
    # Precompute common terms for speed
    tE = np.tanh(alpha_loss * E)
    sech2E = 1.0 - tE**2
    
    loss_cut = 0.0
    dL_dE = np.zeros(num_nodes)
    
    # Chain rule for the Cut Loss term
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss_cut += w * tE[edge0] * tE[edge1]
        
        # dL/dE for connected nodes
        dL_dE[edge0] += w * alpha_loss * sech2E[edge0] * tE[edge1]
        dL_dE[edge1] += w * alpha_loss * sech2E[edge1] * tE[edge0]
        
    # Chain rule for the Regularization term
    S = np.sum(tE**2) / num_nodes
    reg_term = beta_loss * v * (S**2)
    loss = loss_cut + reg_term
    
    # Add regularization gradient to dL/dE
    reg_grad_factor = (4.0 * beta_loss * v * S / num_nodes) * alpha_loss
    dL_dE += reg_grad_factor * tE * sech2E
    
    return loss, dL_dE

# ==========================================
# 4.5. PARAMETER SHIFT RULE JACOBIAN (CORRECTED)
# ==========================================
def param_shift_gradient(theta):
    """Calculates the full gradient using batched Parameter Shift + Chain Rule."""
    num_params = len(theta)
    shift = np.pi / 2.0
    
    # Step A: Build a batched array of shifted parameters
    theta_shifted = []
    for j in range(num_params):
        t_plus = theta.copy()
        t_plus[j] += shift
        t_minus = theta.copy()
        t_minus[j] -= shift
        theta_shifted.append(t_plus)
        theta_shifted.append(t_minus)
        
    theta_shifted = np.array(theta_shifted) # Shape: (2*num_params, num_params)
    
    # FIX: Add a new axis to force a Cartesian product broadcast
    # Now it evaluates all 240 parameter sets against all 60 observables
    theta_shifted_broadcast = theta_shifted[:, np.newaxis, :] 
    
    # Step B: Run batched estimator
    job = estimator.run([
        (ansatz, observables_x, theta_shifted_broadcast),
        (ansatz, observables_y, theta_shifted_broadcast),
        (ansatz, observables_z, theta_shifted_broadcast)
    ])
    results = job.result()
    
    # Safely extract and orient the results to shape (len(observables), 2*num_params)
    def extract_evs(pub_result):
        evs = np.atleast_2d(pub_result.data.evs)
        # Transpose if the shape is (240, 60) to make it (60, 240)
        if evs.shape[0] == 2 * num_params:
            evs = evs.T
        return evs

    evs_x = extract_evs(results[0])
    evs_y = extract_evs(results[1])
    evs_z = extract_evs(results[2])
    
    # Stack vertically to get shape (num_nodes, 2*num_params)
    all_evs = np.vstack([evs_x, evs_y, evs_z])
    
    # Step C: Compute Jacobian dE/dTheta
    J = np.zeros((num_nodes, num_params))
    for j in range(num_params):
        evs_plus = all_evs[:, 2*j]
        evs_minus = all_evs[:, 2*j + 1]
        J[:, j] = (evs_plus - evs_minus) / 2.0
        
    # Step D: Apply Chain Rule
    E_current = evaluate_expectations(theta)
    loss, dL_dE = calc_loss_and_chain_rule(E_current)
    grad = dL_dE @ J
    
    # Save for bit-swap decoding
    global experiment_result
    experiment_result.append({"loss": loss, "exp_map": dict(enumerate(E_current))})
    
    return loss, grad

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue.npy"
num_new_params = ansatz.num_parameters # For reps=4 and 12 qubits, this will be 120

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize_param_shift(theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    print(f"Starting optimization with Parameter Shift Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    for i in range(max_iter):
        t += 1
        
        loss, grad = param_shift_gradient(theta_i)
        print(f"--- Adam Step {i+1}/{max_iter} --- | Loss: {loss:.6f}")
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            break
            
        theta_i = theta_next
        
    return theta_i

best_params = adam_optimize_param_shift(
    theta0=initial_params,
    lr=0.1,        
    max_iter=200
)

print("\nAdam Optimization Complete!")
np.save("pce_optimal_theta_adam_reps4qbt12Continue.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"\nInitial Cut Size before Local Search: {best_cut}")

# Single-bit swap search as described in the paper
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] # Flip 1 to 0 or 0 to 1
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    # Keep the swap if it improves the cut
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue.npy...
Starting optimization with Parameter Shift Adam (lr=0.1)...
--- Adam Step 1/200 --- | Loss: -4594.667293
--- Adam Step 2/200 --- | Loss: -2933.229157
--- Adam Step 3/200 --- | Loss: -4084.220926
--- Adam Step 4/200 --- | Loss: -4395.096132
--- Adam Step 5/200 --- | Loss: -4175.363481
--- Adam Step 6/200 --- | Loss: -4104.679435
--- Adam Step 7/200 --- | Loss: -4198.527152
--- Adam Step 8/200 --- | Loss: -4243.749003
--- Adam Step 9/200 --- | Loss: -4218.751811
--- Adam Step 10/200 --- | Loss: -4202.757321
--- Adam Step 11/200 --- | Loss: -4247.074296
--- Adam Step 12/200 --- | Loss: -4285.485873
--- Adam Step 13/200 --- | Loss: -4309.596040
--- Adam Step 14/200 --- | Loss: -4329.970292
--- Adam Step 15/200 --- | Loss: -4349.780966
--- Adam Step 16/200 --- | Loss: -4365.458380
--- Adam Step 17/200 --- | Loss: -4375.556897
--- Adam Step 18/200 --- | Loss: -4392.086059
--- Adam Step 19/2

KeyboardInterrupt: 

In [2]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12                 # Increased to 12 to support 180 nodes at k=2
k = 2                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
# High shots reduce statistical noise for the parameter shift gradients
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. LOSS FUNCTION & CHAIN RULE GRADIENTS
# ==========================================
alpha_loss = 24  # Rescaling factor scaled to approx n^(k/2)
beta_loss = 0.5
# Edwards-Erdös bound for the regularization parameter v
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def evaluate_expectations(theta):
    """Helper to get pure expectation values for a single parameter array."""
    job = estimator.run([
        (ansatz, observables_x, theta),
        (ansatz, observables_y, theta),
        (ansatz, observables_z, theta)
    ])
    results = job.result()
    
    # Extract 1D arrays of expectation values
    E_x = results[0].data.evs
    E_y = results[1].data.evs
    E_z = results[2].data.evs
    return np.concatenate([E_x, E_y, E_z])

def calc_loss_and_chain_rule(E):
    """Analytically calculates the loss and the gradient dL/dE."""
    # Precompute common terms for speed
    tE = np.tanh(alpha_loss * E)
    sech2E = 1.0 - tE**2
    
    loss_cut = 0.0
    dL_dE = np.zeros(num_nodes)
    
    # Chain rule for the Cut Loss term
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss_cut += w * tE[edge0] * tE[edge1]
        
        # dL/dE for connected nodes
        dL_dE[edge0] += w * alpha_loss * sech2E[edge0] * tE[edge1]
        dL_dE[edge1] += w * alpha_loss * sech2E[edge1] * tE[edge0]
        
    # Chain rule for the Regularization term
    S = np.sum(tE**2) / num_nodes
    reg_term = beta_loss * v * (S**2)
    loss = loss_cut + reg_term
    
    # Add regularization gradient to dL/dE
    reg_grad_factor = (4.0 * beta_loss * v * S / num_nodes) * alpha_loss
    dL_dE += reg_grad_factor * tE * sech2E
    
    return loss, dL_dE

# ==========================================
# 4.5. PARAMETER SHIFT RULE JACOBIAN (CORRECTED)
# ==========================================
def param_shift_gradient(theta):
    """Calculates the full gradient using batched Parameter Shift + Chain Rule."""
    num_params = len(theta)
    shift = np.pi / 2.0
    
    # Step A: Build a batched array of shifted parameters
    theta_shifted = []
    for j in range(num_params):
        t_plus = theta.copy()
        t_plus[j] += shift
        t_minus = theta.copy()
        t_minus[j] -= shift
        theta_shifted.append(t_plus)
        theta_shifted.append(t_minus)
        
    theta_shifted = np.array(theta_shifted) # Shape: (2*num_params, num_params)
    
    # FIX: Add a new axis to force a Cartesian product broadcast
    # Now it evaluates all 240 parameter sets against all 60 observables
    theta_shifted_broadcast = theta_shifted[:, np.newaxis, :] 
    
    # Step B: Run batched estimator
    job = estimator.run([
        (ansatz, observables_x, theta_shifted_broadcast),
        (ansatz, observables_y, theta_shifted_broadcast),
        (ansatz, observables_z, theta_shifted_broadcast)
    ])
    results = job.result()
    
    # Safely extract and orient the results to shape (len(observables), 2*num_params)
    def extract_evs(pub_result):
        evs = np.atleast_2d(pub_result.data.evs)
        # Transpose if the shape is (240, 60) to make it (60, 240)
        if evs.shape[0] == 2 * num_params:
            evs = evs.T
        return evs

    evs_x = extract_evs(results[0])
    evs_y = extract_evs(results[1])
    evs_z = extract_evs(results[2])
    
    # Stack vertically to get shape (num_nodes, 2*num_params)
    all_evs = np.vstack([evs_x, evs_y, evs_z])
    
    # Step C: Compute Jacobian dE/dTheta
    J = np.zeros((num_nodes, num_params))
    for j in range(num_params):
        evs_plus = all_evs[:, 2*j]
        evs_minus = all_evs[:, 2*j + 1]
        J[:, j] = (evs_plus - evs_minus) / 2.0
        
    # Step D: Apply Chain Rule
    E_current = evaluate_expectations(theta)
    loss, dL_dE = calc_loss_and_chain_rule(E_current)
    grad = dL_dE @ J
    
    # Save for bit-swap decoding
    global experiment_result
    experiment_result.append({"loss": loss, "exp_map": dict(enumerate(E_current))})
    
    return loss, grad

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue2010.npy"
num_new_params = ansatz.num_parameters # For reps=4 and 12 qubits, this will be 120

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize_param_shift(theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    print(f"Starting optimization with Parameter Shift Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    for i in range(max_iter):
        t += 1
        
        loss, grad = param_shift_gradient(theta_i)
        print(f"--- Adam Step {i+1}/{max_iter} --- | Loss: {loss:.6f}")
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            break
            
        theta_i = theta_next
        
    return theta_i

best_params = adam_optimize_param_shift(
    theta0=initial_params,
    lr=0.1,        
    max_iter=1000
)

print("\nAdam Optimization Complete!")
np.save("pce_optimal_theta_adam_reps4qbt12Continue2010_linear4.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"\nInitial Cut Size before Local Search: {best_cut}")

# Single-bit swap search as described in the paper
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] # Flip 1 to 0 or 0 to 1
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    # Keep the swap if it improves the cut
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...


/tmp/ipykernel_200474/402719646.py:59: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)


Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue2010.npy...
Padding parameters! Transferring 72 angles into a 120 angle circuit.
Starting optimization with Parameter Shift Adam (lr=0.1)...
--- Adam Step 1/1000 --- | Loss: 225.566392
--- Adam Step 2/1000 --- | Loss: -555.467991
--- Adam Step 3/1000 --- | Loss: -907.612243
--- Adam Step 4/1000 --- | Loss: -1210.540134
--- Adam Step 5/1000 --- | Loss: -1474.417814
--- Adam Step 6/1000 --- | Loss: -1656.407332
--- Adam Step 7/1000 --- | Loss: -1811.339392
--- Adam Step 8/1000 --- | Loss: -2029.067685
--- Adam Step 9/1000 --- | Loss: -2233.993836
--- Adam Step 10/1000 --- | Loss: -2434.029967
--- Adam Step 11/1000 --- | Loss: -2577.395746
--- Adam Step 12/1000 --- | Loss: -2626.776555
--- Adam Step 13/1000 --- | Loss: -2716.069795
--- Adam Step 14/1000 --- | Loss: -2820.895074
--- Adam Step 15/1000 --- | Loss: -2918.349216
--- Adam Step 16/1000 --- | Loss: -3023.106035
--- Adam Step 17/1000 --- | Loss: -3119.4531

hi


In [4]:
# ==========================================
# 7. DEEP DECODING & BIT-SWAP POST-PROCESSING
# ==========================================

# 1. Extract initial bitstring from the Quantum Circuit's Expectation Values
cur_bits = np.zeros(num_nodes, dtype=int)
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits[i] = 1
    else:
        cur_bits[i] = 0

# Helper to calculate the full absolute cut size once
def calculate_full_cut(graph, bits):
    cut = 0
    for u, v, data in graph.edges(data=True):
        if bits[u] != bits[v]:
            cut += data.get('weight', 1.0)
    return cut

current_cut = calculate_full_cut(graph, cur_bits)
print(f"\nInitial Quantum Cut (Before Local Search): {current_cut}")

# 2. Iterative 1-Bit and 2-Bit Swap Search
search_iteration = 2
improvement_found = True

while improvement_found:
    improvement_found = False
    search_iteration += 1
    
    # --- PHASE 1: 1-BIT SWAP SEARCH ---
    # We sweep through all nodes. If flipping a node improves the cut, we keep it.
    for u in graph.nodes():
        delta = 0
        # Calculate exactly how the cut changes if we flip node `u`
        for v in graph.neighbors(u):
            w = graph[u][v].get('weight', 1.0)
            if cur_bits[u] == cur_bits[v]:
                delta += w  # They were in the same partition. Flipping u cuts this edge.
            else:
                delta -= w  # They were in different partitions. Flipping u un-cuts it.
                
        # If flipping strictly improves the cut, apply the flip
        if delta > 0:
            cur_bits[u] = 1 - cur_bits[u]
            current_cut += delta
            improvement_found = True

    # --- PHASE 2: 2-BIT SWAP SEARCH ---
    # If 1-bit swaps get stuck in a local minimum, we try flipping pairs of connected nodes
    if not improvement_found:
        for u, v in graph.edges():
            delta = 0
            
            # Change in cut for neighbors of u (excluding v)
            for n_u in graph.neighbors(u):
                if n_u == v: continue
                w = graph[u][n_u].get('weight', 1.0)
                if cur_bits[u] == cur_bits[n_u]: delta += w
                else: delta -= w
                    
            # Change in cut for neighbors of v (excluding u)
            for n_v in graph.neighbors(v):
                if n_v == u: continue
                w = graph[v][n_v].get('weight', 1.0)
                if cur_bits[v] == cur_bits[n_v]: delta += w
                else: delta -= w
                    
            if delta > 0:
                cur_bits[u] = 1 - cur_bits[u]
                cur_bits[v] = 1 - cur_bits[v]
                current_cut += delta
                improvement_found = True
                break # Break to restart the 1-bit sweep with the new landscape

print(f"Post-processing complete after {search_iteration} sweeps.")
print(f"Final Deep-Optimized Cut Size: {current_cut}")


Initial Quantum Cut (Before Local Search): 6583.841310242843
Post-processing complete after 12 sweeps.
Final Deep-Optimized Cut Size: 6840.25992087964


In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12                 # Increased to 12 to support 180 nodes at k=2
k = 2                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
# High shots reduce statistical noise for the parameter shift gradients
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. LOSS FUNCTION & CHAIN RULE GRADIENTS
# ==========================================
alpha_loss = 24  # Rescaling factor scaled to approx n^(k/2)
beta_loss = 0.5
# Edwards-Erdös bound for the regularization parameter v
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def evaluate_expectations(theta):
    """Helper to get pure expectation values for a single parameter array."""
    job = estimator.run([
        (ansatz, observables_x, theta),
        (ansatz, observables_y, theta),
        (ansatz, observables_z, theta)
    ])
    results = job.result()
    
    # Extract 1D arrays of expectation values
    E_x = results[0].data.evs
    E_y = results[1].data.evs
    E_z = results[2].data.evs
    return np.concatenate([E_x, E_y, E_z])

def calc_loss_and_chain_rule(E):
    """Analytically calculates the loss and the gradient dL/dE."""
    # Precompute common terms for speed
    tE = np.tanh(alpha_loss * E)
    sech2E = 1.0 - tE**2
    
    loss_cut = 0.0
    dL_dE = np.zeros(num_nodes)
    
    # Chain rule for the Cut Loss term
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss_cut += w * tE[edge0] * tE[edge1]
        
        # dL/dE for connected nodes
        dL_dE[edge0] += w * alpha_loss * sech2E[edge0] * tE[edge1]
        dL_dE[edge1] += w * alpha_loss * sech2E[edge1] * tE[edge0]
        
    # Chain rule for the Regularization term
    S = np.sum(tE**2) / num_nodes
    reg_term = beta_loss * v * (S**2)
    loss = loss_cut + reg_term
    
    # Add regularization gradient to dL/dE
    reg_grad_factor = (4.0 * beta_loss * v * S / num_nodes) * alpha_loss
    dL_dE += reg_grad_factor * tE * sech2E
    
    return loss, dL_dE

# ==========================================
# 4.5. PARAMETER SHIFT RULE JACOBIAN (CORRECTED)
# ==========================================
def param_shift_gradient(theta):
    """Calculates the full gradient using batched Parameter Shift + Chain Rule."""
    num_params = len(theta)
    shift = np.pi / 2.0
    
    # Step A: Build a batched array of shifted parameters
    theta_shifted = []
    for j in range(num_params):
        t_plus = theta.copy()
        t_plus[j] += shift
        t_minus = theta.copy()
        t_minus[j] -= shift
        theta_shifted.append(t_plus)
        theta_shifted.append(t_minus)
        
    theta_shifted = np.array(theta_shifted) # Shape: (2*num_params, num_params)
    
    # FIX: Add a new axis to force a Cartesian product broadcast
    # Now it evaluates all 240 parameter sets against all 60 observables
    theta_shifted_broadcast = theta_shifted[:, np.newaxis, :] 
    
    # Step B: Run batched estimator
    job = estimator.run([
        (ansatz, observables_x, theta_shifted_broadcast),
        (ansatz, observables_y, theta_shifted_broadcast),
        (ansatz, observables_z, theta_shifted_broadcast)
    ])
    results = job.result()
    
    # Safely extract and orient the results to shape (len(observables), 2*num_params)
    def extract_evs(pub_result):
        evs = np.atleast_2d(pub_result.data.evs)
        # Transpose if the shape is (240, 60) to make it (60, 240)
        if evs.shape[0] == 2 * num_params:
            evs = evs.T
        return evs

    evs_x = extract_evs(results[0])
    evs_y = extract_evs(results[1])
    evs_z = extract_evs(results[2])
    
    # Stack vertically to get shape (num_nodes, 2*num_params)
    all_evs = np.vstack([evs_x, evs_y, evs_z])
    
    # Step C: Compute Jacobian dE/dTheta
    J = np.zeros((num_nodes, num_params))
    for j in range(num_params):
        evs_plus = all_evs[:, 2*j]
        evs_minus = all_evs[:, 2*j + 1]
        J[:, j] = (evs_plus - evs_minus) / 2.0
        
    # Step D: Apply Chain Rule
    E_current = evaluate_expectations(theta)
    loss, dL_dE = calc_loss_and_chain_rule(E_current)
    grad = dL_dE @ J
    
    # Save for bit-swap decoding
    global experiment_result
    experiment_result.append({"loss": loss, "exp_map": dict(enumerate(E_current))})
    
    return loss, grad

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue2010_linear4.npy"
num_new_params = ansatz.num_parameters # For reps=4 and 12 qubits, this will be 120

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize_param_shift(theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    print(f"Starting optimization with Parameter Shift Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    for i in range(max_iter):
        t += 1
        
        loss, grad = param_shift_gradient(theta_i)
        print(f"--- Adam Step {i+1}/{max_iter} --- | Loss: {loss:.6f}")
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            break
            
        theta_i = theta_next
        
    return theta_i

best_params = adam_optimize_param_shift(
    theta0=initial_params,
    lr=0.1,        
    max_iter=1000
)

print("\nAdam Optimization Complete!")
np.save("pce_optimal_theta_adam_reps4qbt12Continue2010_linear6.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"\nInitial Cut Size before Local Search: {best_cut}")

# Single-bit swap search as described in the paper
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] # Flip 1 to 0 or 0 to 1
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    # Keep the swap if it improves the cut
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue2010_linear4.npy...
Starting optimization with Parameter Shift Adam (lr=0.1)...


/tmp/ipykernel_200474/310847011.py:59: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)


--- Adam Step 1/1000 --- | Loss: -5079.877110
--- Adam Step 2/1000 --- | Loss: -2131.137542
--- Adam Step 3/1000 --- | Loss: -3565.915300
--- Adam Step 4/1000 --- | Loss: -4109.418107
--- Adam Step 5/1000 --- | Loss: -3903.744178
--- Adam Step 6/1000 --- | Loss: -4384.596007
--- Adam Step 7/1000 --- | Loss: -4394.408118
--- Adam Step 8/1000 --- | Loss: -4347.432302
--- Adam Step 9/1000 --- | Loss: -4515.026588
--- Adam Step 10/1000 --- | Loss: -4668.815423
--- Adam Step 11/1000 --- | Loss: -4630.707595
--- Adam Step 12/1000 --- | Loss: -4623.083467
--- Adam Step 13/1000 --- | Loss: -4725.522965
--- Adam Step 14/1000 --- | Loss: -4820.166722
--- Adam Step 15/1000 --- | Loss: -4814.928423
--- Adam Step 16/1000 --- | Loss: -4826.551332
--- Adam Step 17/1000 --- | Loss: -4854.024026
--- Adam Step 18/1000 --- | Loss: -4916.345121
--- Adam Step 19/1000 --- | Loss: -4952.219293
--- Adam Step 20/1000 --- | Loss: -4955.139011
--- Adam Step 21/1000 --- | Loss: -4958.314300
--- Adam Step 22/1000 

In [6]:
print("hi")

hi


In [13]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12                 # Increased to 12 to support 180 nodes at k=2
k = 2                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
# High shots reduce statistical noise for the parameter shift gradients
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. LOSS FUNCTION & CHAIN RULE GRADIENTS
# ==========================================
alpha_loss = 24  # Rescaling factor scaled to approx n^(k/2)
beta_loss = 0.5
# Edwards-Erdös bound for the regularization parameter v
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def evaluate_expectations(theta):
    """Helper to get pure expectation values for a single parameter array."""
    job = estimator.run([
        (ansatz, observables_x, theta),
        (ansatz, observables_y, theta),
        (ansatz, observables_z, theta)
    ])
    results = job.result()
    
    # Extract 1D arrays of expectation values
    E_x = results[0].data.evs
    E_y = results[1].data.evs
    E_z = results[2].data.evs
    return np.concatenate([E_x, E_y, E_z])

def calc_loss_and_chain_rule(E):
    """Analytically calculates the loss and the gradient dL/dE."""
    # Precompute common terms for speed
    tE = np.tanh(alpha_loss * E)
    sech2E = 1.0 - tE**2
    
    loss_cut = 0.0
    dL_dE = np.zeros(num_nodes)
    
    # Chain rule for the Cut Loss term
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss_cut += w * tE[edge0] * tE[edge1]
        
        # dL/dE for connected nodes
        dL_dE[edge0] += w * alpha_loss * sech2E[edge0] * tE[edge1]
        dL_dE[edge1] += w * alpha_loss * sech2E[edge1] * tE[edge0]
        
    # Chain rule for the Regularization term
    S = np.sum(tE**2) / num_nodes
    reg_term = beta_loss * v * (S**2)
    loss = loss_cut + reg_term
    
    # Add regularization gradient to dL/dE
    reg_grad_factor = (4.0 * beta_loss * v * S / num_nodes) * alpha_loss
    dL_dE += reg_grad_factor * tE * sech2E
    
    return loss, dL_dE

# ==========================================
# 4.5. PARAMETER SHIFT RULE JACOBIAN (CORRECTED)
# ==========================================
def param_shift_gradient(theta):
    """Calculates the full gradient using batched Parameter Shift + Chain Rule."""
    num_params = len(theta)
    shift = np.pi / 2.0
    
    # Step A: Build a batched array of shifted parameters
    theta_shifted = []
    for j in range(num_params):
        t_plus = theta.copy()
        t_plus[j] += shift
        t_minus = theta.copy()
        t_minus[j] -= shift
        theta_shifted.append(t_plus)
        theta_shifted.append(t_minus)
        
    theta_shifted = np.array(theta_shifted) # Shape: (2*num_params, num_params)
    
    # FIX: Add a new axis to force a Cartesian product broadcast
    # Now it evaluates all 240 parameter sets against all 60 observables
    theta_shifted_broadcast = theta_shifted[:, np.newaxis, :] 
    
    # Step B: Run batched estimator
    job = estimator.run([
        (ansatz, observables_x, theta_shifted_broadcast),
        (ansatz, observables_y, theta_shifted_broadcast),
        (ansatz, observables_z, theta_shifted_broadcast)
    ])
    results = job.result()
    
    # Safely extract and orient the results to shape (len(observables), 2*num_params)
    def extract_evs(pub_result):
        evs = np.atleast_2d(pub_result.data.evs)
        # Transpose if the shape is (240, 60) to make it (60, 240)
        if evs.shape[0] == 2 * num_params:
            evs = evs.T
        return evs

    evs_x = extract_evs(results[0])
    evs_y = extract_evs(results[1])
    evs_z = extract_evs(results[2])
    
    # Stack vertically to get shape (num_nodes, 2*num_params)
    all_evs = np.vstack([evs_x, evs_y, evs_z])
    
    # Step C: Compute Jacobian dE/dTheta
    J = np.zeros((num_nodes, num_params))
    for j in range(num_params):
        evs_plus = all_evs[:, 2*j]
        evs_minus = all_evs[:, 2*j + 1]
        J[:, j] = (evs_plus - evs_minus) / 2.0
        
    # Step D: Apply Chain Rule
    E_current = evaluate_expectations(theta)
    loss, dL_dE = calc_loss_and_chain_rule(E_current)
    grad = dL_dE @ J
    
    # Save for bit-swap decoding
    global experiment_result
    experiment_result.append({"loss": loss, "exp_map": dict(enumerate(E_current))})
    
    return loss, grad

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue2010_linear6_zrzmijo.npy"
num_new_params = ansatz.num_parameters # For reps=4 and 12 qubits, this will be 120

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize_param_shift(theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    print(f"Starting optimization with Parameter Shift Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    for i in range(max_iter):
        t += 1
        
        loss, grad = param_shift_gradient(theta_i)
        print(f"--- Adam Step {i+1}/{max_iter} --- | Loss: {loss:.6f}")
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            break
            
        theta_i = theta_next
        
    return theta_i

best_params = adam_optimize_param_shift(
    theta0=initial_params,
    lr=0.1,        
    max_iter=10
)

print("\nAdam Optimization Complete!")
np.save("pce_optimal_theta_adam_reps4qbt12Continue2010_linear623.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"\nInitial Cut Size before Local Search: {best_cut}")

# Single-bit swap search as described in the paper
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] # Flip 1 to 0 or 0 to 1
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    # Keep the swap if it improves the cut
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue2010_linear6_zrzmijo.npy...
Starting optimization with Parameter Shift Adam (lr=0.1)...


/tmp/ipykernel_200474/1163486773.py:59: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)


--- Adam Step 1/10 --- | Loss: -4829.430403
--- Adam Step 2/10 --- | Loss: -2701.060495
--- Adam Step 3/10 --- | Loss: -3699.488164
--- Adam Step 4/10 --- | Loss: -4465.936251


KeyboardInterrupt: 

In [11]:
# ==========================================
# 7. DEEP DECODING & BIT-SWAP POST-PROCESSING
# ==========================================

# 1. Extract initial bitstring from the Quantum Circuit's Expectation Values
cur_bits = np.zeros(num_nodes, dtype=int)
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits[i] = 1
    else:
        cur_bits[i] = 0

# Helper to calculate the full absolute cut size once
def calculate_full_cut(graph, bits):
    cut = 0
    for u, v, data in graph.edges(data=True):
        if bits[u] != bits[v]:
            cut += data.get('weight', 1.0)
    return cut

current_cut = calculate_full_cut(graph, cur_bits)
print(f"\nInitial Quantum Cut (Before Local Search): {current_cut}")

# 2. Iterative 1-Bit and 2-Bit Swap Search
search_iteration = 2
improvement_found = True

while improvement_found:
    improvement_found = False
    search_iteration += 1
    
    # --- PHASE 1: 1-BIT SWAP SEARCH ---
    # We sweep through all nodes. If flipping a node improves the cut, we keep it.
    for u in graph.nodes():
        delta = 0
        # Calculate exactly how the cut changes if we flip node `u`
        for v in graph.neighbors(u):
            w = graph[u][v].get('weight', 1.0)
            if cur_bits[u] == cur_bits[v]:
                delta += w  # They were in the same partition. Flipping u cuts this edge.
            else:
                delta -= w  # They were in different partitions. Flipping u un-cuts it.
                
        # If flipping strictly improves the cut, apply the flip
        if delta > 0:
            cur_bits[u] = 1 - cur_bits[u]
            current_cut += delta
            improvement_found = True

    # --- PHASE 2: 2-BIT SWAP SEARCH ---
    # If 1-bit swaps get stuck in a local minimum, we try flipping pairs of connected nodes
    if not improvement_found:
        for u, v in graph.edges():
            delta = 0
            
            # Change in cut for neighbors of u (excluding v)
            for n_u in graph.neighbors(u):
                if n_u == v: continue
                w = graph[u][n_u].get('weight', 1.0)
                if cur_bits[u] == cur_bits[n_u]: delta += w
                else: delta -= w
                    
            # Change in cut for neighbors of v (excluding u)
            for n_v in graph.neighbors(v):
                if n_v == u: continue
                w = graph[v][n_v].get('weight', 1.0)
                if cur_bits[v] == cur_bits[n_v]: delta += w
                else: delta -= w
                    
            if delta > 0:
                cur_bits[u] = 1 - cur_bits[u]
                cur_bits[v] = 1 - cur_bits[v]
                current_cut += delta
                improvement_found = True
                break # Break to restart the 1-bit sweep with the new landscape

print(f"Post-processing complete after {search_iteration} sweeps.")
print(f"Final Deep-Optimized Cut Size: {current_cut}")

IndexError: list index out of range

In [14]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile
from qiskit.primitives import BackendEstimatorV2


# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12                 # Increased to 12 to support 180 nodes at k=2
k = 2                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
# High shots reduce statistical noise for the parameter shift gradients
estimator.options.default_shots = 10000

experiment_result = []


from pytket.extensions.qiskit import qiskit_to_tk
from pytket.extensions.pyquil import tk_to_pyquil
from pyquil.paulis import sI, sX, sY, sZ
from pytket.passes import AutoRebase
from pyquil.api import WavefunctionSimulator
import numpy as np
from pytket.extensions.qiskit import qiskit_to_tk
from pytket.extensions.pyquil import tk_to_pyquil
from pytket.passes import AutoRebase
from pytket.circuit import OpType
from pyquil.paulis import sI, sX, sY, sZ
from pyquil.api import WavefunctionSimulator
import numpy as np

# ==========================================
# 8. TRANSLATE TO NATIVE PYQUIL VIA PYTKET
# ==========================================
print("\nTranslating Qiskit circuit to PyQuil via PyTKET...")

# 1. Bind the trained parameters to your Qiskit ansatz
best_params_loaded = np.load("pce_optimal_theta_adam_reps4qbt12Continue2010_linear4.npy")
bound_circuit = ansatz.assign_parameters(best_params_loaded)

# 2. Convert Qiskit -> TKET -> PyQuil Program
tket_circ = qiskit_to_tk(bound_circuit)


rebase = AutoRebase({OpType.CX, OpType.Rx, OpType.Rz})
rebase.apply(tket_circ)

pyquil_prog = tk_to_pyquil(tket_circ)

# 3. Translate Qiskit SparsePauliOp observables to PyQuil PauliTerms
def qiskit_to_pyquil_pauli(qiskit_op):
    """
    Converts a Qiskit SparsePauliOp to a PyQuil PauliTerm.
    Note: Qiskit strings are read right-to-left for qubit 0 -> n.
    """
    pauli_str = qiskit_op.paulis.to_labels()[0]
    term = sI()  # Start with the Identity multiplier
    
    # Enumerate backwards to match Qiskit's endianness
    for qubit_idx, char in enumerate(reversed(pauli_str)):
        if char == 'X':
            term *= sX(qubit_idx)
        elif char == 'Y':
            term *= sY(qubit_idx)
        elif char == 'Z':
            term *= sZ(qubit_idx)
    return term

print("Translating observables...")
pyquil_obs_x = [qiskit_to_pyquil_pauli(op) for op in observables_x]
pyquil_obs_y = [qiskit_to_pyquil_pauli(op) for op in observables_y]
pyquil_obs_z = [qiskit_to_pyquil_pauli(op) for op in observables_z]

# ==========================================
# 9. EVALUATE ON PYQUIL QVM & DECODE
# ==========================================
print("Evaluating exact expectation values on PyQuil QVM WavefunctionSimulator...")

wfn_sim = WavefunctionSimulator()

# Calculate expectation values by passing the entire list of PauliTerms directly!
# This natively returns a numpy array, which is much faster than looping.
E_x_qvm = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_x))
E_y_qvm = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_y))
E_z_qvm = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_z))

E_final_qvm = np.concatenate([E_x_qvm, E_y_qvm, E_z_qvm])
print("QVM Evaluation Complete!\n")

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

# Decode the bits directly from the QVM expectations
cur_bits = []
for i in range(num_nodes):
    if E_final_qvm[i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut based on QVM
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"Initial Cut Size from PyQuil QVM before Local Search: {best_cut}")

# Single-bit swap search
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post QVM):", best_cut)


from pyquil.quilbase import Gate

two_qubit_gates = [
    inst for inst in pyquil_prog
    if isinstance(inst, Gate) and len(inst.qubits) == 2
]

print("Number of 2-qubit gates:", len(two_qubit_gates))

Loading dataset...

Translating Qiskit circuit to PyQuil via PyTKET...
Translating observables...
Evaluating exact expectation values on PyQuil QVM WavefunctionSimulator...


/tmp/ipykernel_200474/3314144265.py:61: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)


QVM Evaluation Complete!

Initial Cut Size from PyQuil QVM before Local Search: 3783.037674049377
Final Optimized Cut Size (Post QVM): 5774.979137953752
Number of 2-qubit gates: 44


In [19]:
import numpy as np
import pandas as pd
import networkx as nx
from qiskit import transpile
from qiskit.circuit.library import EfficientSU2
from qiskit_aer.primitives import EstimatorV2

# 1. SETUP GRAPH & CONSTANTS
df = pd.read_parquet("DataB.parquet")
graph = nx.Graph()
for _, row in df.iterrows():
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())
num_qubits = 12
k = 2

# 2. RECONSTRUCT OBSERVABLES (PCE)
def build_pce_strings(pauli, node_count, n, k):
    from itertools import combinations
    encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= node_count: break
        p_list = ["I"] * n
        for q_idx in c: p_list[q_idx] = pauli
        encoding.append("".join(p_list))
    return encoding

list_size = num_nodes // 3
ops_strings = (build_pce_strings("X", list_size, num_qubits, k) +
               build_pce_strings("Y", list_size, num_qubits, k) +
               build_pce_strings("Z", list_size, num_qubits, k))

from qiskit.quantum_info import SparsePauliOp
observables = [SparsePauliOp.from_list([(s, 1)]) for s in ops_strings]

# 3. LOAD SAVED PARAMETERS & BIND TO ANSATZ
saved_weights_path = "pce_optimal_theta_adam_reps4qbt12Continue2010_linear4.npy"
best_params = np.load(saved_weights_path)

ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)
# Bind the loaded parameters to the circuit
bound_circuit = ansatz.assign_parameters(best_params)
bound_circuit = transpile(bound_circuit, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

# 4. EXECUTE INFERENCE
print("Running circuit with saved parameters...")
estimator = EstimatorV2()
job = estimator.run([(bound_circuit, observables)])
result = job.result()[0].data.evs

# 5. DECODE & CALCULATE CUT
def get_cut_size(graph, bits):
    cut = 0
    for u, v, data in graph.edges(data=True):
        if bits[u] != bits[v]:
            cut += data.get('weight', 1.0)
    return cut

# Thresholding expectation values to bits (0 or 1)
final_bits = [1 if val >= 0 else 0 for val in result]
final_cut = get_cut_size(graph, final_bits)

print("-" * 30)
print(f"Results for: {saved_weights_path}")
print(f"Final Cut Size: {final_cut}")
print("-" * 30)

Running circuit with saved parameters...
------------------------------
Results for: pce_optimal_theta_adam_reps4qbt12Continue2010_linear4.npy
Final Cut Size: 6583.841310242843
------------------------------


/tmp/ipykernel_200474/1120493510.py:41: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)


In [20]:
import numpy as np
import pandas as pd
import networkx as nx
import random
from qiskit import transpile
from qiskit.circuit.library import EfficientSU2
from qiskit_aer.primitives import EstimatorV2
from qiskit.quantum_info import SparsePauliOp
from itertools import combinations

# ==========================================
# 1. SETUP GRAPH & CONSTANTS
# ==========================================
df = pd.read_parquet("DataB.parquet")
graph = nx.Graph()
for _, row in df.iterrows():
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())
num_qubits = 12
k = 2

# ==========================================
# 2. CLASSICAL LOCAL SEARCH ALGORITHM
# ==========================================
def get_cut_size(graph, bits):
    """Calculates the Max-Cut size for a given bitstring."""
    cut = 0
    for u, v, data in graph.edges(data=True):
        if bits[u] != bits[v]:
            cut += data.get('weight', 1.0)
    return cut

def local_search_maxcut(graph, initial_bits):
    """Greedy local search that flips nodes to improve the cut."""
    current_bits = list(initial_bits)
    nodes = list(graph.nodes())
    improved = True
    
    while improved:
        improved = False
        random.shuffle(nodes) # Prevent looping on ties
        
        for node in nodes:
            current_side = current_bits[node]
            gain = 0
            
            # Calculate the net change in cut weight if we flip this node
            for neighbor in graph.neighbors(node):
                weight = graph[node][neighbor].get('weight', 1.0)
                if current_bits[neighbor] == current_side:
                    gain += weight  # Flipping adds this edge to the cut
                else:
                    gain -= weight  # Flipping removes this edge from the cut
            
            # If the cut gets bigger, flip it!
            if gain > 0:
                current_bits[node] = 1 - current_side
                improved = True
                
    final_cut = get_cut_size(graph, current_bits)
    return final_cut, current_bits

# ==========================================
# 3. RECONSTRUCT OBSERVABLES (PCE)
# ==========================================
def build_pce_strings(pauli, node_count, n, k):
    encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= node_count: break
        p_list = ["I"] * n
        for q_idx in c: p_list[q_idx] = pauli
        encoding.append("".join(p_list))
    return encoding

list_size = num_nodes // 3
ops_strings = (build_pce_strings("X", list_size, num_qubits, k) +
               build_pce_strings("Y", list_size, num_qubits, k) +
               build_pce_strings("Z", list_size, num_qubits, k))

observables = [SparsePauliOp.from_list([(s, 1)]) for s in ops_strings]

# ==========================================
# 4. GET QUANTUM STARTING POINT
# ==========================================
saved_weights_path = "pce_optimal_theta_adam_reps4qbt12Continue2010_linear4.npy"
best_params = np.load(saved_weights_path)

ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)
bound_circuit = ansatz.assign_parameters(best_params)
bound_circuit = transpile(bound_circuit, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

print("Running quantum circuit inference...")
estimator = EstimatorV2()
job = estimator.run([(bound_circuit, observables)])
result = job.result()[0].data.evs

# Threshold expectation values to bits (0 or 1)
quantum_initial_bits = [1 if val >= 0 else 0 for val in result]
quantum_initial_cut = get_cut_size(graph, quantum_initial_bits)

# ==========================================
# 5. RUN COMPARISON EXPERIMENT
# ==========================================
print("\n" + "="*40)
print(" EXPERIMENT: QUANTUM VS. CLASSICAL START ")
print("="*40)

# --- A. Quantum + Local Search ---
print(f"-> Base Quantum Cut (Before Optimization): {quantum_initial_cut}")
q_final_cut, _ = local_search_maxcut(graph, quantum_initial_bits)
print(f"-> Optimized Quantum Cut (After Local Search): {q_final_cut}")

# --- B. Classical Random + Local Search ---
num_random_trials = 5
classical_cuts = []

for i in range(num_random_trials):
    # Generate a completely random bitstring
    random_bits = [random.randint(0, 1) for _ in range(num_nodes)]
    c_final_cut, _ = local_search_maxcut(graph, random_bits)
    classical_cuts.append(c_final_cut)

print("-" * 40)
print(f"-> Classical Random Cut (Avg of {num_random_trials} runs): {np.mean(classical_cuts)}")
print(f"-> Classical Random Cut (Best of {num_random_trials} runs): {max(classical_cuts)}")
print("="*40)

Running quantum circuit inference...

 EXPERIMENT: QUANTUM VS. CLASSICAL START 
-> Base Quantum Cut (Before Optimization): 6583.841310242843
-> Optimized Quantum Cut (After Local Search): 6625.727891191809
----------------------------------------
-> Classical Random Cut (Avg of 5 runs): 6412.537832707793
-> Classical Random Cut (Best of 5 runs): 6727.589077022296


/tmp/ipykernel_200474/308143841.py:89: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)


In [30]:
import numpy as np
import pandas as pd
import networkx as nx
import random
from qiskit import transpile
from qiskit.circuit.library import EfficientSU2
from qiskit_aer.primitives import EstimatorV2
from qiskit.quantum_info import SparsePauliOp
from itertools import combinations

# ==========================================
# 1. SETUP GRAPH & CONSTANTS
# ==========================================
df = pd.read_parquet("DataB.parquet")
graph = nx.Graph()
for _, row in df.iterrows():
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())
num_qubits = 12
k = 2

# ==========================================
# 2. ALGORITHMS: GREEDY & SIMULATED ANNEALING
# ==========================================
def get_cut_size(graph, bits):
    cut = 0
    for u, v, data in graph.edges(data=True):
        if bits[u] != bits[v]:
            cut += data.get('weight', 1.0)
    return cut

def local_search_greedy(graph, initial_bits):
    """Greedy local search (1-opt hill climbing)."""
    current_bits = list(initial_bits)
    nodes = list(graph.nodes())
    improved = True
    
    while improved:
        improved = False
        random.shuffle(nodes)
        for node in nodes:
            current_side = current_bits[node]
            gain = sum(graph[node][nbr].get('weight', 1.0) if current_bits[nbr] == current_side 
                       else -graph[node][nbr].get('weight', 1.0) for nbr in graph.neighbors(node))
            
            if gain > 0:
                current_bits[node] = 1 - current_side
                improved = True
                
    return get_cut_size(graph, current_bits), current_bits

def simulated_annealing(graph, initial_bits, t_init=10.0, alpha=0.99, max_steps=3000):
    """Simulated Annealing to escape local maxima."""
    current_bits = list(initial_bits)
    best_bits = list(initial_bits)
    
    current_cut = get_cut_size(graph, current_bits)
    best_cut = current_cut
    
    temp = t_init
    nodes = list(graph.nodes())
    
    for step in range(max_steps):
        node = random.choice(nodes)
        current_side = current_bits[node]
        
        # Calculate gain if flipped
        gain = sum(graph[node][nbr].get('weight', 1.0) if current_bits[nbr] == current_side 
                   else -graph[node][nbr].get('weight', 1.0) for nbr in graph.neighbors(node))
        
        # Accept if it improves the cut, OR probabilistically if it's worse (based on temp)
        if gain > 0 or random.random() < np.exp(gain / temp):
            current_bits[node] = 1 - current_side
            current_cut += gain
            
            if current_cut > best_cut:
                best_cut = current_cut
                best_bits = list(current_bits)
        
        # Cool down the temperature
        temp *= alpha
        if temp < 1e-5: 
            break
            
    return best_cut, best_bits

# ==========================================
# 3. QUANTUM INFERENCE (PCE)
# ==========================================
def build_pce_strings(pauli, node_count, n, k):
    encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= node_count: break
        p_list = ["I"] * n
        for q_idx in c: p_list[q_idx] = pauli
        encoding.append("".join(p_list))
    return encoding

def deep_greedy_search(graph, initial_bits):
    """Variable-depth local search (Kernighan-Lin/Fiduccia-Mattheyses inspired)."""
    current_bits = list(initial_bits)
    best_overall_cut = get_cut_size(graph, current_bits)
    
    while True:
        unlocked_nodes = set(graph.nodes())
        temp_bits = list(current_bits)
        temp_cut = best_overall_cut
        
        sequence_history = [] # Stores (node_flipped, resulting_cut)
        
        # Explore a deep sequence of flips
        while unlocked_nodes:
            best_gain = -float('inf')
            best_node = None
            
            # Find the best node to flip among unlocked ones (even if gain is negative)
            for node in unlocked_nodes:
                current_side = temp_bits[node]
                gain = sum(graph[node][nbr].get('weight', 1.0) if temp_bits[nbr] == current_side 
                           else -graph[node][nbr].get('weight', 1.0) for nbr in graph.neighbors(node))
                
                if gain > best_gain:
                    best_gain = gain
                    best_node = node
            
            # Flip the best node found
            temp_bits[best_node] = 1 - temp_bits[best_node]
            temp_cut += best_gain
            unlocked_nodes.remove(best_node)
            sequence_history.append((best_node, temp_cut))
        
        # Find the absolute peak in our deep exploration sequence
        max_cut_in_seq = -float('inf')
        best_seq_end = -1
        
        for i, (node, cut_val) in enumerate(sequence_history):
            if cut_val > max_cut_in_seq:
                max_cut_in_seq = cut_val
                best_seq_end = i
                
        # If the highest peak we saw is better than our starting point, move there!
        if max_cut_in_seq > best_overall_cut:
            best_overall_cut = max_cut_in_seq
            # Apply all the flips required to reach that peak
            for i in range(best_seq_end + 1):
                node_to_flip = sequence_history[i][0]
                current_bits[node_to_flip] = 1 - current_bits[node_to_flip]
        else:
            # We looked as deep as possible and found no higher peaks. We are done.
            break
            
    return best_overall_cut, current_bits

list_size = num_nodes // 3
ops_strings = (build_pce_strings("X", list_size, num_qubits, k) +
               build_pce_strings("Y", list_size, num_qubits, k) +
               build_pce_strings("Z", list_size, num_qubits, k))

observables = [SparsePauliOp.from_list([(s, 1)]) for s in ops_strings]

saved_weights_path = "pce_optimal_theta_adam_reps4qbt12Continue2010_linear4.npy"
best_params = np.load(saved_weights_path)

ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)
bound_circuit = ansatz.assign_parameters(best_params)
bound_circuit = transpile(bound_circuit, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

print("Running quantum circuit inference...\n")
estimator = EstimatorV2()
result = estimator.run([(bound_circuit, observables)]).result()[0].data.evs

# Threshold expectation values to bits (0 or 1)
quantum_initial_bits = [1 if val >= 0 else 0 for val in result]
quantum_initial_cut = get_cut_size(graph, quantum_initial_bits)

# ==========================================
# 4. EXPLICIT COMPARISON EXPERIMENT
# ==========================================
num_trials = 1000

print("="*60)
print(" MAX-CUT OPTIMIZATION COMPARISON ")
print("="*60)
print(f"Base Quantum Cut (No optimization)      : {quantum_initial_cut:.2f}\n")

# --- 1. GREEDY LOCAL SEARCH ---
print("--- 1. GREEDY HILL-CLIMBING ---")
q_greedy_cut, _ = local_search_greedy(graph, quantum_initial_bits)
print(f"Quantum Start + Greedy                  : {q_greedy_cut:.2f}")

c_greedy_cuts = [local_search_greedy(graph, [random.randint(0, 1) for _ in range(num_nodes)])[0] for _ in range(num_trials)]
print(f"Random Start + Greedy (Avg of {num_trials})        : {np.mean(c_greedy_cuts):.2f}  | Best: {max(c_greedy_cuts):.2f}\n")

# --- 2. SIMULATED ANNEALING ---
print("--- 2. SIMULATED ANNEALING ---")
q_sa_cut, _ = simulated_annealing(graph, quantum_initial_bits)
print(f"Quantum Start + SA                      : {q_sa_cut:.2f}")

c_sa_cuts = [simulated_annealing(graph, [random.randint(0, 1) for _ in range(num_nodes)])[0] for _ in range(num_trials)]
print(f"Random Start + SA (Avg of {num_trials})            : {np.mean(c_sa_cuts):.2f}  | Best: {max(c_sa_cuts):.2f}")
print("="*60)


# --- 3. DEEP GREEDY (VARIABLE-DEPTH) ---
print("--- 3. DEEP GREEDY (KL-INSPIRED) ---")
q_deep_cut, _ = deep_greedy_search(graph, quantum_initial_bits)
print(f"Quantum Start + Deep Greedy             : {q_deep_cut:.2f}")

num_trials = 1
c_deep_cuts = [deep_greedy_search(graph, [random.randint(0, 1) for _ in range(num_nodes)])[0] for _ in range(num_trials)]
print(f"Random Start + Deep Greedy (Avg of {num_trials})   : {np.mean(c_deep_cuts):.2f}  | Best: {max(c_deep_cuts):.2f}")
print("="*60)

/tmp/ipykernel_200474/1177080153.py:165: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)


Running quantum circuit inference...

 MAX-CUT OPTIMIZATION COMPARISON 
Base Quantum Cut (No optimization)      : 6583.84

--- 1. GREEDY HILL-CLIMBING ---
Quantum Start + Greedy                  : 6625.73
Random Start + Greedy (Avg of 1000)        : 6417.66  | Best: 6778.49

--- 2. SIMULATED ANNEALING ---
Quantum Start + SA                      : 6625.73
Random Start + SA (Avg of 1000)            : 6441.26  | Best: 6774.47
--- 3. DEEP GREEDY (KL-INSPIRED) ---
Quantum Start + Deep Greedy             : 6956.53


KeyboardInterrupt: 

In [33]:
import numpy as np
import pandas as pd
import networkx as nx
import random
from qiskit import transpile
from qiskit.circuit.library import EfficientSU2
from qiskit_aer.primitives import EstimatorV2
from qiskit.quantum_info import SparsePauliOp
from itertools import combinations

# ==========================================
# 1. SETUP GRAPH & CONSTANTS
# ==========================================
df = pd.read_parquet("DataB.parquet")
graph = nx.Graph()
for _, row in df.iterrows():
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())
num_qubits = 12
k = 2

# ==========================================
# 2. ALGORITHMS: GREEDY & SIMULATED ANNEALING
# ==========================================
def get_cut_size(graph, bits):
    cut = 0
    for u, v, data in graph.edges(data=True):
        if bits[u] != bits[v]:
            cut += data.get('weight', 1.0)
    return cut

def local_search_greedy(graph, initial_bits):
    """Greedy local search (1-opt hill climbing)."""
    current_bits = list(initial_bits)
    nodes = list(graph.nodes())
    improved = True
    
    while improved:
        improved = False
        random.shuffle(nodes)
        for node in nodes:
            current_side = current_bits[node]
            gain = sum(graph[node][nbr].get('weight', 1.0) if current_bits[nbr] == current_side 
                       else -graph[node][nbr].get('weight', 1.0) for nbr in graph.neighbors(node))
            
            if gain > 0:
                current_bits[node] = 1 - current_side
                improved = True
                
    return get_cut_size(graph, current_bits), current_bits

def simulated_annealing(graph, initial_bits, t_init=10.0, alpha=0.99, max_steps=3000):
    """Simulated Annealing to escape local maxima."""
    current_bits = list(initial_bits)
    best_bits = list(initial_bits)
    
    current_cut = get_cut_size(graph, current_bits)
    best_cut = current_cut
    
    temp = t_init
    nodes = list(graph.nodes())
    
    for step in range(max_steps):
        node = random.choice(nodes)
        current_side = current_bits[node]
        
        # Calculate gain if flipped
        gain = sum(graph[node][nbr].get('weight', 1.0) if current_bits[nbr] == current_side 
                   else -graph[node][nbr].get('weight', 1.0) for nbr in graph.neighbors(node))
        
        # Accept if it improves the cut, OR probabilistically if it's worse (based on temp)
        if gain > 0 or random.random() < np.exp(gain / temp):
            current_bits[node] = 1 - current_side
            current_cut += gain
            
            if current_cut > best_cut:
                best_cut = current_cut
                best_bits = list(current_bits)
        
        # Cool down the temperature
        temp *= alpha
        if temp < 1e-5: 
            break
            
    return best_cut, best_bits

# ==========================================
# 3. QUANTUM INFERENCE (PCE)
# ==========================================
def build_pce_strings(pauli, node_count, n, k):
    encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= node_count: break
        p_list = ["I"] * n
        for q_idx in c: p_list[q_idx] = pauli
        encoding.append("".join(p_list))
    return encoding

def deep_greedy_search(graph, initial_bits):
    """Optimized Variable-depth local search (Fiduccia-Mattheyses style)."""
    current_bits = list(initial_bits)
    best_overall_cut = get_cut_size(graph, current_bits)
    
    while True:
        temp_bits = list(current_bits)
        temp_cut = best_overall_cut
        
        # 1. Precalculate initial gains for all nodes just ONCE per pass
        gains = {}
        for node in graph.nodes():
            side = temp_bits[node]
            gains[node] = sum(graph[node][nbr].get('weight', 1.0) if temp_bits[nbr] == side 
                              else -graph[node][nbr].get('weight', 1.0) for nbr in graph.neighbors(node))
            
        unlocked = set(graph.nodes())
        sequence_history = []
        
        max_cut_in_seq = -float('inf')
        best_seq_end = -1
        
        # 2. Explore the deep sequence
        for step in range(len(graph.nodes())):
            if not unlocked:
                break
                
            # Find the unlocked node with the highest precalculated gain
            best_node = max(unlocked, key=lambda n: gains[n])
            best_gain = gains[best_node]
            
            # Flip it and record the cut
            temp_bits[best_node] = 1 - temp_bits[best_node]
            temp_cut += best_gain
            unlocked.remove(best_node)
            sequence_history.append((best_node, temp_cut))
            
            if temp_cut > max_cut_in_seq:
                max_cut_in_seq = temp_cut
                best_seq_end = step
                
            # 3. FAST UPDATE: Only update the gains of the neighbors!
            for nbr in graph.neighbors(best_node):
                if nbr in unlocked:
                    weight = graph[best_node][nbr].get('weight', 1.0)
                    # If they are now on the SAME side, flipping the neighbor will separate them (+gain)
                    if temp_bits[nbr] == temp_bits[best_node]:
                        gains[nbr] += 2 * weight
                    # If they are now on OPPOSITE sides, flipping the neighbor will join them (-gain)
                    else:
                        gains[nbr] -= 2 * weight
                        
        # 4. Check if our deep dive found a higher peak
        if max_cut_in_seq > best_overall_cut:
            best_overall_cut = max_cut_in_seq
            # Commit the flips up to the best peak we found
            for i in range(best_seq_end + 1):
                node_to_flip = sequence_history[i][0]
                current_bits[node_to_flip] = 1 - current_bits[node_to_flip]
        else:
            # No higher peaks found, we are at the top!
            break
            
    return best_overall_cut, current_bits

list_size = num_nodes // 3
ops_strings = (build_pce_strings("X", list_size, num_qubits, k) +
               build_pce_strings("Y", list_size, num_qubits, k) +
               build_pce_strings("Z", list_size, num_qubits, k))

observables = [SparsePauliOp.from_list([(s, 1)]) for s in ops_strings]

saved_weights_path = "pce_optimal_theta_adam_reps4qbt12Continue2010_linear4.npy"
best_params = np.load(saved_weights_path)

ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)
bound_circuit = ansatz.assign_parameters(best_params)
bound_circuit = transpile(bound_circuit, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

print("Running quantum circuit inference...\n")
estimator = EstimatorV2()
result = estimator.run([(bound_circuit, observables)]).result()[0].data.evs

# Threshold expectation values to bits (0 or 1)
quantum_initial_bits = [1 if val >= 0 else 0 for val in result]
quantum_initial_cut = get_cut_size(graph, quantum_initial_bits)

# ==========================================
# 4. EXPLICIT COMPARISON EXPERIMENT
# ==========================================
num_trials = 1000

print("="*60)
print(" MAX-CUT OPTIMIZATION COMPARISON ")
print("="*60)
print(f"Base Quantum Cut (No optimization)      : {quantum_initial_cut:.2f}\n")

# --- 1. GREEDY LOCAL SEARCH ---
print("--- 1. GREEDY HILL-CLIMBING ---")
q_greedy_cut, _ = local_search_greedy(graph, quantum_initial_bits)
print(f"Quantum Start + Greedy                  : {q_greedy_cut:.2f}")

c_greedy_cuts = [local_search_greedy(graph, [random.randint(0, 1) for _ in range(num_nodes)])[0] for _ in range(num_trials)]
print(f"Random Start + Greedy (Avg of {num_trials})        : {np.mean(c_greedy_cuts):.2f}  | Best: {max(c_greedy_cuts):.2f}\n")

# --- 2. SIMULATED ANNEALING ---
print("--- 2. SIMULATED ANNEALING ---")
q_sa_cut, _ = simulated_annealing(graph, quantum_initial_bits)
print(f"Quantum Start + SA                      : {q_sa_cut:.2f}")

c_sa_cuts = [simulated_annealing(graph, [random.randint(0, 1) for _ in range(num_nodes)])[0] for _ in range(num_trials)]
print(f"Random Start + SA (Avg of {num_trials})            : {np.mean(c_sa_cuts):.2f}  | Best: {max(c_sa_cuts):.2f}")
print("="*60)


# --- 3. DEEP GREEDY (VARIABLE-DEPTH) ---
print("--- 3. DEEP GREEDY (KL-INSPIRED) ---")
q_deep_cut, _ = deep_greedy_search(graph, quantum_initial_bits)
print(f"Quantum Start + Deep Greedy             : {q_deep_cut:.2f}")

num_trials = 1
c_deep_cuts = [deep_greedy_search(graph, [random.randint(0, 1) for _ in range(num_nodes)])[0] for _ in range(num_trials)]
print(f"Random Start + Deep Greedy (Avg of {num_trials})   : {np.mean(c_deep_cuts):.2f}  | Best: {max(c_deep_cuts):.2f}")
print("="*60)

/tmp/ipykernel_200474/2071616669.py:175: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)


Running quantum circuit inference...

 MAX-CUT OPTIMIZATION COMPARISON 
Base Quantum Cut (No optimization)      : 6583.84

--- 1. GREEDY HILL-CLIMBING ---
Quantum Start + Greedy                  : 6625.73
Random Start + Greedy (Avg of 1000)        : 6411.76  | Best: 6732.64

--- 2. SIMULATED ANNEALING ---
Quantum Start + SA                      : 6708.84
Random Start + SA (Avg of 1000)            : 6441.76  | Best: 6776.61
--- 3. DEEP GREEDY (KL-INSPIRED) ---
Quantum Start + Deep Greedy             : 6956.53
Random Start + Deep Greedy (Avg of 1)   : 6960.12  | Best: 6960.12


In [65]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile
from qiskit.primitives import BackendEstimatorV2


# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 12                 # Increased to 12 to support 180 nodes at k=2
k = 2                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
# High shots reduce statistical noise for the parameter shift gradients
estimator.options.default_shots = 10000

experiment_result = []


from pytket.extensions.qiskit import qiskit_to_tk
from pytket.extensions.pyquil import tk_to_pyquil
from pyquil.paulis import sI, sX, sY, sZ
from pytket.passes import AutoRebase
from pyquil.api import WavefunctionSimulator
import numpy as np
from pytket.extensions.qiskit import qiskit_to_tk
from pytket.extensions.pyquil import tk_to_pyquil
from pytket.passes import AutoRebase
from pytket.circuit import OpType
from pyquil.paulis import sI, sX, sY, sZ
from pyquil.api import WavefunctionSimulator
import numpy as np

# ==========================================
# 8. TRANSLATE TO NATIVE PYQUIL VIA PYTKET
# ==========================================
print("\nTranslating Qiskit circuit to PyQuil via PyTKET...")

# 1. Bind the trained parameters to your Qiskit ansatz
ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])
best_params_loaded = np.load("pce_optimal_theta_adam_reps4qbt12Continue2010_linear4.npy")
bound_circuit = ansatz.assign_parameters(best_params_loaded)

# 2. Convert Qiskit -> TKET -> PyQuil Program
tket_circ = qiskit_to_tk(bound_circuit)


rebase = AutoRebase({OpType.CX, OpType.Rx, OpType.Rz})
rebase.apply(tket_circ)

pyquil_prog = tk_to_pyquil(tket_circ)

# 3. Translate Qiskit SparsePauliOp observables to PyQuil PauliTerms
def qiskit_to_pyquil_pauli(qiskit_op):
    """
    Converts a Qiskit SparsePauliOp to a PyQuil PauliTerm.
    Note: Qiskit strings are read right-to-left for qubit 0 -> n.
    """
    pauli_str = qiskit_op.paulis.to_labels()[0]
    term = sI()  # Start with the Identity multiplier
    
    # Enumerate backwards to match Qiskit's endianness
    for qubit_idx, char in enumerate(reversed(pauli_str)):
        if char == 'X':
            term *= sX(qubit_idx)
        elif char == 'Y':
            term *= sY(qubit_idx)
        elif char == 'Z':
            term *= sZ(qubit_idx)
    return term

print("Translating observables...")
pyquil_obs_x = [qiskit_to_pyquil_pauli(op) for op in observables_x]
pyquil_obs_y = [qiskit_to_pyquil_pauli(op) for op in observables_y]
pyquil_obs_z = [qiskit_to_pyquil_pauli(op) for op in observables_z]

# ==========================================
# 9. EVALUATE ON PYQUIL QVM & DECODE
# ==========================================
print("Evaluating exact expectation values on PyQuil QVM WavefunctionSimulator...")

wfn_sim = WavefunctionSimulator()

# Calculate expectation values by passing the entire list of PauliTerms directly!
# This natively returns a numpy array, which is much faster than looping.
E_x_qvm = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_x))
E_y_qvm = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_y))
E_z_qvm = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_z))

E_final_qvm = np.concatenate([E_x_qvm, E_y_qvm, E_z_qvm])
print("QVM Evaluation Complete!\n")

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

# Decode the bits directly from the QVM expectations
cur_bits = []
for i in range(num_nodes):
    if E_final_qvm[i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut based on QVM
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"Initial Cut Size from PyQuil QVM before Local Search: {best_cut}")

# Single-bit swap search
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post QVM):", best_cut)


from pyquil.quilbase import Gate

two_qubit_gates = [
    inst for inst in pyquil_prog
    if isinstance(inst, Gate) and len(inst.qubits) == 2
]

print("Number of 2-qubit gates:", len(two_qubit_gates))

Loading dataset...

Translating Qiskit circuit to PyQuil via PyTKET...
Translating observables...
Evaluating exact expectation values on PyQuil QVM WavefunctionSimulator...


/tmp/ipykernel_200474/703458222.py:61: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='pairwise', reps=4)
/tmp/ipykernel_200474/703458222.py:91: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)


QVM Evaluation Complete!

Initial Cut Size from PyQuil QVM before Local Search: 6583.841310242843
Final Optimized Cut Size (Post QVM): 6625.727891191809
Number of 2-qubit gates: 44


In [36]:
from pyquil import Program, get_qc
from pyquil.gates import H, RX, MEASURE
from pyquil.quil import address_qubits
import numpy as np

# ==========================================
# 9. APPLY MAPPING & EXECUTE ON ANKAA-3
# ==========================================
print("Preparing circuit for Rigetti Ankaa-3...")

# Your chosen high-fidelity patch
qubit_mapping = {
    0: 37, 1: 38, 2: 44, 3: 45, 4: 51, 5: 52, 
    6: 53, 7: 58, 8: 59, 9: 60, 10: 30, 11: 31
}

# 1. Map the logical PyTKET program to your physical Ankaa-3 qubits
mapped_base_prog = address_qubits(pyquil_prog, qubit_mapping)

qc = get_qc("Ankaa-3")
shots = 10000

def run_mapped_basis(mapped_prog, basis, mapping, qc, shots):
    """Appends basis transformations, compiles for Ankaa-3, and executes."""
    # Enforce our mapping starting position
    prog = Program(f'PRAGMA INITIAL_REWIRE "PARTIAL"')
    prog += mapped_prog
    
    # Declare classical register to hold our 12 bits
    ro = prog.declare("ro", "BIT", len(mapping))
    
    # 1. APPLY ALL GATES FIRST
    # Apply basis rotations to the physical qubits
    for logical_q, physical_q in mapping.items():
        if basis == 'X':
            prog += H(physical_q)
        elif basis == 'Y':
            prog += RX(np.pi/2, physical_q)
            
    # 2. APPLY ALL MEASUREMENTS LAST
    # Store physical qubit measurement in the corresponding logical index
    for logical_q, physical_q in mapping.items():
        prog += MEASURE(physical_q, ro[logical_q])
        
    prog.wrap_in_numshots_loop(shots)
    
    print(f"Compiling {basis}-basis circuit...")
    exe = qc.compile(prog)
    
    print(f"Executing {basis}-basis circuit on Ankaa-3...")
    result = qc.run(exe)
    
    return result.get_register_map().get("ro")

# 2. Run the 3 global tasks on the physical hardware
res_x = run_mapped_basis(mapped_base_prog, 'X', qubit_mapping, qc, shots)
res_y = run_mapped_basis(mapped_base_prog, 'Y', qubit_mapping, qc, shots)
res_z = run_mapped_basis(mapped_base_prog, 'Z', qubit_mapping, qc, shots)

# ==========================================
# 10. DECODE BITSTRINGS & CALCULATE CUT
# ==========================================
def get_expectations(bitstrings, qiskit_observables, shots):
    """Translates raw 0s and 1s into expectation values for our specific observables."""
    spins = 1 - 2 * bitstrings 
    evs = []
    
    for obs in qiskit_observables:
        pauli_str = list(reversed(obs.paulis.to_labels()[0]))
        obs_val = np.ones(shots)
        
        for q, char in enumerate(pauli_str):
            if char != 'I':
                obs_val *= spins[:, q] 
                
        evs.append(np.mean(obs_val))
    return np.array(evs)

print("\nCalculating expectations from Ankaa-3 telemetry...")
# Because we mapped the readouts back to logical indices (0-11), 
# our original observables list works perfectly here!
E_x_qpu = get_expectations(res_x, observables_x, shots)
E_y_qpu = get_expectations(res_y, observables_y, shots)
E_z_qpu = get_expectations(res_z, observables_z, shots)

E_final_qpu = np.concatenate([E_x_qpu, E_y_qpu, E_z_qpu])
print("Ankaa-3 QPU Evaluation Complete!\n")

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if E_final_qpu[i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"Initial Cut Size from physical Ankaa-3 before Local Search: {best_cut}")

for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post Ankaa-3 execution):", best_cut)

Preparing circuit for Rigetti Ankaa-3...
Compiling X-basis circuit...
Executing X-basis circuit on Ankaa-3...
Compiling Y-basis circuit...
Executing Y-basis circuit on Ankaa-3...
Compiling Z-basis circuit...
Executing Z-basis circuit on Ankaa-3...

Calculating expectations from Ankaa-3 telemetry...
Ankaa-3 QPU Evaluation Complete!

Initial Cut Size from physical Ankaa-3 before Local Search: 2945.932224176066
Final Optimized Cut Size (Post Ankaa-3 execution): 6229.4915848839055


In [38]:
compiled_prog = exe.program
two_q_gates = [inst for inst in compiled_prog if getattr(inst, 'qubits', None) and len(inst.qubits) == 2]
print(f"Compiled 2Q gate count: {len(two_q_gates)}")

NameError: name 'exe' is not defined

In [46]:
E_y_qpu.shape

(60,)

In [43]:
print("\n==================================================")
print(" EXPECTATION VALUE COMPARISON (QVM vs. ANKAA-3) ")
print("==================================================")
print(f"{'Node':<6} | {'QVM (Ideal)':<12} | {'QPU (Noisy)':<12} | {'Difference'}")
print("-" * 50)

# Assuming E_final_qvm and E_final_qpu are both length num_nodes
for i in range(num_nodes):
    qvm_val = E_final_qvm[i]
    qpu_val = E_final_qpu[i]
    diff = abs(qvm_val - qpu_val)
    
    # Flag rows where the noise caused a sign flip (which ruins the cut logic)
    warning = " <-- SIGN FLIP!" if (qvm_val * qpu_val < 0) else ""
    
    print(f"{i:<6} | {qvm_val:>12.4f} | {qpu_val:>12.4f} | {diff:>10.4f}{warning}")


 EXPECTATION VALUE COMPARISON (QVM vs. ANKAA-3) 
Node   | QVM (Ideal)  | QPU (Noisy)  | Difference
--------------------------------------------------
0      |      -0.1076 |      -0.0006 |     0.1070
1      |       0.0902 |       0.0106 |     0.0796
2      |      -0.0789 |       0.0108 |     0.0897 <-- SIGN FLIP!
3      |      -0.0834 |      -0.0026 |     0.0808
4      |       0.1359 |      -0.0124 |     0.1483 <-- SIGN FLIP!
5      |       0.1217 |      -0.0004 |     0.1221 <-- SIGN FLIP!
6      |      -0.0679 |      -0.0004 |     0.0675
7      |       0.0624 |       0.0062 |     0.0562
8      |       0.0544 |       0.0198 |     0.0346
9      |      -0.0116 |      -0.0016 |     0.0100
10     |      -0.0601 |      -0.0174 |     0.0427
11     |      -0.0842 |      -0.0120 |     0.0722
12     |      -0.0823 |       0.0038 |     0.0861 <-- SIGN FLIP!
13     |      -0.0806 |       0.0244 |     0.1050 <-- SIGN FLIP!
14     |      -0.1707 |       0.0086 |     0.1793 <-- SIGN FLIP!
15     | 

In [66]:
from pytket.extensions.qiskit import qiskit_to_tk
from pytket.extensions.pyquil import tk_to_pyquil
from pyquil.paulis import sI, sX, sY, sZ
from pytket.passes import AutoRebase
from pyquil.api import WavefunctionSimulator
import numpy as np
from pytket.circuit import OpType
from pyquil.quilbase import Gate
from pyquil import Program, get_qc
from pyquil.gates import H, RX, MEASURE
from pyquil.quil import address_qubits

# Import Mitiq for Pauli Twirling
#from mitiq import rc

# ==========================================
# 8. PAULI TWIRLING & TRANSLATE VIA PYTKET
# ==========================================
# ... (your previous imports) ...
from pyquil.quil import address_qubits

# Corrected Import for Mitiq Pauli Twirling
from mitiq import pt

# ==========================================
# 8. PAULI TWIRLING & TRANSLATE VIA PYTKET
# ==========================================
print("\nApplying Pauli Twirling and translating to PyQuil...")

ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])
best_params_loaded = np.load("pce_optimal_theta_adam_reps4qbt12Continue2010_linear4.npy")
bound_circuit = ansatz.assign_parameters(best_params_loaded)

# --- MITIQ: Generate twirled ensemble ---
num_rc_circuits = 6
print(f"Generating {num_rc_circuits} twirled circuits...")

# Corrected Mitiq function call
twirled_qiskit_circuits = pt.generate_pauli_twirl_variants(
    bound_circuit, 
    num_circuits=num_rc_circuits
)

rebase = AutoRebase({OpType.CX, OpType.Rx, OpType.Rz})
pyquil_rc_progs = []

# Translate each twirled circuit to PyQuil
for circ in twirled_qiskit_circuits:
    tket_circ = qiskit_to_tk(circ)
    rebase.apply(tket_circ)
    pyquil_rc_progs.append(tk_to_pyquil(tket_circ))

def qiskit_to_pyquil_pauli(qiskit_op):
    pauli_str = qiskit_op.paulis.to_labels()[0]
    term = sI() 
    for qubit_idx, char in enumerate(reversed(pauli_str)):
        if char == 'X':
            term *= sX(qubit_idx)
        elif char == 'Y':
            term *= sY(qubit_idx)
        elif char == 'Z':
            term *= sZ(qubit_idx)
    return term

print("Translating observables...")
pyquil_obs_x = [qiskit_to_pyquil_pauli(op) for op in observables_x]
pyquil_obs_y = [qiskit_to_pyquil_pauli(op) for op in observables_y]
pyquil_obs_z = [qiskit_to_pyquil_pauli(op) for op in observables_z]

# ==========================================
# 9. EVALUATE ON PYQUIL QVM & DECODE
# ==========================================
print("Evaluating exact expectation values on PyQuil QVM...")

wfn_sim = WavefunctionSimulator()

# For the QVM, evaluating just the first circuit is sufficient 
# since simulators without noise don't suffer from coherent errors
E_x_qvm = np.real(wfn_sim.expectation(pyquil_rc_progs[0], pauli_terms=pyquil_obs_x))
E_y_qvm = np.real(wfn_sim.expectation(pyquil_rc_progs[0], pauli_terms=pyquil_obs_y))
E_z_qvm = np.real(wfn_sim.expectation(pyquil_rc_progs[0], pauli_terms=pyquil_obs_z))

E_final_qvm = np.concatenate([E_x_qvm, E_y_qvm, E_z_qvm])

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = [1 if val >= 0 else 0 for val in E_final_qvm]
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

# Local Search (QVM)
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post QVM):", best_cut)



Applying Pauli Twirling and translating to PyQuil...
Generating 6 twirled circuits...


/tmp/ipykernel_200474/3307758463.py:30: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)


Translating observables...
Evaluating exact expectation values on PyQuil QVM...
Final Optimized Cut Size (Post QVM): 6625.727891191809


In [56]:
# ==========================================
# 10. APPLY MAPPING & EXECUTE ON ANKAA-3
# ==========================================
print("Preparing twirled ensemble for Rigetti Ankaa-3...")

qubit_mapping = {
    0: 37, 1: 38, 2: 44, 3: 45, 4: 51, 5: 52, 
    6: 53, 7: 58, 8: 59, 9: 60, 10: 30, 11: 31
}

qc = get_qc("Ankaa-3")
total_shots = 10000
# Distribute shots evenly across the RC ensemble
shots_per_circ = total_shots // num_rc_circuits

def run_mapped_basis(mapped_prog, basis, mapping, qc, shots):
    prog = Program(f'PRAGMA INITIAL_REWIRE "PARTIAL"')
    prog += mapped_prog
    ro = prog.declare("ro", "BIT", len(mapping))
    
    for logical_q, physical_q in mapping.items():
        if basis == 'X':
            prog += H(physical_q)
        elif basis == 'Y':
            prog += RX(np.pi/2, physical_q)
            
    for logical_q, physical_q in mapping.items():
        prog += MEASURE(physical_q, ro[logical_q])
        
    prog.wrap_in_numshots_loop(shots)
    exe = qc.compile(prog)
    result = qc.run(exe)
    return result.get_register_map().get("ro")

def get_expectations(bitstrings, qiskit_observables, shots):
    spins = 1 - 2 * bitstrings 
    evs = []
    for obs in qiskit_observables:
        pauli_str = list(reversed(obs.paulis.to_labels()[0]))
        obs_val = np.ones(shots)
        for q, char in enumerate(pauli_str):
            if char != 'I':
                obs_val *= spins[:, q] 
        evs.append(np.mean(obs_val))
    return np.array(evs)

# Execute the ensemble
E_x_qpu_total, E_y_qpu_total, E_z_qpu_total = 0, 0, 0

print(f"\nExecuting {num_rc_circuits} twirled circuits on Ankaa-3 ({shots_per_circ} shots each)...")
for i, prog in enumerate(pyquil_rc_progs):
    print(f"  -> Running circuit {i+1}/{num_rc_circuits}")
    mapped_prog = address_qubits(prog, qubit_mapping)
    print("1")
    res_x = run_mapped_basis(mapped_prog, 'X', qubit_mapping, qc, shots_per_circ)
    print("2")
    res_y = run_mapped_basis(mapped_prog, 'Y', qubit_mapping, qc, shots_per_circ)
    print("3")
    res_z = run_mapped_basis(mapped_prog, 'Z', qubit_mapping, qc, shots_per_circ)
    print("4")
    
    E_x_qpu_total += get_expectations(res_x, observables_x, shots_per_circ)
    E_y_qpu_total += get_expectations(res_y, observables_y, shots_per_circ)
    E_z_qpu_total += get_expectations(res_z, observables_z, shots_per_circ)

# Average out the expectation values
E_x_qpu = E_x_qpu_total / num_rc_circuits
E_y_qpu = E_y_qpu_total / num_rc_circuits
E_z_qpu = E_z_qpu_total / num_rc_circuits

E_final_qpu = np.concatenate([E_x_qpu, E_y_qpu, E_z_qpu])
print("Ankaa-3 QPU Evaluation Complete!\n")

# --- QPU Decoding & Local Search ---
cur_bits = [1 if val >= 0 else 0 for val in E_final_qpu]
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"Initial Cut Size from Ankaa-3 before Local Search: {best_cut}")

for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post Ankaa-3 execution):", best_cut)

Preparing twirled ensemble for Rigetti Ankaa-3...

Executing 6 twirled circuits on Ankaa-3 (1666 shots each)...
  -> Running circuit 1/6
1
2
3
4
  -> Running circuit 2/6
1
2
3
4
  -> Running circuit 3/6
1
2
3
4
  -> Running circuit 4/6
1
2
3
4
  -> Running circuit 5/6
1
2
3
4
  -> Running circuit 6/6
1
2
3
4
Ankaa-3 QPU Evaluation Complete!

Initial Cut Size from Ankaa-3 before Local Search: 3016.9591019848917
Final Optimized Cut Size (Post Ankaa-3 execution): 6063.874811110602


In [53]:
print("hi")

hi


In [61]:
from pytket.extensions.qiskit import qiskit_to_tk
from pytket.extensions.pyquil import tk_to_pyquil
from pytket.passes import AutoRebase
from pytket.circuit import OpType
from pyquil.paulis import sI, sX, sY, sZ
from pyquil.api import WavefunctionSimulator
import numpy as np

# ==========================================
# 8. PREPARE OBSERVABLES & BASE ANSATZ
# ==========================================
def qiskit_to_pyquil_pauli(qiskit_op):
    pauli_str = qiskit_op.paulis.to_labels()[0]
    term = sI() 
    for qubit_idx, char in enumerate(reversed(pauli_str)):
        if char == 'X':
            term *= sX(qubit_idx)
        elif char == 'Y':
            term *= sY(qubit_idx)
        elif char == 'Z':
            term *= sZ(qubit_idx)
    return term

print("\nTranslating observables to PyQuil...")
pyquil_obs_x = [qiskit_to_pyquil_pauli(op) for op in observables_x]
pyquil_obs_y = [qiskit_to_pyquil_pauli(op) for op in observables_y]
pyquil_obs_z = [qiskit_to_pyquil_pauli(op) for op in observables_z]

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

# Base ansatz setup
ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])
num_params = ansatz.num_parameters
rebase = AutoRebase({OpType.CX, OpType.Rx, OpType.Rz})
wfn_sim = WavefunctionSimulator()

# ==========================================
# 9. RUN RANDOM BASELINE EXPERIMENT
# ==========================================
num_random_trials = 10
random_cuts_initial = []
random_cuts_optimized = []

print(f"\nEvaluating {num_random_trials} random parameter sets on PyQuil QVM...")

for trial in range(num_random_trials):
    # 1. Generate random parameters between 0 and 2*pi
    random_params = np.random.uniform(0, 2 * np.pi, num_params)
    bound_circuit = ansatz.assign_parameters(random_params)

    # 2. Translate to PyQuil
    tket_circ = qiskit_to_tk(bound_circuit)
    rebase.apply(tket_circ)
    pyquil_prog = tk_to_pyquil(tket_circ)

    # 3. Evaluate exact expectation values
    E_x = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_x))
    E_y = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_y))
    E_z = np.real(wfn_sim.expectation(pyquil_prog, pauli_terms=pyquil_obs_z))
    E_final_qvm = np.concatenate([E_x, E_y, E_z])

    # 4. Decode Bits & Calculate Initial Cut
    cur_bits = [1 if val >= 0 else 0 for val in E_final_qvm]
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(cur_bits):
        cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
            
    initial_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    random_cuts_initial.append(initial_cut)

    # 5. Single-bit Swap Local Search
    best_cut = initial_cut
    best_bits = cur_bits.copy()

    for node in range(num_nodes):
        swapped_bits = best_bits.copy()
        swapped_bits[node] = 1 - swapped_bits[node] 
        
        cur_part = [set(), set()]
        for i, bit in enumerate(swapped_bits):
            cur_part[0].add(i) if bit > 0 else cur_part[1].add(i)
                
        cut_size = calc_cut_size(graph, cur_part[0], cur_part[1])
        
        if cut_size > best_cut:
            best_cut = cut_size
            best_bits = swapped_bits
            
    random_cuts_optimized.append(best_cut)
    
    print(f"Trial {trial+1:02d} | Initial Cut: {initial_cut:.2f} | Post-Search Cut: {best_cut:.2f}")

# ==========================================
# 10. DISPLAY BASELINE STATISTICS
# ==========================================
print("\n" + "="*40)
print("🎯 RANDOM BASELINE RESULTS")
print("="*40)
print(f"Average Initial Cut:   {np.mean(random_cuts_initial):.2f}")
print(f"Average Optimized Cut: {np.mean(random_cuts_optimized):.2f}")
print(f"Worst Optimized Cut (Lower Bound): {np.min(random_cuts_optimized):.2f}")
print(f"Best Optimized Cut (Random Luck):  {np.max(random_cuts_optimized):.2f}")
print("="*40)


Translating observables to PyQuil...

Evaluating 10 random parameter sets on PyQuil QVM...


/tmp/ipykernel_200474/2765870543.py:38: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=4)


Trial 01 | Initial Cut: 3675.75 | Post-Search Cut: 6187.34
Trial 02 | Initial Cut: 3091.22 | Post-Search Cut: 6122.01
Trial 03 | Initial Cut: 3903.59 | Post-Search Cut: 6119.82
Trial 04 | Initial Cut: 3755.24 | Post-Search Cut: 6091.78
Trial 05 | Initial Cut: 3348.22 | Post-Search Cut: 6051.02
Trial 06 | Initial Cut: 3931.06 | Post-Search Cut: 5899.96
Trial 07 | Initial Cut: 3668.41 | Post-Search Cut: 6170.93
Trial 08 | Initial Cut: 3546.98 | Post-Search Cut: 6136.35
Trial 09 | Initial Cut: 4000.72 | Post-Search Cut: 6189.89
Trial 10 | Initial Cut: 3166.83 | Post-Search Cut: 6280.48

🎯 RANDOM BASELINE RESULTS
Average Initial Cut:   3608.80
Average Optimized Cut: 6124.96
Worst Optimized Cut (Lower Bound): 5899.96
Best Optimized Cut (Random Luck):  6280.48


In [69]:
# ==========================================
# 10. APPLY MAPPING & EXECUTE ON ANKAA-3
# ==========================================
from pyquil.quil import Program
#from pyquil.quilatom import Pragma
from pyquil.gates import H, RX, MEASURE
import numpy as np

print("Preparing twirled ensemble for Rigetti Ankaa-3...")

# 🏆 The Optimal Comprehensive Chain mapping (Highest T1/T2, f1Q, fRO, fISWAP)
qubit_mapping = {
    0: 2, 
    1: 9, 
    2: 8, 
    3: 7, 
    4: 14, 
    5: 21, 
    6: 22, 
    7: 15, 
    8: 16, 
    9: 23, 
    10: 30, 
    11: 31
}

qc = get_qc("Ankaa-3")
total_shots = 10000
# Distribute shots evenly across the RC ensemble
shots_per_circ = total_shots // num_rc_circuits

def run_mapped_basis(mapped_prog, basis, mapping, qc, shots):
    # FORCE the compiler to use our exact physical qubits, no auto-rerouting!
    prog = Program(f'PRAGMA INITIAL_REWIRE "NAIVE"')
    prog += mapped_prog
    
    # Declare the readout register (ro) sized to our logical circuit
    ro = prog.declare("ro", "BIT", len(mapping))
    
    # Append the basis transformation gates to the physical qubits
    for logical_q, physical_q in mapping.items():
        if basis == 'X':
            prog += H(physical_q)
        elif basis == 'Y':
            prog += RX(np.pi/2, physical_q)
            
    # Measure the physical qubits into the logical indices of the readout register
    for logical_q, physical_q in mapping.items():
        prog += MEASURE(physical_q, ro[logical_q])
        
    prog.wrap_in_numshots_loop(shots)
    exe = qc.compile(prog)
    result = qc.run(exe)
    
    # PyQuil 3.x+ syntax for extracting readout results
    if hasattr(result, "readout_data"):
        return result.readout_data.get("ro") # PyQuil 4.x
    else:
        return result.get_register_map().get("ro") # Older PyQuil 3.x

def get_expectations(bitstrings, qiskit_observables, shots):
    # Convert {0, 1} bitstrings to {+1, -1} spin expectations
    spins = 1 - 2 * bitstrings 
    evs = []
    for obs in qiskit_observables:
        pauli_str = list(reversed(obs.paulis.to_labels()[0]))
        obs_val = np.ones(shots)
        for q, char in enumerate(pauli_str):
            if char != 'I':
                obs_val *= spins[:, q] 
        evs.append(np.mean(obs_val))
    return np.array(evs)

# Execute the ensemble
E_x_qpu_total, E_y_qpu_total, E_z_qpu_total = 0, 0, 0

print(f"\nExecuting {num_rc_circuits} twirled circuits on Ankaa-3 ({shots_per_circ} shots each)...")
for i, prog in enumerate(pyquil_rc_progs):
    print(f"  -> Running circuit {i+1}/{num_rc_circuits}")
    
    # This correctly physically addresses the circuit using our mapping dictionary
    mapped_prog = address_qubits(prog, qubit_mapping)
    
    print("     [1/3] Measuring X basis...")
    res_x = run_mapped_basis(mapped_prog, 'X', qubit_mapping, qc, shots_per_circ)
    print("     [2/3] Measuring Y basis...")
    res_y = run_mapped_basis(mapped_prog, 'Y', qubit_mapping, qc, shots_per_circ)
    print("     [3/3] Measuring Z basis...")
    res_z = run_mapped_basis(mapped_prog, 'Z', qubit_mapping, qc, shots_per_circ)
    
    E_x_qpu_total += get_expectations(res_x, observables_x, shots_per_circ)
    E_y_qpu_total += get_expectations(res_y, observables_y, shots_per_circ)
    E_z_qpu_total += get_expectations(res_z, observables_z, shots_per_circ)

# Average out the expectation values
E_x_qpu = E_x_qpu_total / num_rc_circuits
E_y_qpu = E_y_qpu_total / num_rc_circuits
E_z_qpu = E_z_qpu_total / num_rc_circuits

E_final_qpu = np.concatenate([E_x_qpu, E_y_qpu, E_z_qpu])
print("\nAnkaa-3 QPU Evaluation Complete!\n")

# --- QPU Decoding & Local Search ---
cur_bits = [1 if val >= 0 else 0 for val in E_final_qpu]
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"Initial Cut Size from Ankaa-3 before Local Search: {best_cut}")

for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post Ankaa-3 execution):", best_cut)

Preparing twirled ensemble for Rigetti Ankaa-3...

Executing 6 twirled circuits on Ankaa-3 (1666 shots each)...
  -> Running circuit 1/6
     [1/3] Measuring X basis...


/tmp/ipykernel_200474/1127522019.py:56: DeprecationWarning: Call to deprecated function (or staticmethod) readout_data. (This property is ambiguous now that the `get_raw_readout_data()` method exists and will be removed in future versions. Use the `get_register_map()` method instead.) -- Deprecated since version 4.0.0.
  if hasattr(result, "readout_data"):
/tmp/ipykernel_200474/1127522019.py:57: DeprecationWarning: Call to deprecated function (or staticmethod) readout_data. (This property is ambiguous now that the `get_raw_readout_data()` method exists and will be removed in future versions. Use the `get_register_map()` method instead.) -- Deprecated since version 4.0.0.
  return result.readout_data.get("ro") # PyQuil 4.x


     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...
  -> Running circuit 2/6
     [1/3] Measuring X basis...
     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...
  -> Running circuit 3/6
     [1/3] Measuring X basis...
     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...
  -> Running circuit 4/6
     [1/3] Measuring X basis...
     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...
  -> Running circuit 5/6
     [1/3] Measuring X basis...
     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...
  -> Running circuit 6/6
     [1/3] Measuring X basis...
     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...

Ankaa-3 QPU Evaluation Complete!

Initial Cut Size from Ankaa-3 before Local Search: 3732.4668785193085
Final Optimized Cut Size (Post Ankaa-3 execution): 6120.941899205181


In [70]:
from pytket.extensions.qiskit import qiskit_to_tk
from pytket.extensions.pyquil import tk_to_pyquil
from pyquil.paulis import sI, sX, sY, sZ
from pytket.passes import AutoRebase
from pyquil.api import WavefunctionSimulator
import numpy as np
from pytket.circuit import OpType
from pyquil.quilbase import Gate
from pyquil import Program, get_qc
from pyquil.gates import H, RX, MEASURE
from pyquil.quil import address_qubits

# Import Mitiq for Pauli Twirling
#from mitiq import rc

# ==========================================
# 8. PAULI TWIRLING & TRANSLATE VIA PYTKET
# ==========================================
# ... (your previous imports) ...
from pyquil.quil import address_qubits

# Corrected Import for Mitiq Pauli Twirling
from mitiq import pt

# ==========================================
# 8. PAULI TWIRLING & TRANSLATE VIA PYTKET
# ==========================================
print("\nApplying Pauli Twirling and translating to PyQuil...")

ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=2)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])
best_params_loaded = np.load("pce_optimal_theta_adam_reps4qbt12Continue2010_linear.npy")
bound_circuit = ansatz.assign_parameters(best_params_loaded)

# --- MITIQ: Generate twirled ensemble ---
num_rc_circuits = 6
print(f"Generating {num_rc_circuits} twirled circuits...")

# Corrected Mitiq function call
twirled_qiskit_circuits = pt.generate_pauli_twirl_variants(
    bound_circuit, 
    num_circuits=num_rc_circuits
)

rebase = AutoRebase({OpType.CX, OpType.Rx, OpType.Rz})
pyquil_rc_progs = []

# Translate each twirled circuit to PyQuil
for circ in twirled_qiskit_circuits:
    tket_circ = qiskit_to_tk(circ)
    rebase.apply(tket_circ)
    pyquil_rc_progs.append(tk_to_pyquil(tket_circ))

def qiskit_to_pyquil_pauli(qiskit_op):
    pauli_str = qiskit_op.paulis.to_labels()[0]
    term = sI() 
    for qubit_idx, char in enumerate(reversed(pauli_str)):
        if char == 'X':
            term *= sX(qubit_idx)
        elif char == 'Y':
            term *= sY(qubit_idx)
        elif char == 'Z':
            term *= sZ(qubit_idx)
    return term

print("Translating observables...")
pyquil_obs_x = [qiskit_to_pyquil_pauli(op) for op in observables_x]
pyquil_obs_y = [qiskit_to_pyquil_pauli(op) for op in observables_y]
pyquil_obs_z = [qiskit_to_pyquil_pauli(op) for op in observables_z]

# ==========================================
# 9. EVALUATE ON PYQUIL QVM & DECODE
# ==========================================
print("Evaluating exact expectation values on PyQuil QVM...")

wfn_sim = WavefunctionSimulator()

# For the QVM, evaluating just the first circuit is sufficient 
# since simulators without noise don't suffer from coherent errors
E_x_qvm = np.real(wfn_sim.expectation(pyquil_rc_progs[0], pauli_terms=pyquil_obs_x))
E_y_qvm = np.real(wfn_sim.expectation(pyquil_rc_progs[0], pauli_terms=pyquil_obs_y))
E_z_qvm = np.real(wfn_sim.expectation(pyquil_rc_progs[0], pauli_terms=pyquil_obs_z))

E_final_qvm = np.concatenate([E_x_qvm, E_y_qvm, E_z_qvm])

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = [1 if val >= 0 else 0 for val in E_final_qvm]
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

# Local Search (QVM)
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post QVM):", best_cut)



Applying Pauli Twirling and translating to PyQuil...
Generating 6 twirled circuits...


/tmp/ipykernel_200474/2756668700.py:30: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=2)


Translating observables...
Evaluating exact expectation values on PyQuil QVM...
Final Optimized Cut Size (Post QVM): 6586.1080232722825


In [71]:
# ==========================================
# 10. APPLY MAPPING & EXECUTE ON ANKAA-3
# ==========================================
from pyquil.quil import Program
#from pyquil.quilatom import Pragma
from pyquil.gates import H, RX, MEASURE
import numpy as np

print("Preparing twirled ensemble for Rigetti Ankaa-3...")

# 🏆 The Optimal Comprehensive Chain mapping (Highest T1/T2, f1Q, fRO, fISWAP)
qubit_mapping = {
    0: 2, 
    1: 9, 
    2: 8, 
    3: 7, 
    4: 14, 
    5: 21, 
    6: 22, 
    7: 15, 
    8: 16, 
    9: 23, 
    10: 30, 
    11: 31
}

qc = get_qc("Ankaa-3")
total_shots = 10000
# Distribute shots evenly across the RC ensemble
shots_per_circ = total_shots // num_rc_circuits

def run_mapped_basis(mapped_prog, basis, mapping, qc, shots):
    # FORCE the compiler to use our exact physical qubits, no auto-rerouting!
    prog = Program(f'PRAGMA INITIAL_REWIRE "NAIVE"')
    prog += mapped_prog
    
    # Declare the readout register (ro) sized to our logical circuit
    ro = prog.declare("ro", "BIT", len(mapping))
    
    # Append the basis transformation gates to the physical qubits
    for logical_q, physical_q in mapping.items():
        if basis == 'X':
            prog += H(physical_q)
        elif basis == 'Y':
            prog += RX(np.pi/2, physical_q)
            
    # Measure the physical qubits into the logical indices of the readout register
    for logical_q, physical_q in mapping.items():
        prog += MEASURE(physical_q, ro[logical_q])
        
    prog.wrap_in_numshots_loop(shots)
    exe = qc.compile(prog)
    result = qc.run(exe)
    
    # PyQuil 3.x+ syntax for extracting readout results
    if hasattr(result, "readout_data"):
        return result.readout_data.get("ro") # PyQuil 4.x
    else:
        return result.get_register_map().get("ro") # Older PyQuil 3.x

def get_expectations(bitstrings, qiskit_observables, shots):
    # Convert {0, 1} bitstrings to {+1, -1} spin expectations
    spins = 1 - 2 * bitstrings 
    evs = []
    for obs in qiskit_observables:
        pauli_str = list(reversed(obs.paulis.to_labels()[0]))
        obs_val = np.ones(shots)
        for q, char in enumerate(pauli_str):
            if char != 'I':
                obs_val *= spins[:, q] 
        evs.append(np.mean(obs_val))
    return np.array(evs)

# Execute the ensemble
E_x_qpu_total, E_y_qpu_total, E_z_qpu_total = 0, 0, 0

print(f"\nExecuting {num_rc_circuits} twirled circuits on Ankaa-3 ({shots_per_circ} shots each)...")
for i, prog in enumerate(pyquil_rc_progs):
    print(f"  -> Running circuit {i+1}/{num_rc_circuits}")
    
    # This correctly physically addresses the circuit using our mapping dictionary
    mapped_prog = address_qubits(prog, qubit_mapping)
    
    print("     [1/3] Measuring X basis...")
    res_x = run_mapped_basis(mapped_prog, 'X', qubit_mapping, qc, shots_per_circ)
    print("     [2/3] Measuring Y basis...")
    res_y = run_mapped_basis(mapped_prog, 'Y', qubit_mapping, qc, shots_per_circ)
    print("     [3/3] Measuring Z basis...")
    res_z = run_mapped_basis(mapped_prog, 'Z', qubit_mapping, qc, shots_per_circ)
    
    E_x_qpu_total += get_expectations(res_x, observables_x, shots_per_circ)
    E_y_qpu_total += get_expectations(res_y, observables_y, shots_per_circ)
    E_z_qpu_total += get_expectations(res_z, observables_z, shots_per_circ)

# Average out the expectation values
E_x_qpu = E_x_qpu_total / num_rc_circuits
E_y_qpu = E_y_qpu_total / num_rc_circuits
E_z_qpu = E_z_qpu_total / num_rc_circuits

E_final_qpu = np.concatenate([E_x_qpu, E_y_qpu, E_z_qpu])
print("\nAnkaa-3 QPU Evaluation Complete!\n")

# --- QPU Decoding & Local Search ---
cur_bits = [1 if val >= 0 else 0 for val in E_final_qpu]
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"Initial Cut Size from Ankaa-3 before Local Search: {best_cut}")

for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post Ankaa-3 execution):", best_cut)

Preparing twirled ensemble for Rigetti Ankaa-3...

Executing 6 twirled circuits on Ankaa-3 (1666 shots each)...
  -> Running circuit 1/6
     [1/3] Measuring X basis...


/tmp/ipykernel_200474/1127522019.py:56: DeprecationWarning: Call to deprecated function (or staticmethod) readout_data. (This property is ambiguous now that the `get_raw_readout_data()` method exists and will be removed in future versions. Use the `get_register_map()` method instead.) -- Deprecated since version 4.0.0.
  if hasattr(result, "readout_data"):
/tmp/ipykernel_200474/1127522019.py:57: DeprecationWarning: Call to deprecated function (or staticmethod) readout_data. (This property is ambiguous now that the `get_raw_readout_data()` method exists and will be removed in future versions. Use the `get_register_map()` method instead.) -- Deprecated since version 4.0.0.
  return result.readout_data.get("ro") # PyQuil 4.x


     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...
  -> Running circuit 2/6
     [1/3] Measuring X basis...
     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...
  -> Running circuit 3/6
     [1/3] Measuring X basis...
     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...
  -> Running circuit 4/6
     [1/3] Measuring X basis...
     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...
  -> Running circuit 5/6
     [1/3] Measuring X basis...
     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...
  -> Running circuit 6/6
     [1/3] Measuring X basis...
     [2/3] Measuring Y basis...
     [3/3] Measuring Z basis...

Ankaa-3 QPU Evaluation Complete!

Initial Cut Size from Ankaa-3 before Local Search: 4023.8021473872473
Final Optimized Cut Size (Post Ankaa-3 execution): 6340.6940459265215


In [72]:
from pytket.extensions.qiskit import qiskit_to_tk
from pytket.extensions.pyquil import tk_to_pyquil
from pyquil.paulis import sI, sX, sY, sZ
from pytket.passes import AutoRebase
from pyquil.api import WavefunctionSimulator
import numpy as np
from pytket.circuit import OpType
from pyquil.quilbase import Gate
from pyquil import Program, get_qc
from pyquil.gates import H, RX, MEASURE
from pyquil.quil import address_qubits

# Import Mitiq for Pauli Twirling
#from mitiq import rc

# ==========================================
# 8. PAULI TWIRLING & TRANSLATE VIA PYTKET
# ==========================================
# ... (your previous imports) ...
from pyquil.quil import address_qubits

# Corrected Import for Mitiq Pauli Twirling
from mitiq import pt

# ==========================================
# 8. PAULI TWIRLING & TRANSLATE VIA PYTKET
# ==========================================
# ==========================================
# 8. TRANSLATE VIA PYTKET (NO PAULI TWIRLING)
# ==========================================
print("\nTranslating optimal circuit to PyQuil...")

ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=2)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])
best_params_loaded = np.load("pce_optimal_theta_adam_reps4qbt12Continue2010_linear.npy")
bound_circuit = ansatz.assign_parameters(best_params_loaded)

rebase = AutoRebase({OpType.CX, OpType.Rx, OpType.Rz})

# Translate the single bound circuit to PyQuil (No Mitiq PT ensemble)
tket_circ = qiskit_to_tk(bound_circuit)
rebase.apply(tket_circ)
pyquil_prog = tk_to_pyquil(tket_circ)

def qiskit_to_pyquil_pauli(qiskit_op):
    pauli_str = qiskit_op.paulis.to_labels()[0]
    term = sI() 
    for qubit_idx, char in enumerate(reversed(pauli_str)):
        if char == 'X':
            term *= sX(qubit_idx)
        elif char == 'Y':
            term *= sY(qubit_idx)
        elif char == 'Z':
            term *= sZ(qubit_idx)
    return term

print("Translating observables...")
pyquil_obs_x = [qiskit_to_pyquil_pauli(op) for op in observables_x]
pyquil_obs_y = [qiskit_to_pyquil_pauli(op) for op in observables_y]
pyquil_obs_z = [qiskit_to_pyquil_pauli(op) for op in observables_z]

# ==========================================
# 9. EVALUATE ON PYQUIL QVM & DECODE
# ==========================================
print("Evaluating exact expectation values on PyQuil QVM...")

wfn_sim = WavefunctionSimulator()

# For the QVM, evaluating just the first circuit is sufficient 
# since simulators without noise don't suffer from coherent errors
E_x_qvm = np.real(wfn_sim.expectation(pyquil_rc_progs[0], pauli_terms=pyquil_obs_x))
E_y_qvm = np.real(wfn_sim.expectation(pyquil_rc_progs[0], pauli_terms=pyquil_obs_y))
E_z_qvm = np.real(wfn_sim.expectation(pyquil_rc_progs[0], pauli_terms=pyquil_obs_z))

E_final_qvm = np.concatenate([E_x_qvm, E_y_qvm, E_z_qvm])

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = [1 if val >= 0 else 0 for val in E_final_qvm]
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

# Local Search (QVM)
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post QVM):", best_cut)



Translating optimal circuit to PyQuil...
Translating observables...
Evaluating exact expectation values on PyQuil QVM...


/tmp/ipykernel_200474/1887562770.py:33: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=2)


Final Optimized Cut Size (Post QVM): 6586.1080232722825


In [74]:
# ==========================================
# 10. APPLY MAPPING & EXECUTE ON ANKAA-3 WITH T-REX
# ==========================================
from pyquil.quil import Program
from pyquil.gates import H, RX, MEASURE, X
import numpy as np

print("Preparing mapped circuit for Rigetti Ankaa-3 with T-REx...")

qubit_mapping = {
    0: 2, 1: 9, 2: 8, 3: 7, 4: 14, 5: 21, 
    6: 22, 7: 15, 8: 16, 9: 23, 10: 30, 11: 31
}

qc = get_qc("Ankaa-3")
total_shots = 10000

# Create a T-REx ensemble for the single circuit
num_trex_twirls = 10 
shots_per_twirl = total_shots // num_trex_twirls

# --- T-REx Enabled Execution Function ---
def run_mapped_basis(mapped_prog, basis, mapping, qc, shots, trex_bits=None):
    prog = Program(f'PRAGMA INITIAL_REWIRE "NAIVE"')
    prog += mapped_prog
    
    ro = prog.declare("ro", "BIT", len(mapping))
    
    for logical_q, physical_q in mapping.items():
        if basis == 'X':
            prog += H(physical_q)
        elif basis == 'Y':
            prog += RX(np.pi/2, physical_q)
            
        # T-REx: Apply X gate if twirl bit is 1
        if trex_bits is not None and trex_bits[logical_q] == 1:
            prog += X(physical_q)
            
    for logical_q, physical_q in mapping.items():
        prog += MEASURE(physical_q, ro[logical_q])
        
    prog.wrap_in_numshots_loop(shots)
    exe = qc.compile(prog)
    result = qc.run(exe)
    
    if hasattr(result, "readout_data"):
        return result.readout_data.get("ro") 
    else:
        return result.get_register_map().get("ro")

# --- T-REx Enabled Expectation Function ---
def get_expectations(bitstrings, qiskit_observables, shots, ro_error_rate=0.03):
    spins = 1 - 2 * bitstrings 
    evs = []
    
    for obs in qiskit_observables:
        pauli_str = list(reversed(obs.paulis.to_labels()[0]))
        obs_val = np.ones(shots)
        weight = 0  
        
        for q, char in enumerate(pauli_str):
            if char != 'I':
                obs_val *= spins[:, q] 
                weight += 1
                
        raw_ev = np.mean(obs_val)
        
        # T-REx: Invert the symmetric readout channel based on observable weight
        if weight > 0:
            mitigated_ev = raw_ev / ((1 - 2 * ro_error_rate) ** weight)
        else:
            mitigated_ev = raw_ev
            
        evs.append(mitigated_ev)
        
    return np.array(evs)

# Execute the single mapped circuit using a T-REx twirling loop
E_x_qpu_total, E_y_qpu_total, E_z_qpu_total = 0, 0, 0
ASSUMED_RO_ERROR = 0.03 

mapped_prog = address_qubits(pyquil_prog, qubit_mapping)

print(f"\nExecuting {num_trex_twirls} T-REx measurement variants on Ankaa-3 ({shots_per_twirl} shots each)...")

for i in range(num_trex_twirls):
    print(f"  -> Running T-REx variant {i+1}/{num_trex_twirls}")
    
    # Generate random T-REx bitstrings for each basis
    trex_x = np.random.randint(0, 2, size=len(qubit_mapping))
    trex_y = np.random.randint(0, 2, size=len(qubit_mapping))
    trex_z = np.random.randint(0, 2, size=len(qubit_mapping))
    
    res_x = run_mapped_basis(mapped_prog, 'X', qubit_mapping, qc, shots_per_twirl, trex_x)
    res_y = run_mapped_basis(mapped_prog, 'Y', qubit_mapping, qc, shots_per_twirl, trex_y)
    res_z = run_mapped_basis(mapped_prog, 'Z', qubit_mapping, qc, shots_per_twirl, trex_z)
    
    # T-REx: Classically reverse the bitflips using XOR
    res_x = res_x ^ trex_x
    res_y = res_y ^ trex_y
    res_z = res_z ^ trex_z
    
    E_x_qpu_total += get_expectations(res_x, observables_x, shots_per_twirl, ASSUMED_RO_ERROR)
    E_y_qpu_total += get_expectations(res_y, observables_y, shots_per_twirl, ASSUMED_RO_ERROR)
    E_z_qpu_total += get_expectations(res_z, observables_z, shots_per_twirl, ASSUMED_RO_ERROR)

# Average out the expectation values across the T-REx ensemble
E_x_qpu = E_x_qpu_total / num_trex_twirls
E_y_qpu = E_y_qpu_total / num_trex_twirls
E_z_qpu = E_z_qpu_total / num_trex_twirls

E_final_qpu = np.concatenate([E_x_qpu, E_y_qpu, E_z_qpu])
print("\nAnkaa-3 QPU Evaluation Complete!\n")

# ... (Proceed with your QPU Decoding & Local Search as normal) ...
# --- QPU Decoding & Local Search ---
cur_bits = [1 if val >= 0 else 0 for val in E_final_qpu]
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"Initial Cut Size from Ankaa-3 before Local Search: {best_cut}")

for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post Ankaa-3 execution):", best_cut)
# --- QPU Decoding & Local Search ---
cur_bits = [1 if val >= 0 else 0 for val in E_final_qpu]
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"Initial Cut Size from Ankaa-3 before Local Search: {best_cut}")

for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post Ankaa-3 execution):", best_cut)

Preparing mapped circuit for Rigetti Ankaa-3 with T-REx...

Executing 10 T-REx measurement variants on Ankaa-3 (1000 shots each)...
  -> Running T-REx variant 1/10


/tmp/ipykernel_200474/643466130.py:46: DeprecationWarning: Call to deprecated function (or staticmethod) readout_data. (This property is ambiguous now that the `get_raw_readout_data()` method exists and will be removed in future versions. Use the `get_register_map()` method instead.) -- Deprecated since version 4.0.0.
  if hasattr(result, "readout_data"):
/tmp/ipykernel_200474/643466130.py:47: DeprecationWarning: Call to deprecated function (or staticmethod) readout_data. (This property is ambiguous now that the `get_raw_readout_data()` method exists and will be removed in future versions. Use the `get_register_map()` method instead.) -- Deprecated since version 4.0.0.
  return result.readout_data.get("ro")


  -> Running T-REx variant 2/10
  -> Running T-REx variant 3/10
  -> Running T-REx variant 4/10
  -> Running T-REx variant 5/10
  -> Running T-REx variant 6/10
  -> Running T-REx variant 7/10
  -> Running T-REx variant 8/10
  -> Running T-REx variant 9/10
  -> Running T-REx variant 10/10

Ankaa-3 QPU Evaluation Complete!

Initial Cut Size from Ankaa-3 before Local Search: 3840.729376015568
Final Optimized Cut Size (Post Ankaa-3 execution): 6064.431556513115
Initial Cut Size from Ankaa-3 before Local Search: 3840.729376015568
Final Optimized Cut Size (Post Ankaa-3 execution): 6064.431556513115


In [75]:
# ==========================================
# 10. APPLY MAPPING & EXECUTE ON ANKAA-3 (PT + T-REX)
# ==========================================
from pyquil.quil import Program
from pyquil.gates import H, RX, MEASURE, X
import numpy as np

print("Preparing mapped PT ensemble for Rigetti Ankaa-3 with T-REx...")

qubit_mapping = {
    0: 2, 1: 9, 2: 8, 3: 7, 4: 14, 5: 21, 
    6: 22, 7: 15, 8: 16, 9: 23, 10: 30, 11: 31
}

qc = get_qc("Ankaa-3")
total_shots = 10000

# We assume 'pyquil_rc_progs' is your list of Pauli-Twirled circuits from Section 8
num_circs = len(pyquil_rc_progs)
shots_per_circ = total_shots // num_circs

# 1. --- Execution Function with T-REx ---
def run_mapped_basis(mapped_prog, basis, mapping, qc, shots, trex_bits=None):
    prog = Program(f'PRAGMA INITIAL_REWIRE "NAIVE"')
    prog += mapped_prog
    
    ro = prog.declare("ro", "BIT", len(mapping))
    
    for logical_q, physical_q in mapping.items():
        if basis == 'X':
            prog += H(physical_q)
        elif basis == 'Y':
            prog += RX(np.pi/2, physical_q)
            
        # T-REx: Apply X gate if twirl bit is 1
        if trex_bits is not None and trex_bits[logical_q] == 1:
            prog += X(physical_q)
            
    for logical_q, physical_q in mapping.items():
        prog += MEASURE(physical_q, ro[logical_q])
        
    prog.wrap_in_numshots_loop(shots)
    exe = qc.compile(prog)
    result = qc.run(exe)
    
    if hasattr(result, "readout_data"):
        return result.readout_data.get("ro") 
    else:
        return result.get_register_map().get("ro")

# 2. --- Expectation Function tracking Raw vs Mitigated ---
def get_expectations_with_raw(bitstrings, qiskit_observables, shots, ro_error_rate):
    spins = 1 - 2 * bitstrings 
    raw_evs = []
    mit_evs = []
    
    for obs in qiskit_observables:
        pauli_str = list(reversed(obs.paulis.to_labels()[0]))
        obs_val = np.ones(shots)
        weight = 0  
        
        for q, char in enumerate(pauli_str):
            if char != 'I':
                obs_val *= spins[:, q] 
                weight += 1
                
        raw_ev = np.mean(obs_val)
        
        # T-REx: Invert the symmetric readout channel based on observable weight
        if weight > 0:
            mitigated_ev = raw_ev / ((1 - 2 * ro_error_rate) ** weight)
        else:
            mitigated_ev = raw_ev
            
        raw_evs.append(raw_ev)
        mit_evs.append(mitigated_ev)
        
    return np.array(raw_evs), np.array(mit_evs)

# 3. --- Execute the Pauli Twirled Ensemble ---
raw_E_x_total, mit_E_x_total = 0, 0
raw_E_y_total, mit_E_y_total = 0, 0
raw_E_z_total, mit_E_z_total = 0, 0

# Lowered assumed symmetric error rate to 1.5%
ASSUMED_RO_ERROR = 0.015 

print(f"\nExecuting {num_circs} Pauli-Twirled variants on Ankaa-3 ({shots_per_circ} shots each)...")

for i, prog in enumerate(pyquil_rc_progs):
    print(f"  -> Running PT variant {i+1}/{num_circs} with T-REx")
    mapped_prog = address_qubits(prog, qubit_mapping)
    
    # Generate random T-REx bitstrings for each basis, for this specific PT circuit
    trex_x = np.random.randint(0, 2, size=len(qubit_mapping))
    trex_y = np.random.randint(0, 2, size=len(qubit_mapping))
    trex_z = np.random.randint(0, 2, size=len(qubit_mapping))
    
    res_x = run_mapped_basis(mapped_prog, 'X', qubit_mapping, qc, shots_per_circ, trex_x)
    res_y = run_mapped_basis(mapped_prog, 'Y', qubit_mapping, qc, shots_per_circ, trex_y)
    res_z = run_mapped_basis(mapped_prog, 'Z', qubit_mapping, qc, shots_per_circ, trex_z)
    
    # T-REx: Classically reverse the bitflips using XOR
    res_x = res_x ^ trex_x
    res_y = res_y ^ trex_y
    res_z = res_z ^ trex_z
    
    # Calculate and accumulate
    raw_x, mit_x = get_expectations_with_raw(res_x, observables_x, shots_per_circ, ASSUMED_RO_ERROR)
    raw_y, mit_y = get_expectations_with_raw(res_y, observables_y, shots_per_circ, ASSUMED_RO_ERROR)
    raw_z, mit_z = get_expectations_with_raw(res_z, observables_z, shots_per_circ, ASSUMED_RO_ERROR)
    
    raw_E_x_total += raw_x
    mit_E_x_total += mit_x
    
    raw_E_y_total += raw_y
    mit_E_y_total += mit_y
    
    raw_E_z_total += raw_z
    mit_E_z_total += mit_z

# Average out the expectation values across the ensemble
mit_E_x_qpu = mit_E_x_total / num_circs
mit_E_y_qpu = mit_E_y_total / num_circs
mit_E_z_qpu = mit_E_z_total / num_circs

raw_E_z_qpu = raw_E_z_total / num_circs  # Saving just Z for the printout

E_final_qpu = np.concatenate([mit_E_x_qpu, mit_E_y_qpu, mit_E_z_qpu])
print("\nAnkaa-3 QPU Evaluation Complete!")

# 4. --- Print T-REx Effect Comparison ---
print("\n--- T-REx Mitigation Effect (Z-Basis Snapshot) ---")
print(f"Assumed Symmetric Error: {ASSUMED_RO_ERROR*100}%")
print(f"{'Observable Weight':<20} | {'Raw Expectation':<20} | {'Mitigated Expectation':<20}")
print("-" * 65)

# Print the first 10 Z-observables to see the scaling effect
for idx in range(min(10, len(observables_z))):
    pauli_str = observables_z[idx].paulis.to_labels()[0]
    weight = len([char for char in pauli_str if char != 'I'])
    
    raw_val = raw_E_z_qpu[idx]
    mit_val = mit_E_z_qpu[idx]
    
    print(f"Weight {weight:<13} | {raw_val:<20.4f} | {mit_val:<20.4f}")
print("-" * 65)

# 5. --- QPU Decoding & Local Search (Using Mitigated Data) ---
cur_bits = [1 if val >= 0 else 0 for val in E_final_qpu]
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
        
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"\nInitial Cut Size from Ankaa-3 before Local Search: {best_cut}")

for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] 
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size (Post Ankaa-3 execution):", best_cut)

Preparing mapped PT ensemble for Rigetti Ankaa-3 with T-REx...

Executing 6 Pauli-Twirled variants on Ankaa-3 (1666 shots each)...
  -> Running PT variant 1/6 with T-REx


/tmp/ipykernel_200474/3551159505.py:46: DeprecationWarning: Call to deprecated function (or staticmethod) readout_data. (This property is ambiguous now that the `get_raw_readout_data()` method exists and will be removed in future versions. Use the `get_register_map()` method instead.) -- Deprecated since version 4.0.0.
  if hasattr(result, "readout_data"):
/tmp/ipykernel_200474/3551159505.py:47: DeprecationWarning: Call to deprecated function (or staticmethod) readout_data. (This property is ambiguous now that the `get_raw_readout_data()` method exists and will be removed in future versions. Use the `get_register_map()` method instead.) -- Deprecated since version 4.0.0.
  return result.readout_data.get("ro")


  -> Running PT variant 2/6 with T-REx
  -> Running PT variant 3/6 with T-REx
  -> Running PT variant 4/6 with T-REx
  -> Running PT variant 5/6 with T-REx
  -> Running PT variant 6/6 with T-REx

Ankaa-3 QPU Evaluation Complete!

--- T-REx Mitigation Effect (Z-Basis Snapshot) ---
Assumed Symmetric Error: 1.5%
Observable Weight    | Raw Expectation      | Mitigated Expectation
-----------------------------------------------------------------
Weight 2             | 0.8473               | 0.9006              
Weight 2             | 0.7585               | 0.8061              
Weight 2             | 0.7793               | 0.8283              
Weight 2             | 0.6643               | 0.7060              
Weight 2             | 0.8361               | 0.8887              
Weight 2             | 0.8323               | 0.8846              
Weight 2             | 0.6150               | 0.6537              
Weight 2             | 0.0294               | 0.0313              
Weight 2           

In [76]:
# ==========================================
# 10. APPLY MAPPING & EXECUTE ON ANKAA-3
# ==========================================
from pyquil.quil import Program
#from pyquil.quilatom import Pragma
from pyquil.gates import H, RX, MEASURE
import numpy as np

print("Preparing twirled ensemble for Rigetti Ankaa-3...")

# 🏆 The Top 5 Optimal Comprehensive Chain Mappings
top_5_mappings = {
    "Rank 1": {0: 20, 1: 19, 2: 12, 3: 5, 4: 4, 5: 11, 6: 18, 7: 25, 8: 24, 9: 31, 10: 30, 11: 29},
    "Rank 2": {0: 20, 1: 19, 2: 12, 3: 5, 4: 4, 5: 11, 6: 18, 7: 25, 8: 24, 9: 31, 10: 38, 11: 39},
    "Rank 3": {0: 10, 1: 11, 2: 4, 3: 5, 4: 12, 5: 19, 6: 18, 7: 25, 8: 24, 9: 31, 10: 30, 11: 29},
    "Rank 4": {0: 10, 1: 11, 2: 4, 3: 5, 4: 12, 5: 19, 6: 18, 7: 25, 8: 24, 9: 31, 10: 38, 11: 39},
    "Rank 5": {0: 20, 1: 19, 2: 12, 3: 5, 4: 4, 5: 11, 6: 18, 7: 25, 8: 24, 9: 31, 10: 30, 11: 23}
}

qc = get_qc("Ankaa-3")
total_shots = 10000
# Distribute shots evenly across the RC ensemble
shots_per_circ = total_shots // num_rc_circuits

def run_mapped_basis(mapped_prog, basis, mapping, qc, shots):
    # FORCE the compiler to use our exact physical qubits, no auto-rerouting!
    prog = Program('PRAGMA INITIAL_REWIRE "NAIVE"')
    prog += mapped_prog
    
    # Declare the readout register (ro) sized to our logical circuit
    ro = prog.declare("ro", "BIT", len(mapping))
    
    # Append the basis transformation gates to the physical qubits
    for logical_q, physical_q in mapping.items():
        if basis == 'X':
            prog += H(physical_q)
        elif basis == 'Y':
            prog += RX(np.pi/2, physical_q)
            
    # Measure the physical qubits into the logical indices of the readout register
    for logical_q, physical_q in mapping.items():
        prog += MEASURE(physical_q, ro[logical_q])
        
    prog.wrap_in_numshots_loop(shots)
    exe = qc.compile(prog)
    result = qc.run(exe)
    
    # PyQuil 3.x+ syntax for extracting readout results
    if hasattr(result, "readout_data"):
        return result.readout_data.get("ro") # PyQuil 4.x
    else:
        return result.get_register_map().get("ro") # Older PyQuil 3.x

def get_expectations(bitstrings, qiskit_observables, shots):
    # Convert {0, 1} bitstrings to {+1, -1} spin expectations
    spins = 1 - 2 * bitstrings 
    evs = []
    for obs in qiskit_observables:
        pauli_str = list(reversed(obs.paulis.to_labels()[0]))
        obs_val = np.ones(shots)
        for q, char in enumerate(pauli_str):
            if char != 'I':
                obs_val *= spins[:, q] 
        evs.append(np.mean(obs_val))
    return np.array(evs)

# Dictionary to store final results for the summary table
mapping_results = {}

print(f"\n=======================================================")
print(f" STARTING MULTI-MAPPING EXECUTION ON ANKAA-3")
print(f"=======================================================\n")

# Iterate through the top 5 mappings
for rank_name, qubit_mapping in top_5_mappings.items():
    print(f"\n--- Evaluating {rank_name} ---")
    print(f"Mapping: {qubit_mapping}")
    
    E_x_qpu_total, E_y_qpu_total, E_z_qpu_total = 0, 0, 0

    print(f"Executing {num_rc_circuits} twirled circuits ({shots_per_circ} shots each)...")
    for i, prog in enumerate(pyquil_rc_progs):
        # This correctly physically addresses the circuit using our mapping dictionary
        mapped_prog = address_qubits(prog, qubit_mapping)
        
        # Suppress repeated verbose output for circuits, keeping it clean
        res_x = run_mapped_basis(mapped_prog, 'X', qubit_mapping, qc, shots_per_circ)
        res_y = run_mapped_basis(mapped_prog, 'Y', qubit_mapping, qc, shots_per_circ)
        res_z = run_mapped_basis(mapped_prog, 'Z', qubit_mapping, qc, shots_per_circ)
        
        E_x_qpu_total += get_expectations(res_x, observables_x, shots_per_circ)
        E_y_qpu_total += get_expectations(res_y, observables_y, shots_per_circ)
        E_z_qpu_total += get_expectations(res_z, observables_z, shots_per_circ)

    # Average out the expectation values
    E_x_qpu = E_x_qpu_total / num_rc_circuits
    E_y_qpu = E_y_qpu_total / num_rc_circuits
    E_z_qpu = E_z_qpu_total / num_rc_circuits

    E_final_qpu = np.concatenate([E_x_qpu, E_y_qpu, E_z_qpu])
    print(f"QPU Execution Complete for {rank_name}.")

    # --- QPU Decoding & Local Search ---
    cur_bits = [1 if val >= 0 else 0 for val in E_final_qpu]
    cur_partition = [set(), set()]
    for i, bit in enumerate(cur_bits):
        cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
            
    best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    initial_cut = best_cut
    best_bits = cur_bits.copy()

    print(f"  -> Initial Cut Size (Before Local Search): {initial_cut}")

    for node in range(num_nodes):
        swapped_bits = best_bits.copy()
        swapped_bits[node] = 1 - swapped_bits[node] 
        
        cur_partition = [set(), set()]
        for i, bit in enumerate(swapped_bits):
            cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
                
        cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
        if cut_size > best_cut:
            best_cut = cut_size
            best_bits = swapped_bits

    final_cut = best_cut
    print(f"  -> Final Optimized Cut Size: {final_cut}")
    
    # Save to results dictionary
    mapping_results[rank_name] = {
        "Initial Cut": initial_cut,
        "Final Cut": final_cut
    }

# ==========================================
# 11. FINAL SUMMARIZING TABLE
# ==========================================
print("\n" + "="*60)
print(f"{'ANKAA-3 MULTI-MAPPING EXECUTION SUMMARY':^60}")
print("="*60)
print(f"{'Mapping Rank':<15} | {'Initial Cut Size':<20} | {'Optimized Cut Size':<20}")
print("-" * 60)

for rank_name, results in mapping_results.items():
    init_cut = results["Initial Cut"]
    fin_cut = results["Final Cut"]
    print(f"{rank_name:<15} | {init_cut:<20} | {fin_cut:<20}")

print("="*60 + "\n")

Preparing twirled ensemble for Rigetti Ankaa-3...

 STARTING MULTI-MAPPING EXECUTION ON ANKAA-3


--- Evaluating Rank 1 ---
Mapping: {0: 20, 1: 19, 2: 12, 3: 5, 4: 4, 5: 11, 6: 18, 7: 25, 8: 24, 9: 31, 10: 30, 11: 29}
Executing 6 twirled circuits (1666 shots each)...


/tmp/ipykernel_200474/3996620912.py:49: DeprecationWarning: Call to deprecated function (or staticmethod) readout_data. (This property is ambiguous now that the `get_raw_readout_data()` method exists and will be removed in future versions. Use the `get_register_map()` method instead.) -- Deprecated since version 4.0.0.
  if hasattr(result, "readout_data"):
/tmp/ipykernel_200474/3996620912.py:50: DeprecationWarning: Call to deprecated function (or staticmethod) readout_data. (This property is ambiguous now that the `get_raw_readout_data()` method exists and will be removed in future versions. Use the `get_register_map()` method instead.) -- Deprecated since version 4.0.0.
  return result.readout_data.get("ro") # PyQuil 4.x


QPU Execution Complete for Rank 1.
  -> Initial Cut Size (Before Local Search): 3563.2499803574347
  -> Final Optimized Cut Size: 5921.416276711309

--- Evaluating Rank 2 ---
Mapping: {0: 20, 1: 19, 2: 12, 3: 5, 4: 4, 5: 11, 6: 18, 7: 25, 8: 24, 9: 31, 10: 38, 11: 39}
Executing 6 twirled circuits (1666 shots each)...
QPU Execution Complete for Rank 2.
  -> Initial Cut Size (Before Local Search): 3072.7445579481932
  -> Final Optimized Cut Size: 6096.217877125056

--- Evaluating Rank 3 ---
Mapping: {0: 10, 1: 11, 2: 4, 3: 5, 4: 12, 5: 19, 6: 18, 7: 25, 8: 24, 9: 31, 10: 30, 11: 29}
Executing 6 twirled circuits (1666 shots each)...


KeyboardInterrupt: 

In [ ]:
print("hi")

In [77]:
# ==========================================
# 10. APPLY MAPPING & EXECUTE ON ANKAA-3
# ==========================================
from pyquil.quil import Program
#from pyquil.quilatom import Pragma
from pyquil.gates import H, RX, MEASURE
import numpy as np

print("Preparing twirled ensemble for Rigetti Ankaa-3...")

# 🏆 The Top 5 Optimal Comprehensive Chain Mappings
top_5_mappings = {
    #"Rank 1": {0: 20, 1: 19, 2: 12, 3: 5, 4: 4, 5: 11, 6: 18, 7: 25, 8: 24, 9: 31, 10: 30, 11: 29},
    #"Rank 2": {0: 20, 1: 19, 2: 12, 3: 5, 4: 4, 5: 11, 6: 18, 7: 25, 8: 24, 9: 31, 10: 38, 11: 39},
    "Rank 3": {0: 10, 1: 11, 2: 4, 3: 5, 4: 12, 5: 19, 6: 18, 7: 25, 8: 24, 9: 31, 10: 30, 11: 29},
    "Rank 4": {0: 10, 1: 11, 2: 4, 3: 5, 4: 12, 5: 19, 6: 18, 7: 25, 8: 24, 9: 31, 10: 38, 11: 39},
    "Rank 5": {0: 20, 1: 19, 2: 12, 3: 5, 4: 4, 5: 11, 6: 18, 7: 25, 8: 24, 9: 31, 10: 30, 11: 23}
}

qc = get_qc("Ankaa-3")
total_shots = 10000
# Distribute shots evenly across the RC ensemble
shots_per_circ = total_shots // num_rc_circuits

def run_mapped_basis(mapped_prog, basis, mapping, qc, shots):
    # FORCE the compiler to use our exact physical qubits, no auto-rerouting!
    prog = Program('PRAGMA INITIAL_REWIRE "NAIVE"')
    prog += mapped_prog
    
    # Declare the readout register (ro) sized to our logical circuit
    ro = prog.declare("ro", "BIT", len(mapping))
    
    # Append the basis transformation gates to the physical qubits
    for logical_q, physical_q in mapping.items():
        if basis == 'X':
            prog += H(physical_q)
        elif basis == 'Y':
            prog += RX(np.pi/2, physical_q)
            
    # Measure the physical qubits into the logical indices of the readout register
    for logical_q, physical_q in mapping.items():
        prog += MEASURE(physical_q, ro[logical_q])
        
    prog.wrap_in_numshots_loop(shots)
    exe = qc.compile(prog)
    result = qc.run(exe)
    
    # PyQuil 3.x+ syntax for extracting readout results
    if hasattr(result, "readout_data"):
        return result.readout_data.get("ro") # PyQuil 4.x
    else:
        return result.get_register_map().get("ro") # Older PyQuil 3.x

def get_expectations(bitstrings, qiskit_observables, shots):
    # Convert {0, 1} bitstrings to {+1, -1} spin expectations
    spins = 1 - 2 * bitstrings 
    evs = []
    for obs in qiskit_observables:
        pauli_str = list(reversed(obs.paulis.to_labels()[0]))
        obs_val = np.ones(shots)
        for q, char in enumerate(pauli_str):
            if char != 'I':
                obs_val *= spins[:, q] 
        evs.append(np.mean(obs_val))
    return np.array(evs)

# Dictionary to store final results for the summary table
mapping_results = {}

print(f"\n=======================================================")
print(f" STARTING MULTI-MAPPING EXECUTION ON ANKAA-3")
print(f"=======================================================\n")

# Iterate through the top 5 mappings
for rank_name, qubit_mapping in top_5_mappings.items():
    print(f"\n--- Evaluating {rank_name} ---")
    print(f"Mapping: {qubit_mapping}")
    
    E_x_qpu_total, E_y_qpu_total, E_z_qpu_total = 0, 0, 0

    print(f"Executing {num_rc_circuits} twirled circuits ({shots_per_circ} shots each)...")
    for i, prog in enumerate(pyquil_rc_progs):
        # This correctly physically addresses the circuit using our mapping dictionary
        mapped_prog = address_qubits(prog, qubit_mapping)
        
        # Suppress repeated verbose output for circuits, keeping it clean
        res_x = run_mapped_basis(mapped_prog, 'X', qubit_mapping, qc, shots_per_circ)
        res_y = run_mapped_basis(mapped_prog, 'Y', qubit_mapping, qc, shots_per_circ)
        res_z = run_mapped_basis(mapped_prog, 'Z', qubit_mapping, qc, shots_per_circ)
        
        E_x_qpu_total += get_expectations(res_x, observables_x, shots_per_circ)
        E_y_qpu_total += get_expectations(res_y, observables_y, shots_per_circ)
        E_z_qpu_total += get_expectations(res_z, observables_z, shots_per_circ)

    # Average out the expectation values
    E_x_qpu = E_x_qpu_total / num_rc_circuits
    E_y_qpu = E_y_qpu_total / num_rc_circuits
    E_z_qpu = E_z_qpu_total / num_rc_circuits

    E_final_qpu = np.concatenate([E_x_qpu, E_y_qpu, E_z_qpu])
    print(f"QPU Execution Complete for {rank_name}.")

    # --- QPU Decoding & Local Search ---
    cur_bits = [1 if val >= 0 else 0 for val in E_final_qpu]
    cur_partition = [set(), set()]
    for i, bit in enumerate(cur_bits):
        cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
            
    best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    initial_cut = best_cut
    best_bits = cur_bits.copy()

    print(f"  -> Initial Cut Size (Before Local Search): {initial_cut}")

    for node in range(num_nodes):
        swapped_bits = best_bits.copy()
        swapped_bits[node] = 1 - swapped_bits[node] 
        
        cur_partition = [set(), set()]
        for i, bit in enumerate(swapped_bits):
            cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
                
        cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
        if cut_size > best_cut:
            best_cut = cut_size
            best_bits = swapped_bits

    final_cut = best_cut
    print(f"  -> Final Optimized Cut Size: {final_cut}")
    
    # Save to results dictionary
    mapping_results[rank_name] = {
        "Initial Cut": initial_cut,
        "Final Cut": final_cut
    }

# ==========================================
# 11. FINAL SUMMARIZING TABLE
# ==========================================
print("\n" + "="*60)
print(f"{'ANKAA-3 MULTI-MAPPING EXECUTION SUMMARY':^60}")
print("="*60)
print(f"{'Mapping Rank':<15} | {'Initial Cut Size':<20} | {'Optimized Cut Size':<20}")
print("-" * 60)

for rank_name, results in mapping_results.items():
    init_cut = results["Initial Cut"]
    fin_cut = results["Final Cut"]
    print(f"{rank_name:<15} | {init_cut:<20} | {fin_cut:<20}")

print("="*60 + "\n")

Preparing twirled ensemble for Rigetti Ankaa-3...

 STARTING MULTI-MAPPING EXECUTION ON ANKAA-3


--- Evaluating Rank 3 ---
Mapping: {0: 10, 1: 11, 2: 4, 3: 5, 4: 12, 5: 19, 6: 18, 7: 25, 8: 24, 9: 31, 10: 30, 11: 29}
Executing 6 twirled circuits (1666 shots each)...


/tmp/ipykernel_200474/3848654958.py:49: DeprecationWarning: Call to deprecated function (or staticmethod) readout_data. (This property is ambiguous now that the `get_raw_readout_data()` method exists and will be removed in future versions. Use the `get_register_map()` method instead.) -- Deprecated since version 4.0.0.
  if hasattr(result, "readout_data"):
/tmp/ipykernel_200474/3848654958.py:50: DeprecationWarning: Call to deprecated function (or staticmethod) readout_data. (This property is ambiguous now that the `get_raw_readout_data()` method exists and will be removed in future versions. Use the `get_register_map()` method instead.) -- Deprecated since version 4.0.0.
  return result.readout_data.get("ro") # PyQuil 4.x


QPU Execution Complete for Rank 3.
  -> Initial Cut Size (Before Local Search): 3149.095372152267
  -> Final Optimized Cut Size: 6115.794839006137

--- Evaluating Rank 4 ---
Mapping: {0: 10, 1: 11, 2: 4, 3: 5, 4: 12, 5: 19, 6: 18, 7: 25, 8: 24, 9: 31, 10: 38, 11: 39}
Executing 6 twirled circuits (1666 shots each)...
QPU Execution Complete for Rank 4.
  -> Initial Cut Size (Before Local Search): 3998.7890559099073
  -> Final Optimized Cut Size: 6029.056490075492

--- Evaluating Rank 5 ---
Mapping: {0: 20, 1: 19, 2: 12, 3: 5, 4: 4, 5: 11, 6: 18, 7: 25, 8: 24, 9: 31, 10: 30, 11: 23}
Executing 6 twirled circuits (1666 shots each)...
QPU Execution Complete for Rank 5.
  -> Initial Cut Size (Before Local Search): 3458.1882173883205
  -> Final Optimized Cut Size: 5982.676360820268

          ANKAA-3 MULTI-MAPPING EXECUTION SUMMARY           
Mapping Rank    | Initial Cut Size     | Optimized Cut Size  
------------------------------------------------------------
Rank 3          | 3149.0953721

In [78]:
import pandas as pd
import numpy as np
import networkx as nx
import os
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import EstimatorV2
from qiskit import transpile

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    # Assuming columns: node1, node2, weight
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 9                 # Increased to 12 to support 180 nodes at k=2
k = 3                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        # This loop makes it dynamically work for any k
        for qubit_index in c:
            paulis[qubit_index] = pauli
            
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. QUANTUM CIRCUIT & ESTIMATOR SETUP
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=2)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])

estimator = EstimatorV2()
# High shots reduce statistical noise for the parameter shift gradients
estimator.options.default_shots = 10000

experiment_result = []

# ==========================================
# 4. LOSS FUNCTION & CHAIN RULE GRADIENTS
# ==========================================
alpha_loss = 24  # Rescaling factor scaled to approx n^(k/2)
beta_loss = 0.5
# Edwards-Erdös bound for the regularization parameter v
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def evaluate_expectations(theta):
    """Helper to get pure expectation values for a single parameter array."""
    job = estimator.run([
        (ansatz, observables_x, theta),
        (ansatz, observables_y, theta),
        (ansatz, observables_z, theta)
    ])
    results = job.result()
    
    # Extract 1D arrays of expectation values
    E_x = results[0].data.evs
    E_y = results[1].data.evs
    E_z = results[2].data.evs
    return np.concatenate([E_x, E_y, E_z])

def calc_loss_and_chain_rule(E):
    """Analytically calculates the loss and the gradient dL/dE."""
    # Precompute common terms for speed
    tE = np.tanh(alpha_loss * E)
    sech2E = 1.0 - tE**2
    
    loss_cut = 0.0
    dL_dE = np.zeros(num_nodes)
    
    # Chain rule for the Cut Loss term
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss_cut += w * tE[edge0] * tE[edge1]
        
        # dL/dE for connected nodes
        dL_dE[edge0] += w * alpha_loss * sech2E[edge0] * tE[edge1]
        dL_dE[edge1] += w * alpha_loss * sech2E[edge1] * tE[edge0]
        
    # Chain rule for the Regularization term
    S = np.sum(tE**2) / num_nodes
    reg_term = beta_loss * v * (S**2)
    loss = loss_cut + reg_term
    
    # Add regularization gradient to dL/dE
    reg_grad_factor = (4.0 * beta_loss * v * S / num_nodes) * alpha_loss
    dL_dE += reg_grad_factor * tE * sech2E
    
    return loss, dL_dE

# ==========================================
# 4.5. PARAMETER SHIFT RULE JACOBIAN (CORRECTED)
# ==========================================
def param_shift_gradient(theta):
    """Calculates the full gradient using batched Parameter Shift + Chain Rule."""
    num_params = len(theta)
    shift = np.pi / 2.0
    
    # Step A: Build a batched array of shifted parameters
    theta_shifted = []
    for j in range(num_params):
        t_plus = theta.copy()
        t_plus[j] += shift
        t_minus = theta.copy()
        t_minus[j] -= shift
        theta_shifted.append(t_plus)
        theta_shifted.append(t_minus)
        
    theta_shifted = np.array(theta_shifted) # Shape: (2*num_params, num_params)
    
    # FIX: Add a new axis to force a Cartesian product broadcast
    # Now it evaluates all 240 parameter sets against all 60 observables
    theta_shifted_broadcast = theta_shifted[:, np.newaxis, :] 
    
    # Step B: Run batched estimator
    job = estimator.run([
        (ansatz, observables_x, theta_shifted_broadcast),
        (ansatz, observables_y, theta_shifted_broadcast),
        (ansatz, observables_z, theta_shifted_broadcast)
    ])
    results = job.result()
    
    # Safely extract and orient the results to shape (len(observables), 2*num_params)
    def extract_evs(pub_result):
        evs = np.atleast_2d(pub_result.data.evs)
        # Transpose if the shape is (240, 60) to make it (60, 240)
        if evs.shape[0] == 2 * num_params:
            evs = evs.T
        return evs

    evs_x = extract_evs(results[0])
    evs_y = extract_evs(results[1])
    evs_z = extract_evs(results[2])
    
    # Stack vertically to get shape (num_nodes, 2*num_params)
    all_evs = np.vstack([evs_x, evs_y, evs_z])
    
    # Step C: Compute Jacobian dE/dTheta
    J = np.zeros((num_nodes, num_params))
    for j in range(num_params):
        evs_plus = all_evs[:, 2*j]
        evs_minus = all_evs[:, 2*j + 1]
        J[:, j] = (evs_plus - evs_minus) / 2.0
        
    # Step D: Apply Chain Rule
    E_current = evaluate_expectations(theta)
    loss, dL_dE = calc_loss_and_chain_rule(E_current)
    grad = dL_dE @ J
    
    # Save for bit-swap decoding
    global experiment_result
    experiment_result.append({"loss": loss, "exp_map": dict(enumerate(E_current))})
    
    return loss, grad

# ==========================================
# 5. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue2010_linear_k3.npy"
num_new_params = ansatz.num_parameters # For reps=4 and 12 qubits, this will be 120

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    num_old_params = len(old_params)
    
    if num_old_params == num_new_params:
        initial_params = old_params
    elif num_old_params < num_new_params:
        print(f"Padding parameters! Transferring {num_old_params} angles into a {num_new_params} angle circuit.")
        initial_params = np.random.uniform(-0.01, 0.01, num_new_params)
        initial_params[:num_old_params] = old_params
    else:
        print("Warning: Old parameter array is LARGER than new ansatz. Falling back to random.")
        np.random.seed(42)
        initial_params = np.random.rand(num_new_params)
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_new_params)

# ==========================================
# 6. CUSTOM ADAM OPTIMIZER
# ==========================================
def adam_optimize_param_shift(theta0, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=50):
    print(f"Starting optimization with Parameter Shift Adam (lr={lr})...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    t = 0
    
    for i in range(max_iter):
        t += 1
        
        loss, grad = param_shift_gradient(theta_i)
        print(f"--- Adam Step {i+1}/{max_iter} --- | Loss: {loss:.6f}")
        
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**t)
        v_t_hat = v_t / (1 - beta2**t)
        
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            break
            
        theta_i = theta_next
        
    return theta_i

best_params = adam_optimize_param_shift(
    theta0=initial_params,
    lr=0.1,        
    max_iter=500
)

print("\nAdam Optimization Complete!")
np.save("pce_optimal_theta_adam_reps4qbt12Continue2010_linear_k3_second.npy", best_params)

# ==========================================
# 7. DECODING & BIT-SWAP POST-PROCESSING
# ==========================================
def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

cur_bits = []
for i in range(num_nodes):
    if experiment_result[-1]["exp_map"][i] >= 0:
        cur_bits.append(1)
    else:
        cur_bits.append(0)

# Calculate the initial base cut
cur_partition = [set(), set()]
for i, bit in enumerate(cur_bits):
    if bit > 0:
        cur_partition[0].add(i)
    else:
        cur_partition[1].add(i)
best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
best_bits = cur_bits.copy()

print(f"\nInitial Cut Size before Local Search: {best_cut}")

# Single-bit swap search as described in the paper
for node in range(num_nodes):
    swapped_bits = best_bits.copy()
    swapped_bits[node] = 1 - swapped_bits[node] # Flip 1 to 0 or 0 to 1
    
    cur_partition = [set(), set()]
    for i, bit in enumerate(swapped_bits):
        if bit > 0:
            cur_partition[0].add(i)
        else:
            cur_partition[1].add(i)
            
    cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    
    # Keep the swap if it improves the cut
    if cut_size > best_cut:
        best_cut = cut_size
        best_bits = swapped_bits

print("Final Optimized Cut Size:", best_cut)

Loading dataset...
Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue2010_linear_k3.npy...
Starting optimization with Parameter Shift Adam (lr=0.1)...


/tmp/ipykernel_200474/2502479092.py:59: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=2)


--- Adam Step 1/500 --- | Loss: -4039.039703
--- Adam Step 2/500 --- | Loss: -3336.437849
--- Adam Step 3/500 --- | Loss: -3506.585735
--- Adam Step 4/500 --- | Loss: -3629.874697
--- Adam Step 5/500 --- | Loss: -3629.482040
--- Adam Step 6/500 --- | Loss: -3791.060218
--- Adam Step 7/500 --- | Loss: -3840.125522
--- Adam Step 8/500 --- | Loss: -3846.433630
--- Adam Step 9/500 --- | Loss: -3887.554862
--- Adam Step 10/500 --- | Loss: -3919.407843
--- Adam Step 11/500 --- | Loss: -3913.919922
--- Adam Step 12/500 --- | Loss: -3923.278857
--- Adam Step 13/500 --- | Loss: -3961.816248
--- Adam Step 14/500 --- | Loss: -3982.955454
--- Adam Step 15/500 --- | Loss: -3991.041099
--- Adam Step 16/500 --- | Loss: -4008.962370
--- Adam Step 17/500 --- | Loss: -4019.108163
--- Adam Step 18/500 --- | Loss: -4020.527208
--- Adam Step 19/500 --- | Loss: -4035.175747
--- Adam Step 20/500 --- | Loss: -4053.018912
--- Adam Step 21/500 --- | Loss: -4067.324035
--- Adam Step 22/500 --- | Loss: -4070.3945

In [80]:
import os
import numpy as np
import pandas as pd
import networkx as nx
from itertools import combinations

from qiskit.circuit.library import EfficientSU2
from qiskit.quantum_info import SparsePauliOp
from qiskit import transpile

from pytket.extensions.qiskit import qiskit_to_tk
from pytket.extensions.pyquil import tk_to_pyquil
from pytket.passes import AutoRebase
from pytket.circuit import OpType

from pyquil import Program, get_qc
from pyquil.gates import H, RX, MEASURE
from pyquil.quil import address_qubits

# ==========================================
# 1. LOAD GRAPH FROM PARQUET
# ==========================================
print("Loading dataset...")
df = pd.read_parquet("DataB.parquet")

graph = nx.Graph()
for _, row in df.iterrows():
    graph.add_edge(int(row.iloc[0]), int(row.iloc[1]), weight=float(row.iloc[2]))

num_nodes = len(graph.nodes())  # Should be 180
num_qubits = 9                  # Set to 9 based on your mapping dictionary
k = 3                           # Quadratic compression

# ==========================================
# 2. PAULI-CORRELATION ENCODING (PCE)
# ==========================================
list_size = num_nodes // 3
node_x = [i for i in range(list_size)]
node_y = [i for i in range(list_size, 2 * list_size)]
node_z = [i for i in range(2 * list_size, num_nodes)]

def build_pauli_correlation_encoding(pauli, node_list, n, k):
    pauli_correlation_encoding = []
    for idx, c in enumerate(combinations(range(n), k)):
        if idx >= len(node_list):
            break
        paulis = ["I"] * n
        for qubit_index in c:
            paulis[qubit_index] = pauli
        pauli_correlation_encoding.append("".join(paulis))
        
    hamiltonian = []
    for p in pauli_correlation_encoding:
        hamiltonian.append(SparsePauliOp.from_list([(p, 1)]))
    return hamiltonian

observables_x = build_pauli_correlation_encoding("X", node_x, num_qubits, k)
observables_y = build_pauli_correlation_encoding("Y", node_y, num_qubits, k)
observables_z = build_pauli_correlation_encoding("Z", node_z, num_qubits, k)

# ==========================================
# 3. CIRCUIT SETUP & QPU MAPPING
# ==========================================
ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=2)
ansatz = transpile(ansatz, basis_gates=['u1', 'u2', 'u3', 'cx', 'rz', 'sx', 'x'])
num_params = ansatz.num_parameters

# PyTket rebase pass
rebase = AutoRebase({OpType.CX, OpType.Rx, OpType.Rz})

# Setup Rigetti Ankaa-3
qc = get_qc("Ankaa-3")
shots_per_eval = 4000  # Adjust as needed for budget/accuracy

qubit_mapping = {
    0: 2, 1: 3, 2: 4, 3: 11, 4: 18, 5: 19, 6: 12, 7: 5, 8: 6
}

# ==========================================
# 4. QPU EXECUTION HELPER FUNCTIONS
# ==========================================
def run_mapped_basis(mapped_prog, basis, mapping, qc, shots):
    prog = Program(f'PRAGMA INITIAL_REWIRE "NAIVE"')
    prog += mapped_prog
    ro = prog.declare("ro", "BIT", len(mapping))
    
    for logical_q, physical_q in mapping.items():
        if basis == 'X':
            prog += H(physical_q)
        elif basis == 'Y':
            prog += RX(np.pi/2, physical_q)
            
    for logical_q, physical_q in mapping.items():
        prog += MEASURE(physical_q, ro[logical_q])
        
    prog.wrap_in_numshots_loop(shots)
    exe = qc.compile(prog)
    result = qc.run(exe)
    
    if hasattr(result, "readout_data"):
        return result.readout_data.get("ro")
    else:
        return result.get_register_map().get("ro")

def get_expectations(bitstrings, qiskit_observables, shots):
    spins = 1 - 2 * bitstrings 
    evs = []
    for obs in qiskit_observables:
        pauli_str = list(reversed(obs.paulis.to_labels()[0]))
        obs_val = np.ones(shots)
        for q, char in enumerate(pauli_str):
            if char != 'I':
                obs_val *= spins[:, q] 
        evs.append(np.mean(obs_val))
    return np.array(evs)

def evaluate_expectations_qpu(theta):
    """Binds parameters, translates to PyQuil, and measures X, Y, Z on Ankaa-3."""
    # 1. Bind and Translate
    bound_circuit = ansatz.assign_parameters(theta)
    tket_circ = qiskit_to_tk(bound_circuit)
    rebase.apply(tket_circ)
    pyquil_prog = tk_to_pyquil(tket_circ)
    mapped_prog = address_qubits(pyquil_prog, qubit_mapping)
    
    # 2. Execute
    res_x = run_mapped_basis(mapped_prog, 'X', qubit_mapping, qc, shots_per_eval)
    res_y = run_mapped_basis(mapped_prog, 'Y', qubit_mapping, qc, shots_per_eval)
    res_z = run_mapped_basis(mapped_prog, 'Z', qubit_mapping, qc, shots_per_eval)
    
    # 3. Calculate Expectations
    E_x = get_expectations(res_x, observables_x, shots_per_eval)
    E_y = get_expectations(res_y, observables_y, shots_per_eval)
    E_z = get_expectations(res_z, observables_z, shots_per_eval)
    
    return np.concatenate([E_x, E_y, E_z])

# ==========================================
# 5. LOSS FUNCTION & DECODING
# ==========================================
alpha_loss = 24  
beta_loss = 0.5
v = len(graph.edges()) / 2 + (len(graph.nodes()) - 1) / 4

def calc_loss_and_chain_rule(E):
    tE = np.tanh(alpha_loss * E)
    sech2E = 1.0 - tE**2
    
    loss_cut = 0.0
    dL_dE = np.zeros(num_nodes)
    
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        loss_cut += w * tE[edge0] * tE[edge1]
        dL_dE[edge0] += w * alpha_loss * sech2E[edge0] * tE[edge1]
        dL_dE[edge1] += w * alpha_loss * sech2E[edge1] * tE[edge0]
        
    S = np.sum(tE**2) / num_nodes
    reg_term = beta_loss * v * (S**2)
    loss = loss_cut + reg_term
    
    reg_grad_factor = (4.0 * beta_loss * v * S / num_nodes) * alpha_loss
    dL_dE += reg_grad_factor * tE * sech2E
    
    return loss, dL_dE

def calc_cut_size(graph, partition0, partition1):
    cut_size = 0
    for edge0, edge1, data in graph.edges(data=True):
        w = data.get('weight', 1.0)
        if (edge0 in partition0 and edge1 in partition1) or (edge0 in partition1 and edge1 in partition0):
            cut_size += w
    return cut_size

def decode_and_search_cut(E_array):
    """Decodes expectations to bitstrings and performs 1-bit local search."""
    cur_bits = [1 if val >= 0 else 0 for val in E_array]
    cur_partition = [set(), set()]
    for i, bit in enumerate(cur_bits):
        cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
            
    best_cut = calc_cut_size(graph, cur_partition[0], cur_partition[1])
    best_bits = cur_bits.copy()

    for node in range(num_nodes):
        swapped_bits = best_bits.copy()
        swapped_bits[node] = 1 - swapped_bits[node] 
        
        cur_partition = [set(), set()]
        for i, bit in enumerate(swapped_bits):
            cur_partition[0].add(i) if bit > 0 else cur_partition[1].add(i)
                
        cut_size = calc_cut_size(graph, cur_partition[0], cur_partition[1])
        if cut_size > best_cut:
            best_cut = cut_size
            best_bits = swapped_bits
            
    return best_cut

# ==========================================
# 6. QPU PARAMETER SHIFT GRADIENT
# ==========================================
def param_shift_gradient_qpu(theta):
    shift = np.pi / 2.0
    
    # 1. Forward Pass
    print("  -> Evaluating Forward Pass on QPU...")
    E_current = evaluate_expectations_qpu(theta)
    loss, dL_dE = calc_loss_and_chain_rule(E_current)
    
    # 2. Calculate Jacobian via Parameter Shift
    J = np.zeros((num_nodes, num_params))
    for j in range(num_params):
        print(f"  -> Evaluating Parameter Shift {j+1}/{num_params} on QPU...")
        
        t_plus = theta.copy()
        t_plus[j] += shift
        E_plus = evaluate_expectations_qpu(t_plus)
        
        t_minus = theta.copy()
        t_minus[j] -= shift
        E_minus = evaluate_expectations_qpu(t_minus)
        
        J[:, j] = (E_plus - E_minus) / 2.0
        
    # 3. Apply Chain Rule
    grad = dL_dE @ J
    return loss, grad, E_current

# ==========================================
# 7. WARM START INITIALIZATION
# ==========================================
warm_start_file = "pce_optimal_theta_adam_reps4qbt12Continue2010_linear_k3.npy"

if os.path.exists(warm_start_file):
    print(f"Loading warm-start parameters from {warm_start_file}...")
    old_params = np.load(warm_start_file)
    if len(old_params) == num_params:
        initial_params = old_params
    else:
        print(f"Shape mismatch. Padding/truncating to {num_params} parameters.")
        initial_params = np.random.uniform(-0.01, 0.01, num_params)
        transfer_len = min(len(old_params), num_params)
        initial_params[:transfer_len] = old_params[:transfer_len]
else:
    print("No warm-start file found. Initializing with random parameters...")
    np.random.seed(42)
    initial_params = np.random.rand(num_params)

# ==========================================
# 8. QPU ADAM OPTIMIZATION LOOP
# ==========================================
def adam_optimize_qpu(theta0, lr=0.1, beta1=0.9, beta2=0.999, eps=1e-8, max_iter=10):
    print(f"\nStarting QPU Adam Optimization for {max_iter} iterations...")
    theta_i = np.array(theta0)
    
    m_t = np.zeros_like(theta_i)
    v_t = np.zeros_like(theta_i)
    
    for i in range(max_iter):
        print(f"\n==============================================")
        print(f" ITERATION {i+1} / {max_iter}")
        print(f"==============================================")
        
        # Calculate Gradients on QPU
        loss, grad, E_current = param_shift_gradient_qpu(theta_i)
        
        # Calculate Cut for this iteration
        best_cut_iter = decode_and_search_cut(E_current)
        
        print(f"\n[ITERATION {i+1} RESULTS]")
        print(f"-> Loss: {loss:.6f}")
        print(f"-> Optimized Cut Size: {best_cut_iter}")
        
        # Save Parameters
        save_file = f"qpu_params_iter_{i+1}.npy"
        np.save(save_file, theta_i)
        print(f"-> Saved parameters to {save_file}")
        
        # Adam Update
        m_t = beta1 * m_t + (1 - beta1) * grad
        v_t = beta2 * v_t + (1 - beta2) * (grad ** 2)
        
        m_t_hat = m_t / (1 - beta1**(i+1))
        v_t_hat = v_t / (1 - beta2**(i+1))
        
        theta_next = theta_i - lr * m_t_hat / (np.sqrt(v_t_hat) + eps)
        
        if np.linalg.norm(theta_next - theta_i) < 1e-6:
            print("Converged based on minimum step size!")
            theta_i = theta_next
            break
            
        theta_i = theta_next
        
    return theta_i

# Run the optimization!
best_params_qpu = adam_optimize_qpu(initial_params, lr=0.1, max_iter=5)

print("\nHardware Optimization Complete!")
np.save("final_qpu_optimal_theta.npy", best_params_qpu)

Loading dataset...


/tmp/ipykernel_200474/2401556022.py:64: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = EfficientSU2(num_qubits, entanglement='linear', reps=2)


Loading warm-start parameters from pce_optimal_theta_adam_reps4qbt12Continue2010_linear_k3.npy...

Starting QPU Adam Optimization for 5 iterations...

 ITERATION 1 / 5
  -> Evaluating Forward Pass on QPU...


/tmp/ipykernel_200474/2401556022.py:100: DeprecationWarning: Call to deprecated function (or staticmethod) readout_data. (This property is ambiguous now that the `get_raw_readout_data()` method exists and will be removed in future versions. Use the `get_register_map()` method instead.) -- Deprecated since version 4.0.0.
  if hasattr(result, "readout_data"):
/tmp/ipykernel_200474/2401556022.py:101: DeprecationWarning: Call to deprecated function (or staticmethod) readout_data. (This property is ambiguous now that the `get_raw_readout_data()` method exists and will be removed in future versions. Use the `get_register_map()` method instead.) -- Deprecated since version 4.0.0.
  return result.readout_data.get("ro")


  -> Evaluating Parameter Shift 1/54 on QPU...
  -> Evaluating Parameter Shift 2/54 on QPU...
  -> Evaluating Parameter Shift 3/54 on QPU...
  -> Evaluating Parameter Shift 4/54 on QPU...
  -> Evaluating Parameter Shift 5/54 on QPU...
  -> Evaluating Parameter Shift 6/54 on QPU...
  -> Evaluating Parameter Shift 7/54 on QPU...
  -> Evaluating Parameter Shift 8/54 on QPU...
  -> Evaluating Parameter Shift 9/54 on QPU...
  -> Evaluating Parameter Shift 10/54 on QPU...
  -> Evaluating Parameter Shift 11/54 on QPU...
  -> Evaluating Parameter Shift 12/54 on QPU...
  -> Evaluating Parameter Shift 13/54 on QPU...
  -> Evaluating Parameter Shift 14/54 on QPU...
  -> Evaluating Parameter Shift 15/54 on QPU...
  -> Evaluating Parameter Shift 16/54 on QPU...
  -> Evaluating Parameter Shift 17/54 on QPU...


QpuApiError: Call failed during gRPC request: status: Cancelled, message: "Timeout expired", details: [], metadata: MetadataMap { headers: {} }